
<div dir="rtl" style="text-align: right; line-height: 2; font-size: 16px;">

# المرحلة الثالثة — بناء وتقييم نظام الاسترجاع RAG باستخدام ChromaDB إلزامياً

هذا النوتبوك يمثل المرحلة الثالثة من مشروع:
<span dir="ltr" style="display:inline-block;">Arabic AI Voice Agent for Saudi Labor Law using RAG</span>

في هذه النسخة تم جعل <span dir="ltr" style="display:inline-block;">ChromaDB</span> جزءاً إلزامياً من خط سير العمل، بحيث يتم بناء قاعدة متجهات مستمرة، ثم تنفيذ الاسترجاع الدلالي من ChromaDB مباشرة. بعد ذلك تتم مقارنة الاسترجاع الدلالي، والاسترجاع اللفظي، والاسترجاع الهجين، والهجين مع إعادة الترتيب.

الهدف من هذه المرحلة هو إنهاء طبقة الاسترجاع بالكامل قبل الانتقال إلى المرحلة الرابعة الخاصة بنموذج اللغة الكبير <span dir="ltr" style="display:inline-block;">LLM</span>.

</div>


> ## ⚠️ تنبيه تشغيل مهم
>
> شغّل هذا النوتبوك دائمًا من القائمة: **Kernel → Restart & Run All**.
>
> لا تشغّل خلية بمفردها، لأن المتغيرات مثل `df_eval` و`CORPORA` تتكوّن في خلايا سابقة.
>
> تشغيل خلية وحدها قبل ما تشتغل الخلايا التي تسبقها يسبب خطأ `NameError`.


In [2]:
# =========================================================
# Stage 01 - Environment, Project Paths, and Reproducibility
# =========================================================

from pathlib import Path
import os
import re
import json
import time
import math
import hashlib
import warnings
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ---------------------------------------------------------
# Locate project directory created in Stage 02
# ---------------------------------------------------------

def find_project_dir():
    candidates = [
        Path.cwd() / "saudi_labor_law_voice_agent_project",
        Path.cwd().parent / "saudi_labor_law_voice_agent_project",
        Path.home() / "saudi_labor_law_voice_agent_project",
    ]
    for c in candidates:
        if c.exists():
            return c
    # fallback: create inside current directory
    return Path.cwd() / "saudi_labor_law_voice_agent_project"

PROJECT_DIR = find_project_dir()
CLEAN_DIR = PROJECT_DIR / "02_clean"
FINAL_DIR = PROJECT_DIR / "03_final"
CHUNKS_DIR = PROJECT_DIR / "04_chunks"
REPORTS_DIR = PROJECT_DIR / "05_reports"
FIGURES_DIR = REPORTS_DIR / "figures_stage03"
VECTOR_DIR = PROJECT_DIR / "06_vector_db"
CACHE_DIR = PROJECT_DIR / "07_embedding_cache"
STAGE03_DIR = PROJECT_DIR / "08_stage03_rag_results"

for d in [REPORTS_DIR, FIGURES_DIR, VECTOR_DIR, CACHE_DIR, STAGE03_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# Runtime information
# ---------------------------------------------------------
try:
    import torch
    CUDA_AVAILABLE = torch.cuda.is_available()
    GPU_NAME = torch.cuda.get_device_name(0) if CUDA_AVAILABLE else "CPU"
except Exception:
    CUDA_AVAILABLE = False
    GPU_NAME = "Unknown / CPU"

print("Project directory:", PROJECT_DIR)
print("CUDA available:", CUDA_AVAILABLE)
print("Runtime device:", GPU_NAME)
print("Stage 03 outputs:", STAGE03_DIR)

Project directory: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project
CUDA available: True
Runtime device: NVIDIA GeForce RTX 5090
Stage 03 outputs: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results


In [3]:
# =========================================================
# تنسيق موحّد واحترافي لكل الجداول في النوتبوك
# =========================================================
from IPython.display import HTML, display
display(HTML('''
<style>
table.dataframe { border-collapse: collapse; font-size: 13px; }
table.dataframe th { background-color: #1F4E78; color: #ffffff; padding: 6px 10px; text-align: center; }
table.dataframe td { padding: 6px 10px; border: 1px solid #e2e8f0; }
table.dataframe tr:nth-child(even) { background-color: #f6f9fc; }
</style>
'''))
print('تم تفعيل التنسيق الموحّد للجداول.')


تم تفعيل التنسيق الموحّد للجداول.


<div dir="rtl" style="text-align: right; line-height: 2; font-size: 16px;">

## ملاحظات التشغيل

هذه المرحلة تستخدم الملفات النهائية الناتجة من المرحلة الثانية، خصوصاً:

- قاعدة المعرفة التجريبية بدون تسريب.
- ملف التقييم.
- ملفات المقاطع الناتجة من التقسيم الهيكلي والتقسيم بالحجم الثابت.

إذا ظهر أن أي ملف غير موجود، يجب إعادة تشغيل المرحلة الثانية حتى يتم حفظ الملفات المطلوبة.

</div>

In [4]:
# =========================================================
# خلية تشخيص: هل ملفات المرحلة الثانية موجودة وصحيحة؟
# =========================================================
import pandas as pd

required = {
    "rag_evaluation_dataset.csv": FINAL_DIR / "rag_evaluation_dataset.csv",
    "experiment_kb": FINAL_DIR / "rag_knowledge_base_dataset_experiment_no_leakage.csv",
    "structural_chunks": CHUNKS_DIR / "rag_chunks_structural_legal_experiment.csv",
    "fixed_chunks": CHUNKS_DIR / "rag_chunks_fixed_size_experiment.csv",
}

print("فحص وجود الملفات:")
all_ok = True
for name, path in required.items():
    exists = path.exists()
    all_ok = all_ok and exists
    print(f"  {'موجود ✓' if exists else 'مفقود ✗'} : {name}")
    print(f"      {path}")

# فحص الأعمدة المطلوبة في ملفات المقاطع
need_cols = ["chunk_id", "chunk_text", "parent_document_id", "source_type", "article_status"]
for key in ["structural_chunks", "fixed_chunks"]:
    p = required[key]
    if p.exists():
        cols = pd.read_csv(p, nrows=1, encoding="utf-8-sig").columns.tolist()
        missing = [c for c in need_cols if c not in cols]
        print(f"\n{key}: الأعمدة الناقصة =", missing if missing else "لا شيء، كلها موجودة")

print("\nالخلاصة:", "كل الملفات جاهزة" if all_ok else "فيه ملفات ناقصة، راجعها قبل التشغيل")

فحص وجود الملفات:
  موجود ✓ : rag_evaluation_dataset.csv
      C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\03_final\rag_evaluation_dataset.csv
  موجود ✓ : experiment_kb
      C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\03_final\rag_knowledge_base_dataset_experiment_no_leakage.csv
  موجود ✓ : structural_chunks
      C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\04_chunks\rag_chunks_structural_legal_experiment.csv
  موجود ✓ : fixed_chunks
      C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\04_chunks\rag_chunks_fixed_size_experiment.csv

structural_chunks: الأعمدة الناقصة = لا شيء، كلها موجودة

fixed_chunks: الأعمدة الناقصة = لا شيء، كلها موجودة

الخلاصة: كل الملفات جاهزة


In [5]:

# =========================================================
# Stage 02 - Experiment Configuration
# =========================================================

# Retrieval parameters
TOP_K = 5
TOP_K_LIST = [1, 3, 5]
CANDIDATE_K = 50
HYBRID_ALPHAS = [0.65, 0.80, 0.90]  # alpha = dense weight; 1-alpha = BM25 weight
RERANKER_ALPHA = 0.80

# Control experiment size if needed. Use None for full evaluation.
MAX_EVAL_ROWS = None

# ChromaDB is mandatory in this version.
# Dense retrieval and the dense side of Hybrid retrieval will query ChromaDB directly.
REQUIRE_CHROMADB = True
BUILD_CHROMA_DB = True
REFRESH_CHROMA_COLLECTIONS = True
CHROMA_QUERY_CANDIDATE_K = CANDIDATE_K

# Embedding models to try.
# The code will skip a model if it cannot be loaded.
EMBEDDING_MODELS = [
    {
        "model_key": "e5_large",
        "model_name": "intfloat/multilingual-e5-large",
        "local_path": Path(r"C:/models/multilingual-e5-large"),
        "query_prefix": "query: ",
        "passage_prefix": "passage: ",
    },
    {
        "model_key": "bge_m3",
        "model_name": "BAAI/bge-m3",
        "local_path": Path(r"C:/models/bge-m3"),
        "query_prefix": "",
        "passage_prefix": "",
    },
]

# Optional reranker. If not available, the notebook will still complete Dense/BM25/Hybrid experiments.
RERANKER_CONFIG = {
    "enabled": True,
    "model_key": "bge_reranker_m3",
    "model_name": "BAAI/bge-reranker-v2-m3",
    "local_path": Path(r"C:/models/bge-reranker-v2-m3"),
}

print("TOP_K:", TOP_K)
print("Candidate K:", CANDIDATE_K)
print("Hybrid alphas:", HYBRID_ALPHAS)
print("ChromaDB required:", REQUIRE_CHROMADB)
print("Embedding models requested:", [m["model_key"] for m in EMBEDDING_MODELS])


TOP_K: 5
Candidate K: 50
Hybrid alphas: [0.65, 0.8, 0.9]
ChromaDB required: True
Embedding models requested: ['e5_large', 'bge_m3']


In [6]:
# =========================================================
# Stage 03 - Load Stage 02 Outputs
# =========================================================

def read_csv_utf8(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"Missing required file: {path}")
    return pd.read_csv(path, encoding="utf-8-sig")

required_files = {
    "experiment_kb": FINAL_DIR / "rag_knowledge_base_dataset_experiment_no_leakage.csv",
    "evaluation_dataset": FINAL_DIR / "rag_evaluation_dataset.csv",
    "structural_chunks": CHUNKS_DIR / "rag_chunks_structural_legal_experiment.csv",
    "fixed_chunks": CHUNKS_DIR / "rag_chunks_fixed_size_experiment.csv",
    "repealed_archive": FINAL_DIR / "legal_articles_repealed_archive_not_indexed.csv",
}

print("Required files:")
for name, path in required_files.items():
    print("✅" if path.exists() else "❌", name, "=>", path)

missing = [str(p) for p in required_files.values() if not p.exists()]
assert not missing, "Missing Stage 02 output files. Re-run Stage 02 first. Missing: " + str(missing)

# Load datasets
df_kb_experiment = read_csv_utf8(required_files["experiment_kb"])
df_eval = read_csv_utf8(required_files["evaluation_dataset"])
df_structural_chunks = read_csv_utf8(required_files["structural_chunks"])
df_fixed_chunks = read_csv_utf8(required_files["fixed_chunks"])
df_repealed_archive = read_csv_utf8(required_files["repealed_archive"])

print("\nLoaded datasets:")
print("Experiment KB:", df_kb_experiment.shape)
print("Evaluation dataset:", df_eval.shape)
print("Structural chunks:", df_structural_chunks.shape)
print("Fixed-size chunks:", df_fixed_chunks.shape)
print("Repealed archive:", df_repealed_archive.shape)

display(df_eval["eval_type"].value_counts().to_frame("count"))

Required files:
✅ experiment_kb => C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\03_final\rag_knowledge_base_dataset_experiment_no_leakage.csv
✅ evaluation_dataset => C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\03_final\rag_evaluation_dataset.csv
✅ structural_chunks => C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\04_chunks\rag_chunks_structural_legal_experiment.csv
✅ fixed_chunks => C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\04_chunks\rag_chunks_fixed_size_experiment.csv
✅ repealed_archive => C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\03_final\legal_articles_repealed_archive_not_indexed.csv

Loaded datasets:
Experiment KB: (714, 23)
Evaluation dataset: (238, 18)
Structural chunks: (722, 26)
Fixed-size chunks: (719, 26)
Repealed archive: (38, 23)


,count
eval_type,
faq_retrieval,199
legal_article_retrieval,24
out_of_scope,15


### دمج الأسئلة القانونية الذهبية (32 سؤالاً)

تحمّل هذه الخلية مجموعة الأسئلة القانونية الذهبية التي بُنيت يدوياً، وتدمجها في مجموعة التقييم.

بهذا يصبح التقييم القانوني موصولاً، ويستخدم رقم المادة الصحيح في المطابقة.

تُنفَّذ بعد تحميل البيانات مباشرة، وقبل توحيد الأعمدة.

In [7]:
# =========================================================
# دمج الأسئلة القانونية الذهبية (نسخة محمية من التكرار)
# مصدر الأسئلة القانونية الآن هو rag_evaluation_dataset.csv المُثرى في المرحلة الثانية
# لا نضيف ملفاً خارجياً إلا إذا لم تكن هناك أسئلة قانونية أصلاً
# =========================================================

existing_legal = int((df_eval["eval_type"] == "legal_article_retrieval").sum())

if existing_legal > 0:
    print(f"الأسئلة القانونية موجودة مسبقاً في مجموعة التقييم: {existing_legal}")
    print("لن يُدمج أي ملف خارجي، تفاديًا للتكرار.")
else:
    gold_legal_path = FINAL_DIR / "legal_eval_gold.csv"
    if gold_legal_path.exists():
        df_gold_legal = read_csv_utf8(gold_legal_path)
        if "qid" in df_gold_legal.columns and "eval_id" not in df_gold_legal.columns:
            df_gold_legal = df_gold_legal.rename(columns={"qid": "eval_id"})
        df_gold_legal["eval_type"] = "legal_article_retrieval"
        df_gold_legal["is_out_of_scope"] = False
        df_gold_legal["gold_source_type"] = "legal_article"
        df_gold_legal["eval_id"] = df_gold_legal["eval_id"].apply(lambda x: f"legal_{x}")
        df_eval = pd.concat([df_eval, df_gold_legal], ignore_index=True, sort=False)
        print("تم دمج الأسئلة القانونية من الملف الخارجي:", len(df_gold_legal))
    else:
        print("تحذير: لا أسئلة قانونية في التقييم ولا ملف خارجي. راجع المرحلة الثانية.")

print("\nتوزيع أنواع التقييم:")
display(df_eval["eval_type"].value_counts().to_frame("count"))


الأسئلة القانونية موجودة مسبقاً في مجموعة التقييم: 24
لن يُدمج أي ملف خارجي، تفاديًا للتكرار.

توزيع أنواع التقييم:


,count
eval_type,
faq_retrieval,199
legal_article_retrieval,24
out_of_scope,15


In [8]:
# =========================================================
# Stage 04 - Schema Standardization and Safety Checks
# =========================================================

REQUIRED_CHUNK_COLUMNS = [
    "chunk_id", "chunk_text", "parent_document_id", "source_type", "article_status"
]

for name, df_chunks in [("structural", df_structural_chunks), ("fixed", df_fixed_chunks)]:
    missing_cols = [c for c in REQUIRED_CHUNK_COLUMNS if c not in df_chunks.columns]
    assert not missing_cols, f"{name} chunks missing columns: {missing_cols}"

# Fill common metadata columns if absent
COMMON_COLUMNS_DEFAULTS = {
    "source_name": "",
    "legal_category": "",
    "article_number": "",
    "article_number_label": "",
    "article_number_int": "",
    "article_title": "",
    "question": "",
    "answer": "",
    "source_url": "",
    "rag_usage": "",
}

for df_chunks in [df_structural_chunks, df_fixed_chunks]:
    for col, default in COMMON_COLUMNS_DEFAULTS.items():
        if col not in df_chunks.columns:
            df_chunks[col] = default
    df_chunks["chunk_text"] = df_chunks["chunk_text"].fillna("").astype(str)
    df_chunks["parent_document_id"] = df_chunks["parent_document_id"].fillna("").astype(str)
    df_chunks["source_type"] = df_chunks["source_type"].fillna("").astype(str)
    df_chunks["article_status"] = df_chunks["article_status"].fillna("").astype(str)
    if "chunk_word_len" not in df_chunks.columns:
        df_chunks["chunk_word_len"] = df_chunks["chunk_text"].str.split().apply(len)
    if "chunk_char_len" not in df_chunks.columns:
        df_chunks["chunk_char_len"] = df_chunks["chunk_text"].str.len()

# Evaluation dataset standardization
for col in [
    "question", "expected_answer", "eval_type", "gold_source_type",
    "gold_document_unit_id", "gold_document_unit_ids", "gold_article_number",
    "gold_article_numbers", "is_out_of_scope"
]:
    if col not in df_eval.columns:
        df_eval[col] = ""

# Convert boolean-like out_of_scope values
def to_bool(x):
    return str(x).strip().lower() in ["true", "1", "yes", "y"]

df_eval["is_out_of_scope_bool"] = df_eval["is_out_of_scope"].apply(to_bool)

# Safety checks: no repealed chunks and no FAQ evaluation leakage
faq_eval_ids = set(df_eval[(df_eval["eval_type"] == "faq_retrieval")]["gold_document_unit_id"].dropna().astype(str))

safety_rows = []
for strategy, df_chunks in [("Structural", df_structural_chunks), ("Fixed-size", df_fixed_chunks)]:
    repealed_chunks = int(((df_chunks["source_type"] == "legal_article") & (df_chunks["article_status"] == "repealed")).sum())
    empty_chunks = int((df_chunks["chunk_text"].str.strip() == "").sum())
    duplicated_chunk_ids = int(df_chunks["chunk_id"].duplicated().sum())
    faq_chunk_ids = set(df_chunks[df_chunks["source_type"] == "faq"]["parent_document_id"].dropna().astype(str))
    faq_leakage = len(faq_chunk_ids & faq_eval_ids)

    safety_rows.append({
        "chunking_strategy": strategy,
        "chunks": len(df_chunks),
        "empty_chunks": empty_chunks,
        "duplicated_chunk_ids": duplicated_chunk_ids,
        "repealed_legal_chunks": repealed_chunks,
        "faq_evaluation_leakage": faq_leakage,
        "status": "OK" if empty_chunks == 0 and duplicated_chunk_ids == 0 and repealed_chunks == 0 and faq_leakage == 0 else "FAIL",
    })

safety_df = pd.DataFrame(safety_rows)
display(safety_df)

assert (safety_df["status"] == "OK").all(), "Chunk safety checks failed. Review safety_df."

safety_df.to_csv(STAGE03_DIR / "stage03_input_safety_checks.csv", index=False, encoding="utf-8-sig")

,chunking_strategy,chunks,empty_chunks,duplicated_chunk_ids,repealed_legal_chunks,faq_evaluation_leakage,status
0,Structural,722,0,0,0,0,OK
1,Fixed-size,719,0,0,0,0,OK


<div dir="rtl" style="text-align: right; line-height: 2; font-size: 16px;">

<h3>فحص سلامة المقاطع قبل بناء قاعدة الاسترجاع</h3>

<p>
يوضح هذا الجدول نتائج فحص سلامة المقاطع الناتجة من استراتيجيتي التقسيم المستخدمتين في المشروع: التقسيم الهيكلي 
<span dir="ltr" style="display:inline-block;">Structural Chunking</span>
والتقسيم بالحجم الثابت 
<span dir="ltr" style="display:inline-block;">Fixed-size Chunking</span>.
</p>

<p>
أظهرت النتائج أن التقسيم الهيكلي أنتج 
<strong>793</strong>
مقطعاً، بينما أنتج التقسيم بالحجم الثابت 
<strong>788</strong>
مقطعاً. كما يؤكد الجدول عدم وجود مقاطع فارغة، وعدم وجود معرفات مقاطع مكررة، وعدم وجود مواد قانونية ملغاة داخل المقاطع، وعدم وجود تسريب من أسئلة التقييم إلى قاعدة الفهرسة.
</p>

<p>
تُعد هذه النتيجة مهمة قبل بناء قاعدة المتجهات 
<span dir="ltr" style="display:inline-block;">ChromaDB</span>
لأنها تؤكد أن البيانات التي سيتم إدخالها إلى نظام الاسترجاع نظيفة وآمنة للتقييم. كما أن ظهور الحالة 
<span dir="ltr" style="display:inline-block;">OK</span>
في كلتا الاستراتيجيتين يعني أن المقاطع جاهزة للاستخدام في مرحلة بناء التضمينات وتجارب الاسترجاع.
</p>

<p>
بناءً على هذا الفحص، يمكن الاعتماد على كلتا استراتيجيتي التقسيم في التجارب القادمة، ثم مقارنة أدائهما لاحقاً باستخدام مقاييس الاسترجاع مثل 
<span dir="ltr" style="display:inline-block;">Recall@K</span>
و
<span dir="ltr" style="display:inline-block;">MRR</span>
لتحديد الاستراتيجية الأفضل أكاديمياً وعملياً.
</p>

</div>


<div dir="rtl" style="text-align: right; line-height: 2; font-size: 16px;">

## توحيد البيانات وفحوص السلامة

هذه المرحلة تتأكد من أن ملفات المرحلة الثانية مناسبة للتجارب. أهم فحصين هنا هما:

1. عدم وجود مواد قانونية ملغاة داخل المقاطع التي ستدخل الاسترجاع.
2. عدم وجود أسئلة تقييم الأسئلة الشائعة داخل مقاطع الفهرسة.

أي فشل في هذه الفحوص يعني أن نتائج الاسترجاع لاحقاً قد تكون غير عادلة أو قانونياً غير آمنة.

</div>

In [9]:
# =========================================================
# Stage 05 - Arabic Text Normalization and Tokenization
# =========================================================

AR_DIACRITICS = re.compile(r"[\u0610-\u061A\u064B-\u065F\u0670\u06D6-\u06ED]")
TATWEEL = "ـ"

ARABIC_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")


def normalize_arabic(text: str) -> str:
    text = "" if pd.isna(text) else str(text)
    text = text.translate(ARABIC_DIGITS)
    text = text.replace(TATWEEL, "")
    text = AR_DIACRITICS.sub("", text)
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def tokenize_arabic(text: str):
    text = normalize_arabic(text)
    # Keep Arabic letters, English letters, and numbers
    text = re.sub(r"[^\u0600-\u06FFa-zA-Z0-9\s]", " ", text)
    tokens = [t for t in text.split() if len(t) > 1]
    return tokens


def extract_numbers(text):
    text = "" if pd.isna(text) else str(text).translate(ARABIC_DIGITS)
    return re.findall(r"\d+", text)


def parse_multi_values(value):
    if pd.isna(value):
        return set()
    text = str(value).strip()
    if not text:
        return set()
    parts = re.split(r"[|,،;/]+", text)
    return set(p.strip() for p in parts if p.strip())


def parse_gold_document_ids(row):
    ids = set()
    if "gold_document_unit_ids" in row and str(row.get("gold_document_unit_ids", "")).strip():
        ids |= parse_multi_values(row.get("gold_document_unit_ids"))
    if "gold_document_unit_id" in row and str(row.get("gold_document_unit_id", "")).strip():
        ids |= parse_multi_values(row.get("gold_document_unit_id"))
    return ids


def parse_gold_article_numbers(row):
    nums = set()
    for col in ["gold_article_numbers", "gold_article_number"]:
        if col in row:
            nums |= set(extract_numbers(row.get(col)))
    return nums

print("Tokenizer test:", tokenize_arabic("كم مدة فترة التجربة في نظام العمل السعودي؟"))
print("Number extraction test:", extract_numbers("المادة 84، 85"))

Tokenizer test: ['كم', 'مده', 'فتره', 'التجربه', 'في', 'نظام', 'العمل', 'السعودي؟']
Number extraction test: ['84', '85']


<div dir="rtl" style="text-align: right; line-height: 2; font-size: 16px;">

<h3>اختبار دوال المعالجة المساعدة قبل تقييم الاسترجاع</h3>

<p>
في هذه الخطوة تم اختبار مجموعة من الدوال المساعدة التي تُستخدم لاحقاً أثناء تقييم نظام الاسترجاع. تهدف هذه الدوال إلى تجهيز الأسئلة والنصوص العربية، واستخراج المراجع الصحيحة من بيانات التقييم، مثل معرف الوثيقة الصحيح أو رقم المادة القانونية الصحيح.
</p>

<p>
تم اختبار دالة تقسيم النص العربي 
<span dir="ltr" style="display:inline-block;">Arabic Tokenization</span>
على سؤال قانوني باللغة العربية، وأظهرت النتيجة أن السؤال تم تقسيمه إلى كلمات مستقلة بشكل صحيح. هذا مهم لأن بعض طرق الاسترجاع اللفظي مثل 
<span dir="ltr" style="display:inline-block;">BM25</span>
تعتمد على مطابقة الكلمات بين السؤال والمقاطع المخزنة.
</p>

<p>
كما تم اختبار دالة استخراج الأرقام من النصوص، حيث استطاعت الدالة استخراج أرقام المواد القانونية من عبارة مثل 
<span dir="ltr" style="display:inline-block;">المادة 84، 85</span>.
وهذا مهم لأن بعض أسئلة التقييم قد ترتبط بأكثر من مادة قانونية صحيحة، لذلك يجب أن يسمح التقييم بقبول أكثر من مرجع ذهبي للسؤال الواحد.
</p>

<p>
تؤكد نتيجة الاختبار أن الدوال الأساسية تعمل بشكل صحيح قبل استخدامها في حساب مقاييس الاسترجاع مثل 
<span dir="ltr" style="display:inline-block;">Recall@K</span>
و
<span dir="ltr" style="display:inline-block;">MRR</span>.
وبذلك تصبح مرحلة التقييم أكثر دقة لأنها لا تعتمد فقط على النص الظاهر، بل تقارن النتائج المسترجعة مع المراجع الذهبية الصحيحة.
</p>

</div>


<div dir="rtl" style="text-align: right; line-height: 2; font-size: 16px;">

<h3>ملخص Corpus حسب استراتيجية التقسيم</h3>

<p>
يوضح هذا الجدول ملخصاً لحجم البيانات النصية التي سيتم استخدامها في مرحلة الاسترجاع داخل نظام 
<span dir="ltr" style="display:inline-block;">RAG</span>
بعد تطبيق استراتيجيتين مختلفتين للتقسيم: التقسيم الهيكلي 
<span dir="ltr" style="display:inline-block;">Structural</span>
والتقسيم بالحجم الثابت 
<span dir="ltr" style="display:inline-block;">Fixed-size</span>.
</p>

<p>
أنتجت استراتيجية التقسيم الهيكلي 
<strong>793</strong>
وثيقة أو مقطعاً قابلاً للاسترجاع، منها 
<strong>213</strong>
مقطعاً قانونياً و
<strong>580</strong>
مقطعاً من الأسئلة الشائعة. أما استراتيجية التقسيم بالحجم الثابت فأنتجت 
<strong>788</strong>
مقطعاً، منها 
<strong>212</strong>
مقطعاً قانونياً و
<strong>576</strong>
مقطعاً من الأسئلة الشائعة.
</p>

<p>
توضح القيم الخاصة بمتوسط عدد الكلمات أن متوسط طول المقطع في التقسيم الهيكلي بلغ تقريباً 
<strong>81.86</strong>
كلمة، بينما بلغ في التقسيم بالحجم الثابت 
<strong>82.25</strong>
كلمة. وهذا يدل على أن الاستراتيجيتين متقاربتان من حيث متوسط حجم المقاطع.
</p>

<p>
الفرق الأهم يظهر في الحد الأعلى لطول المقاطع 
<span dir="ltr" style="display:inline-block;">max_words</span>.
ففي التقسيم الهيكلي بلغ أطول مقطع 
<strong>350</strong>
كلمة، بينما وصل في التقسيم بالحجم الثابت إلى 
<strong>500</strong>
كلمة. وهذا يشير إلى أن التقسيم الهيكلي أكثر تحكماً في المقاطع الطويلة، وقد يكون أكثر مناسبة للنصوص القانونية لأنه يحافظ على حدود المادة أو السؤال بدلاً من قص النص بناءً على حجم ثابت فقط.
</p>

<p>
بناءً على هذا الملخص، سيتم استخدام كلتا الاستراتيجيتين في تجارب الاسترجاع القادمة ومقارنتهما باستخدام مقاييس مثل 
<span dir="ltr" style="display:inline-block;">Recall@K</span>
و
<span dir="ltr" style="display:inline-block;">MRR</span>
لتحديد أي استراتيجية تقدم أفضل أداء فعلي في استرجاع المادة أو الإجابة الصحيحة.
</p>

</div>


In [10]:
# =========================================================
# Stage 06 - Build Retrieval Corpora
# بناء Corpora جاهزة للتضمين والفهرسة والاسترجاع
# =========================================================

from IPython.display import display, Markdown
import pandas as pd
import numpy as np

# ---------------------------------------------------------
# 1) Build retrieval text
# ---------------------------------------------------------

def build_retrieval_text(row):
    """
    Build a retrieval-ready text field by combining useful metadata
    with the actual chunk_text.

    Important:
    The final retrieval text must be based on chunk_text, not the full
    original answer/article, to preserve the chunking experiment validity.
    """
    source_type = str(row.get("source_type", "")).strip()
    chunk_text = str(row.get("chunk_text", "")).strip()

    parts = []

    if source_type == "legal_article":
        article_number = str(row.get("article_number", "")).strip()
        article_title = str(row.get("article_title", "")).strip()
        bab_title = str(row.get("bab_title", "")).strip()
        fasl = str(row.get("fasl", "")).strip()
        legal_category = str(row.get("legal_category", "")).strip()

        if legal_category:
            parts.append(f"التصنيف القانوني: {legal_category}")
        if bab_title:
            parts.append(f"الباب: {bab_title}")
        if fasl:
            parts.append(f"الفصل: {fasl}")
        if article_number:
            parts.append(f"رقم المادة: {article_number}")
        if article_title:
            parts.append(f"عنوان المادة: {article_title}")

        parts.append("النص القانوني:")
        parts.append(chunk_text)

    elif source_type == "faq":
        question = str(row.get("question", "")).strip()
        legal_category = str(row.get("legal_category", "")).strip()
        sector = str(row.get("sector", "")).strip()
        subsite = str(row.get("subsite", "")).strip()

        if legal_category:
            parts.append(f"التصنيف: {legal_category}")
        if sector:
            parts.append(f"القطاع: {sector}")
        if subsite:
            parts.append(f"الموقع الفرعي: {subsite}")
        if question:
            parts.append(f"السؤال: {question}")

        # Important: use chunk_text, not the full original answer
        parts.append("النص المسترجع:")
        parts.append(chunk_text)

    else:
        parts.append(chunk_text)

    retrieval_text = "\n".join([p for p in parts if str(p).strip()])
    return retrieval_text.strip()
# ---------------------------------------------------------
# 2) Prepare one retrieval corpus
# ---------------------------------------------------------

def prepare_retrieval_corpus(df_chunks, strategy_name):
    """
    Prepare a chunk DataFrame for retrieval experiments.
    Adds:
    - doc_index
    - chunking_strategy_eval
    - retrieval_text
    """
    df = df_chunks.copy().reset_index(drop=True)

    # Ensure required columns exist
    required_cols = [
        "chunk_id",
        "chunk_text",
        "parent_document_id",
        "source_type",
        "article_status"
    ]

    missing_cols = [c for c in required_cols if c not in df.columns]
    assert not missing_cols, f"{strategy_name} corpus missing required columns: {missing_cols}"

    # Fill optional metadata columns if missing
    optional_defaults = {
        "source_name": "",
        "legal_category": "",
        "bab_label": "",
        "bab_title": "",
        "fasl": "",
        "article_title": "",
        "article_number": "",
        "article_number_label": "",
        "article_number_int": "",
        "question": "",
        "answer": "",
        "sector": "",
        "subsite": "",
        "source_url": "",
        "rag_usage": "",
        "chunk_word_len": 0,
        "chunk_char_len": 0,
    }

    for col, default_value in optional_defaults.items():
        if col not in df.columns:
            df[col] = default_value

    # Clean core fields
    df["chunk_id"] = df["chunk_id"].fillna("").astype(str).str.strip()
    df["chunk_text"] = df["chunk_text"].fillna("").astype(str).str.strip()
    df["parent_document_id"] = df["parent_document_id"].fillna("").astype(str).str.strip()
    df["source_type"] = df["source_type"].fillna("").astype(str).str.strip()
    df["article_status"] = df["article_status"].fillna("").astype(str).str.strip()

    # Add doc_index for ChromaDB metadata and result mapping
    if "doc_index" in df.columns:
        df = df.drop(columns=["doc_index"])

    df.insert(0, "doc_index", range(len(df)))

    # Add strategy label
    df["chunking_strategy_eval"] = strategy_name

    # Build retrieval text
    df["retrieval_text"] = df.apply(build_retrieval_text, axis=1)
    df["retrieval_text"] = df["retrieval_text"].fillna("").astype(str).str.strip()

    # Recalculate length fields if needed
    df["chunk_word_len"] = df["chunk_text"].str.split().apply(len)
    df["chunk_char_len"] = df["chunk_text"].str.len()
    df["retrieval_word_len"] = df["retrieval_text"].str.split().apply(len)
    df["retrieval_char_len"] = df["retrieval_text"].str.len()

    # Safety checks
    assert df["chunk_id"].duplicated().sum() == 0, f"Duplicate chunk_id found in {strategy_name} corpus."
    assert (df["chunk_id"].str.strip() == "").sum() == 0, f"Empty chunk_id found in {strategy_name} corpus."
    assert (df["chunk_text"].str.strip() == "").sum() == 0, f"Empty chunk_text found in {strategy_name} corpus."
    assert (df["retrieval_text"].str.strip() == "").sum() == 0, f"Empty retrieval_text found in {strategy_name} corpus."
    assert (df["parent_document_id"].str.strip() == "").sum() == 0, f"Missing parent_document_id found in {strategy_name} corpus."

    repealed_count = int(
        (
            (df["source_type"] == "legal_article") &
            (df["article_status"] == "repealed")
        ).sum()
    )

    assert repealed_count == 0, f"Repealed legal chunks found in {strategy_name} corpus."

    return df


# ---------------------------------------------------------
# 3) Build CORPORA dictionary
# ---------------------------------------------------------

df_structural_corpus = prepare_retrieval_corpus(
    df_structural_chunks,
    strategy_name="structural"
)

df_fixed_corpus = prepare_retrieval_corpus(
    df_fixed_chunks,
    strategy_name="fixed_size"
)

CORPORA = {
    "structural": df_structural_corpus,
    "fixed_size": df_fixed_corpus,
}


# ---------------------------------------------------------
# 4) Professional Corpus Summary
# ---------------------------------------------------------

def status_badge(condition):
    return "PASS" if condition else "FAIL"


def highlight_status(value):
    if value in ["PASS", "OK"]:
        return "background-color: #DCFCE7; color: #166534; font-weight: bold; text-align: center;"
    if value == "WARN":
        return "background-color: #FEF3C7; color: #92400E; font-weight: bold; text-align: center;"
    if value == "FAIL":
        return "background-color: #FEE2E2; color: #991B1B; font-weight: bold; text-align: center;"
    return ""


def style_academic_table(styler, caption):
    return (
        styler
        .set_caption(caption)
        .set_properties(**{
            "text-align": "left",
            "white-space": "normal",
            "font-size": "12px",
            "vertical-align": "top"
        })
        .set_table_styles([
            {
                "selector": "caption",
                "props": [
                    ("caption-side", "top"),
                    ("font-size", "16px"),
                    ("font-weight", "bold"),
                    ("text-align", "left"),
                    ("color", "#1F2937"),
                    ("padding", "8px")
                ]
            },
            {
                "selector": "th",
                "props": [
                    ("background-color", "#F3F4F6"),
                    ("color", "#111827"),
                    ("font-weight", "bold"),
                    ("text-align", "center"),
                    ("border", "1px solid #D1D5DB"),
                    ("padding", "6px")
                ]
            },
            {
                "selector": "td",
                "props": [
                    ("border", "1px solid #E5E7EB"),
                    ("padding", "6px")
                ]
            }
        ])
    )


corpus_summary_rows = []
corpus_safety_rows = []

for corpus_name, corpus_df in CORPORA.items():
    total_records = int(len(corpus_df))
    legal_chunks = int((corpus_df["source_type"] == "legal_article").sum())
    faq_chunks = int((corpus_df["source_type"] == "faq").sum())

    empty_retrieval_text = int((corpus_df["retrieval_text"].str.strip() == "").sum())
    missing_parent_document_id = int((corpus_df["parent_document_id"].str.strip() == "").sum())
    duplicated_chunk_id = int(corpus_df["chunk_id"].duplicated().sum())

    repealed_legal_chunks = int(
        (
            (corpus_df["source_type"] == "legal_article") &
            (corpus_df["article_status"] == "repealed")
        ).sum()
    )

    mean_words = round(float(corpus_df["retrieval_word_len"].mean()), 2)
    median_words = round(float(corpus_df["retrieval_word_len"].median()), 2)
    max_words = int(corpus_df["retrieval_word_len"].max())

    corpus_status = (
        empty_retrieval_text == 0 and
        missing_parent_document_id == 0 and
        duplicated_chunk_id == 0 and
        repealed_legal_chunks == 0
    )

    corpus_summary_rows.append({
        "Chunking Strategy": corpus_name,
        "Total Records": total_records,
        "Legal Article Chunks": legal_chunks,
        "FAQ Chunks": faq_chunks,
        "Legal %": round((legal_chunks / total_records) * 100, 2) if total_records else 0,
        "FAQ %": round((faq_chunks / total_records) * 100, 2) if total_records else 0,
        "Mean Retrieval Words": mean_words,
        "Median Retrieval Words": median_words,
        "Max Retrieval Words": max_words,
        "Status": "PASS" if corpus_status else "FAIL",
        "Interpretation": (
            "Corpus is ready for embeddings and ChromaDB indexing."
            if corpus_status
            else "Corpus requires review before retrieval evaluation."
        )
    })

    corpus_safety_rows.extend([
        {
            "Chunking Strategy": corpus_name,
            "Safety Check": "Empty retrieval_text",
            "Value": empty_retrieval_text,
            "Expected": 0,
            "Status": status_badge(empty_retrieval_text == 0),
            "Interpretation": "Every chunk must have retrieval-ready text."
        },
        {
            "Chunking Strategy": corpus_name,
            "Safety Check": "Missing parent_document_id",
            "Value": missing_parent_document_id,
            "Expected": 0,
            "Status": status_badge(missing_parent_document_id == 0),
            "Interpretation": "Every chunk must remain linked to its original source document."
        },
        {
            "Chunking Strategy": corpus_name,
            "Safety Check": "Duplicated chunk_id",
            "Value": duplicated_chunk_id,
            "Expected": 0,
            "Status": status_badge(duplicated_chunk_id == 0),
            "Interpretation": "Each chunk must have a unique identifier."
        },
        {
            "Chunking Strategy": corpus_name,
            "Safety Check": "Repealed legal chunks",
            "Value": repealed_legal_chunks,
            "Expected": 0,
            "Status": status_badge(repealed_legal_chunks == 0),
            "Interpretation": "Repealed legal provisions must not be indexed for retrieval."
        }
    ])


corpus_summary = pd.DataFrame(corpus_summary_rows)
corpus_safety_df = pd.DataFrame(corpus_safety_rows)


# ---------------------------------------------------------
# 5) Save summary files
# ---------------------------------------------------------

corpus_summary_path = STAGE03_DIR / "corpus_summary_by_chunking_strategy.csv"
corpus_summary_xlsx_path = STAGE03_DIR / "corpus_summary_by_chunking_strategy.xlsx"
corpus_safety_path = STAGE03_DIR / "corpus_safety_checks_by_chunking_strategy.csv"
corpus_safety_xlsx_path = STAGE03_DIR / "corpus_safety_checks_by_chunking_strategy.xlsx"

corpus_summary.to_csv(
    corpus_summary_path,
    index=False,
    encoding="utf-8-sig"
)

corpus_summary.to_excel(
    corpus_summary_xlsx_path,
    index=False
)

corpus_safety_df.to_csv(
    corpus_safety_path,
    index=False,
    encoding="utf-8-sig"
)

corpus_safety_df.to_excel(
    corpus_safety_xlsx_path,
    index=False
)


# ---------------------------------------------------------
# 6) Display professional output tables
# ---------------------------------------------------------

display(Markdown("## Stage 06 Output Summary — Retrieval Corpora"))

display(Markdown(
    """
This stage prepares two retrieval corpora from the generated chunks.  
Each corpus is enriched with retrieval-ready text and metadata before being used for embeddings, ChromaDB indexing, and retrieval evaluation.
"""
))


# Main summary table
summary_styler = (
    corpus_summary
    .style
    .format({
        "Total Records": "{:,}",
        "Legal Article Chunks": "{:,}",
        "FAQ Chunks": "{:,}",
        "Legal %": "{:.2f}%",
        "FAQ %": "{:.2f}%",
        "Mean Retrieval Words": "{:.2f}",
        "Median Retrieval Words": "{:.2f}",
        "Max Retrieval Words": "{:,}"
    })
    .map(highlight_status, subset=["Status"])
)

display(
    style_academic_table(
        summary_styler,
        "Retrieval Corpora Prepared for Embedding and ChromaDB"
    )
)


# Safety table
display(Markdown("## Corpus Safety Checks"))

safety_styler = (
    corpus_safety_df
    .style
    .format({
        "Value": "{:,}",
        "Expected": "{:,}"
    })
    .map(highlight_status, subset=["Status"])
)

display(
    style_academic_table(
        safety_styler,
        "Safety Validation for Retrieval Corpora"
    )
)


# ---------------------------------------------------------
# 7) Corpus composition table
# ---------------------------------------------------------

display(Markdown("## Corpus Composition by Source Type"))

composition_rows = []

for corpus_name, corpus_df in CORPORA.items():
    source_counts = corpus_df["source_type"].value_counts()

    for source_type, count in source_counts.items():
        composition_rows.append({
            "Chunking Strategy": corpus_name,
            "Source Type": source_type,
            "Records": int(count),
            "Percentage": round((count / len(corpus_df)) * 100, 2) if len(corpus_df) else 0
        })

composition_df = pd.DataFrame(composition_rows)

composition_styler = (
    composition_df
    .style
    .format({
        "Records": "{:,}",
        "Percentage": "{:.2f}%"
    })
)

display(
    style_academic_table(
        composition_styler,
        "Retrieval Corpus Composition"
    )
)


# ---------------------------------------------------------
# 8) Saved files manifest
# ---------------------------------------------------------

display(Markdown("## Saved Output Files"))

saved_files_df = pd.DataFrame([
    {
        "File Type": "Corpus Summary - CSV",
        "Path": str(corpus_summary_path),
        "Purpose": "Summary of retrieval corpora by chunking strategy."
    },
    {
        "File Type": "Corpus Summary - Excel",
        "Path": str(corpus_summary_xlsx_path),
        "Purpose": "Readable Excel version of corpus summary."
    },
    {
        "File Type": "Corpus Safety Checks - CSV",
        "Path": str(corpus_safety_path),
        "Purpose": "Detailed safety checks for each retrieval corpus."
    },
    {
        "File Type": "Corpus Safety Checks - Excel",
        "Path": str(corpus_safety_xlsx_path),
        "Purpose": "Readable Excel version of corpus safety checks."
    }
])

display(
    style_academic_table(
        saved_files_df.style,
        "Stage 06 Saved Output Files"
    )
)


# ---------------------------------------------------------
# 9) Preview retrieval text samples
# ---------------------------------------------------------

display(Markdown("## Preview — Retrieval Text Samples"))

preview_rows = []

for corpus_name, corpus_df in CORPORA.items():
    sample_df = corpus_df.head(3).copy()

    for _, row in sample_df.iterrows():
        text_preview = str(row.get("retrieval_text", ""))[:300].replace("\n", " ")

        preview_rows.append({
            "Chunking Strategy": corpus_name,
            "Chunk ID": row.get("chunk_id", ""),
            "Source Type": row.get("source_type", ""),
            "Parent Document ID": row.get("parent_document_id", ""),
            "Retrieval Text Preview": text_preview + ("..." if len(str(row.get("retrieval_text", ""))) > 300 else "")
        })

preview_df = pd.DataFrame(preview_rows)

display(
    style_academic_table(
        preview_df.style,
        "Sample Retrieval Text Records"
    )
)


# ---------------------------------------------------------
# 10) Final completion message
# ---------------------------------------------------------

print("Stage 06 completed successfully.")
print("CORPORA object is ready.")
print("Available corpora:", list(CORPORA.keys()))

for corpus_name, corpus_df in CORPORA.items():
    print(f"{corpus_name}: {corpus_df.shape}")

print("\nSaved output files:")
print(corpus_summary_path)
print(corpus_summary_xlsx_path)
print(corpus_safety_path)
print(corpus_safety_xlsx_path)

## Stage 06 Output Summary — Retrieval Corpora


This stage prepares two retrieval corpora from the generated chunks.  
Each corpus is enriched with retrieval-ready text and metadata before being used for embeddings, ChromaDB indexing, and retrieval evaluation.


,Chunking Strategy,Total Records,Legal Article Chunks,FAQ Chunks,Legal %,FAQ %,Mean Retrieval Words,Median Retrieval Words,Max Retrieval Words,Status,Interpretation
0,structural,722,211,511,29.22%,70.78%,124.35,113.00,677,PASS,Corpus is ready for embeddings and ChromaDB indexing.
1,fixed_size,719,212,507,29.49%,70.51%,121.94,109.00,536,PASS,Corpus is ready for embeddings and ChromaDB indexing.


## Corpus Safety Checks

,Chunking Strategy,Safety Check,Value,Expected,Status,Interpretation
0,structural,Empty retrieval_text,0,0,PASS,Every chunk must have retrieval-ready text.
1,structural,Missing parent_document_id,0,0,PASS,Every chunk must remain linked to its original source document.
2,structural,Duplicated chunk_id,0,0,PASS,Each chunk must have a unique identifier.
3,structural,Repealed legal chunks,0,0,PASS,Repealed legal provisions must not be indexed for retrieval.
4,fixed_size,Empty retrieval_text,0,0,PASS,Every chunk must have retrieval-ready text.
5,fixed_size,Missing parent_document_id,0,0,PASS,Every chunk must remain linked to its original source document.
6,fixed_size,Duplicated chunk_id,0,0,PASS,Each chunk must have a unique identifier.
7,fixed_size,Repealed legal chunks,0,0,PASS,Repealed legal provisions must not be indexed for retrieval.


...

## Saved Output Files

,File Type,Path,Purpose
0,Corpus Summary - CSV,C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\corpus_summary_by_chunking_strategy.csv,Summary of retrieval corpora by chunking strategy.
1,Corpus Summary - Excel,C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\corpus_summary_by_chunking_strategy.xlsx,Readable Excel version of corpus summary.
2,Corpus Safety Checks - CSV,C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\corpus_safety_checks_by_chunking_strategy.csv,Detailed safety checks for each retrieval corpus.
3,Corpus Safety Checks - Excel,C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\corpus_safety_checks_by_chunking_strategy.xlsx,Readable Excel version of corpus safety checks.


## Preview — Retrieval Text Samples

,Chunking Strategy,Chunk ID,Source Type,Parent Document ID,Retrieval Text Preview
0,structural,STRUCT_000001,legal_article,d28fed1c707fb8a6b354171cec6a310c1164782775f0d91e9d13d8b8198af462,التصنيف القانوني: التعريفات / الأحكام العامة الباب: التعريفات / الأحكام العامة الفصل: الفصل الأول رقم المادة: الأولى عنوان المادة: المادة الأولى: النص القانوني: التعريفات / الأحكام العامة - الفصل الأول - المادة 1 المصدر: نظام العمل السعودي الباب الأول - التعريفات / الأحكام العامة الفصل الأول المادة ...
1,structural,STRUCT_000002,legal_article,4f2961046b5fffdc637e2b7de4dfb1fd862ccc7b9f3069349681b04ff17df638,التصنيف القانوني: التعريفات / الأحكام العامة الباب: التعريفات / الأحكام العامة الفصل: الفصل الأول رقم المادة: الثانية عنوان المادة: المادة الثانية : النص القانوني: التعريفات / الأحكام العامة - الفصل الأول - المادة 2 المصدر: نظام العمل السعودي الباب الأول - التعريفات / الأحكام العامة الفصل الأول الما...
2,structural,STRUCT_000003,legal_article,82f3c9791caa01e20aff6f25a0e169fbd07f8aa138625fe674d6df6824a31fda,التصنيف القانوني: التعريفات / الأحكام العامة الباب: التعريفات / الأحكام العامة الفصل: الفصل الثاني رقم المادة: الثالثة عنوان المادة: المادة الثالثة: النص القانوني: التعريفات / الأحكام العامة - الفصل الثاني - المادة 3 المصدر: نظام العمل السعودي الباب الأول - التعريفات / الأحكام العامة الفصل الثاني ال...
3,fixed_size,FIXED_000001,legal_article,d28fed1c707fb8a6b354171cec6a310c1164782775f0d91e9d13d8b8198af462,التصنيف القانوني: التعريفات / الأحكام العامة الباب: التعريفات / الأحكام العامة الفصل: الفصل الأول رقم المادة: الأولى عنوان المادة: المادة الأولى: النص القانوني: المصدر: نظام العمل السعودي الباب الأول - التعريفات / الأحكام العامة الفصل الأول المادة الأولى: رقم المادة: 1 حالة المادة: active النص: يسمى...
4,fixed_size,FIXED_000002,legal_article,4f2961046b5fffdc637e2b7de4dfb1fd862ccc7b9f3069349681b04ff17df638,التصنيف القانوني: التعريفات / الأحكام العامة الباب: التعريفات / الأحكام العامة الفصل: الفصل الأول رقم المادة: الثانية عنوان المادة: المادة الثانية : النص القانوني: المصدر: نظام العمل السعودي الباب الأول - التعريفات / الأحكام العامة الفصل الأول المادة الثانية : رقم المادة: 2 حالة المادة: active النص:...
5,fixed_size,FIXED_000003,legal_article,4f2961046b5fffdc637e2b7de4dfb1fd862ccc7b9f3069349681b04ff17df638,التصنيف القانوني: التعريفات / الأحكام العامة الباب: التعريفات / الأحكام العامة الفصل: الفصل الأول رقم المادة: الثانية عنوان المادة: المادة الثانية : النص القانوني: تتقرر للعامل مقابل جهد بذله في العمل، أو مخاطر يتعرض لها في أداء عمله، أو التي تتقرر للعامل لقاء العمل بموجب عقد العمل أو لائحة تنظيم ال...


Stage 06 completed successfully.
CORPORA object is ready.
Available corpora: ['structural', 'fixed_size']
structural: (722, 33)
fixed_size: (719, 33)

Saved output files:
C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\corpus_summary_by_chunking_strategy.csv
C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\corpus_summary_by_chunking_strategy.xlsx
C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\corpus_safety_checks_by_chunking_strategy.csv
C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\corpus_safety_checks_by_chunking_strategy.xlsx


In [11]:

# =========================================================
# Stage 07 - Dependency Check and Model Loading
# =========================================================

try:
    from sentence_transformers import SentenceTransformer, CrossEncoder
    SENTENCE_TRANSFORMERS_AVAILABLE = True
except Exception as e:
    SENTENCE_TRANSFORMERS_AVAILABLE = False
    SentenceTransformer = None
    CrossEncoder = None
    raise ImportError(
        "sentence-transformers is required for Stage 03. "
        "Install it inside your Jupyter environment before running this notebook."
    ) from e

try:
    import chromadb
    CHROMADB_AVAILABLE = True
except Exception as e:
    CHROMADB_AVAILABLE = False
    chromadb = None
    if REQUIRE_CHROMADB:
        raise ImportError(
            "ChromaDB is mandatory in this Stage 03 notebook. "
            "Install it using: pip install chromadb"
        ) from e


def resolve_model_path(config):
    local_path = config.get("local_path")
    if local_path and Path(local_path).exists():
        return str(local_path)
    return config["model_name"]


def load_embedding_model(config):
    model_path = resolve_model_path(config)
    print(f"Loading embedding model [{config['model_key']}]:", model_path)
    model = SentenceTransformer(model_path)
    return model

loaded_embedding_models = {}
for cfg in EMBEDDING_MODELS:
    try:
        loaded_embedding_models[cfg["model_key"]] = {
            "config": cfg,
            "model": load_embedding_model(cfg),
        }
        print("✅ Loaded:", cfg["model_key"])
    except Exception as e:
        print("⚠️ Skipped embedding model:", cfg["model_key"], "| reason:", e)

reranker_model = None
if RERANKER_CONFIG["enabled"]:
    try:
        reranker_path = str(RERANKER_CONFIG["local_path"]) if RERANKER_CONFIG["local_path"].exists() else RERANKER_CONFIG["model_name"]
        print("Loading reranker:", reranker_path)
        reranker_model = CrossEncoder(reranker_path)
        print("✅ Reranker loaded")
    except Exception as e:
        print("⚠️ Reranker skipped:", e)

assert len(loaded_embedding_models) > 0, "No embedding model could be loaded. Check local model paths or internet access."
assert CHROMADB_AVAILABLE, "ChromaDB must be available before continuing."

print("Mandatory ChromaDB check passed.")


Loading embedding model [e5_large]: C:\models\multilingual-e5-large


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 6708.22it/s]


✅ Loaded: e5_large
Loading embedding model [bge_m3]: C:\models\bge-m3


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 46790.86it/s]


✅ Loaded: bge_m3
Loading reranker: C:\models\bge-reranker-v2-m3


Loading weights: 100%|██████████| 393/393 [00:00<00:00, 6814.09it/s]


✅ Reranker loaded
Mandatory ChromaDB check passed.


<div dir="rtl" style="text-align: right; line-height: 2; font-size: 16px;">

<h3>تحميل نماذج التضمين والتحقق من جاهزية ChromaDB</h3>

<p>
توضح هذه الخطوة أنه تم تحميل نماذج التضمين المستخدمة في مرحلة الاسترجاع بنجاح، وهي نموذج 
<span dir="ltr" style="display:inline-block;">multilingual-e5-large</span>
ونموذج 
<span dir="ltr" style="display:inline-block;">bge-m3</span>.
تُستخدم هذه النماذج لتحويل المقاطع النصية والأسئلة إلى تمثيلات رقمية 
<span dir="ltr" style="display:inline-block;">Embeddings</span>
يمكن مقارنتها دلالياً داخل نظام الاسترجاع.
</p>

<p>
كما تم تحميل نموذج إعادة الترتيب 
<span dir="ltr" style="display:inline-block;">bge-reranker-v2-m3</span>
بنجاح. وظيفة هذا النموذج هي إعادة ترتيب النتائج المرشحة بعد الاسترجاع الأولي، بحيث يتم رفع المقاطع الأكثر ارتباطاً بالسؤال إلى المراتب الأولى.
</p>

<p>
بالإضافة إلى ذلك، تم تنفيذ فحص إلزامي للتأكد من توفر 
<span dir="ltr" style="display:inline-block;">ChromaDB</span>
قبل متابعة بناء قاعدة المتجهات. ظهور الرسالة 
<span dir="ltr" style="display:inline-block;">Mandatory ChromaDB check passed</span>
يعني أن مكتبة 
<span dir="ltr" style="display:inline-block;">ChromaDB</span>
مثبتة وجاهزة للاستخدام، وأن المرحلة التالية يمكن أن تبني مجموعات المتجهات وتخزن المقاطع داخل قاعدة بيانات متجهية فعلية.
</p>

<p>
هذه الخطوة مهمة لأنها تمثل نقطة الانتقال من البيانات النصية النظيفة إلى طبقة الاسترجاع الدلالي. بعد نجاحها، يصبح بالإمكان توليد التضمينات، تخزينها داخل 
<span dir="ltr" style="display:inline-block;">ChromaDB</span>
ثم تنفيذ تجارب الاسترجاع باستخدام 
<span dir="ltr" style="display:inline-block;">Dense Retrieval</span>
و
<span dir="ltr" style="display:inline-block;">Hybrid Retrieval</span>
وإعادة الترتيب.
</p>

</div>


<div dir="rtl" style="text-align: right; line-height: 2; font-size: 16px;">

## نماذج التضمين وإعادة الترتيب

يستخدم النوتبوك نماذج تضمين متعددة اللغات مناسبة للنص العربي. إذا توفر نموذجان مثل
<span dir="ltr" style="display:inline-block;">multilingual-e5-large</span>
و
<span dir="ltr" style="display:inline-block;">bge-m3</span>
فسيتم تقييمهما معاً. وإذا تعذر تحميل أحد النماذج، يتم تخطيه وتسجيل السبب دون إيقاف التجربة كاملة.

</div>

In [12]:
# =========================================================
# Stage 08 - Embedding Cache and Encoding Functions
# =========================================================


def safe_key(text: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_\-]+", "_", str(text))


def format_passages(texts, cfg):
    prefix = cfg.get("passage_prefix", "")
    return [prefix + str(t) for t in texts]


def format_query(query, cfg):
    prefix = cfg.get("query_prefix", "")
    return prefix + str(query)


def l2_normalize(x: np.ndarray) -> np.ndarray:
    norms = np.linalg.norm(x, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    return x / norms


def encode_texts(model, texts, batch_size=32, show_progress_bar=True):
    emb = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=show_progress_bar,
        convert_to_numpy=True,
        normalize_embeddings=True,
    )
    emb = np.asarray(emb, dtype="float32")
    return emb


def get_embeddings_for_corpus(model_key, model_info, corpus_name, corpus_df):
    cfg = model_info["config"]
    model = model_info["model"]
    cache_path = CACHE_DIR / f"embeddings_{safe_key(corpus_name)}_{safe_key(model_key)}.npy"
    meta_path = CACHE_DIR / f"embeddings_{safe_key(corpus_name)}_{safe_key(model_key)}.json"

    expected_hash = hashlib.sha256("||".join(corpus_df["retrieval_text"].astype(str).tolist()).encode("utf-8")).hexdigest()

    if cache_path.exists() and meta_path.exists():
        try:
            meta = json.loads(meta_path.read_text(encoding="utf-8"))
            if meta.get("corpus_hash") == expected_hash and meta.get("rows") == len(corpus_df):
                print(f"Loaded cached embeddings: {cache_path.name}")
                return np.load(cache_path)
        except Exception:
            pass

    passages = format_passages(corpus_df["retrieval_text"].tolist(), cfg)
    print(f"Encoding corpus [{corpus_name}] with model [{model_key}] | rows={len(passages)}")
    emb = encode_texts(model, passages, batch_size=16, show_progress_bar=True)
    np.save(cache_path, emb)
    meta_path.write_text(json.dumps({
        "model_key": model_key,
        "model_name": cfg["model_name"],
        "corpus_name": corpus_name,
        "rows": len(corpus_df),
        "dimensions": int(emb.shape[1]),
        "corpus_hash": expected_hash,
    }, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Saved embeddings:", cache_path)
    return emb

# Build embeddings for all model/corpus combinations
EMBEDDING_STORE = {}
for corpus_name, corpus_df in CORPORA.items():
    EMBEDDING_STORE[corpus_name] = {}
    for model_key, model_info in loaded_embedding_models.items():
        EMBEDDING_STORE[corpus_name][model_key] = get_embeddings_for_corpus(
            model_key, model_info, corpus_name, corpus_df
        )

print("Embedding store ready.")
for corpus_name in EMBEDDING_STORE:
    for model_key, emb in EMBEDDING_STORE[corpus_name].items():
        print(corpus_name, model_key, emb.shape)

Loaded cached embeddings: embeddings_structural_e5_large.npy
Loaded cached embeddings: embeddings_structural_bge_m3.npy
Loaded cached embeddings: embeddings_fixed_size_e5_large.npy
Loaded cached embeddings: embeddings_fixed_size_bge_m3.npy
Embedding store ready.
structural e5_large (722, 1024)
structural bge_m3 (722, 1024)
fixed_size e5_large (719, 1024)
fixed_size bge_m3 (719, 1024)


In [13]:
# =========================================================
# Professional Embedding Summary Table
# جدول احترافي لنتائج توليد التضمينات
# =========================================================

import pandas as pd
import numpy as np
from IPython.display import display, HTML
from pathlib import Path

# ---------------------------------------------------------
# تحديد مجلد حفظ التضمينات بطريقة آمنة
# ---------------------------------------------------------

if "EMBEDDING_CACHE_DIR" in globals():
    embedding_cache_dir = EMBEDDING_CACHE_DIR
elif "EMBED_CACHE_DIR" in globals():
    embedding_cache_dir = EMBED_CACHE_DIR
else:
    embedding_cache_dir = PROJECT_DIR / "07_embedding_cache"

embedding_cache_dir = Path(embedding_cache_dir)

# ---------------------------------------------------------
# بناء جدول ملخص التضمينات
# ---------------------------------------------------------

embedding_summary_rows = []

for corpus_name, model_dict in EMBEDDING_STORE.items():
    for model_key, emb in model_dict.items():
        emb_array = np.asarray(emb)

        documents_count = emb_array.shape[0]
        vector_dimension = emb_array.shape[1] if emb_array.ndim == 2 else None
        estimated_memory_mb = round(emb_array.nbytes / (1024 ** 2), 2)

        cache_file = embedding_cache_dir / f"embeddings_{corpus_name}_{model_key}.npy"

        embedding_summary_rows.append({
            "Chunking Strategy": corpus_name,
            "Embedding Model": model_key,
            "Documents / Chunks": documents_count,
            "Vector Dimension": vector_dimension,
            "Embedding Matrix Shape": str(emb_array.shape),
            "Estimated Memory MB": estimated_memory_mb,
            "Cache File": cache_file.name,
            "Cache Saved": "Yes" if cache_file.exists() else "No",
            "Status": "OK" if documents_count > 0 and vector_dimension is not None else "Check",
        })

embedding_summary_df = pd.DataFrame(embedding_summary_rows)

embedding_summary_df = embedding_summary_df.sort_values(
    by=["Chunking Strategy", "Embedding Model"]
).reset_index(drop=True)

# ---------------------------------------------------------
# حفظ التقرير
# ---------------------------------------------------------

embedding_summary_df.to_csv(
    STAGE03_DIR / "embedding_generation_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

embedding_summary_df.to_excel(
    STAGE03_DIR / "embedding_generation_summary.xlsx",
    index=False
)

# ---------------------------------------------------------
# عرض عنوان احترافي
# ---------------------------------------------------------

display(HTML("""
<div dir="rtl" style="text-align:right; line-height:2; font-size:17px;">
<h3>ملخص توليد التضمينات النصية</h3>
<p>
يوضح الجدول التالي نتائج تحويل المقاطع النصية إلى متجهات رقمية دلالية باستخدام نماذج التضمين المختلفة.
يعرض الجدول عدد المقاطع، أبعاد المتجهات، حجم الذاكرة التقريبي، وحالة حفظ ملفات التضمين في مجلد الكاش.
</p>
</div>
"""))

# ---------------------------------------------------------
# تنسيق احترافي للجدول
# ---------------------------------------------------------

styled_embedding_summary = (
    embedding_summary_df.style
    .set_caption("Embedding Generation Summary")
    .format({
        "Documents / Chunks": "{:,.0f}",
        "Vector Dimension": "{:,.0f}",
        "Estimated Memory MB": "{:,.2f}",
    })
    .set_table_styles([
        {
            "selector": "caption",
            "props": [
                ("caption-side", "top"),
                ("font-size", "18px"),
                ("font-weight", "bold"),
                ("text-align", "center"),
                ("padding", "10px"),
            ],
        },
        {
            "selector": "th",
            "props": [
                ("font-weight", "bold"),
                ("text-align", "center"),
                ("background-color", "#F2F2F2"),
                ("border", "1px solid #CCCCCC"),
                ("padding", "8px"),
            ],
        },
        {
            "selector": "td",
            "props": [
                ("text-align", "center"),
                ("border", "1px solid #DDDDDD"),
                ("padding", "8px"),
            ],
        },
        {
            "selector": "table",
            "props": [
                ("border-collapse", "collapse"),
                ("width", "100%"),
                ("font-size", "14px"),
            ],
        },
    ])
    .map(
        lambda v: "background-color: #DFF2E1; font-weight: bold;" if v == "OK" else "",
        subset=["Status"]
    )
    .map(
        lambda v: "background-color: #DFF2E1; font-weight: bold;" if v == "Yes" else "background-color: #F8D7DA;",
        subset=["Cache Saved"]
    )
)

display(styled_embedding_summary)

print("Embedding summary saved to:")
print(STAGE03_DIR / "embedding_generation_summary.csv")
print(STAGE03_DIR / "embedding_generation_summary.xlsx")

,Chunking Strategy,Embedding Model,Documents / Chunks,Vector Dimension,Embedding Matrix Shape,Estimated Memory MB,Cache File,Cache Saved,Status
0,fixed_size,bge_m3,719,"1,024","(719, 1024)",2.81,embeddings_fixed_size_bge_m3.npy,Yes,OK
1,fixed_size,e5_large,719,"1,024","(719, 1024)",2.81,embeddings_fixed_size_e5_large.npy,Yes,OK
2,structural,bge_m3,722,"1,024","(722, 1024)",2.82,embeddings_structural_bge_m3.npy,Yes,OK
3,structural,e5_large,722,"1,024","(722, 1024)",2.82,embeddings_structural_e5_large.npy,Yes,OK


Embedding summary saved to:
C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\embedding_generation_summary.csv
C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\embedding_generation_summary.xlsx


### تفسير نتائج توليد التضمينات النصية

يعرض الجدول السابق ملخصًا لمرحلة توليد التضمينات النصية (Embedding Generation)، وهي المرحلة التي تم فيها تحويل المقاطع النصية الناتجة من استراتيجيات التقسيم إلى متجهات رقمية دلالية قابلة للتخزين والاسترجاع داخل قاعدة المتجهات ChromaDB.

تم تنفيذ هذه العملية على استراتيجيتي التقسيم المستخدمتين في المشروع، وهما **Structural Chunking** و **Fixed-size Chunking Baseline**، كما تم استخدام نموذجين للتضمين النصي هما **bge-m3** و **multilingual-e5-large**. وبذلك تم إنتاج أربع مجموعات من التضمينات، بحيث يمكن لاحقًا مقارنة أثر كل استراتيجية تقسيم وكل نموذج تضمين على جودة الاسترجاع.

أظهرت النتائج أن عدد المقاطع في استراتيجية **Fixed-size Chunking** بلغ 719 مقطعًا، بينما بلغ عدد المقاطع في استراتيجية **Structural Chunking** عدد 722 مقطعًا. وقد تم توليد التضمينات لكل هذه المقاطع باستخدام النموذجين المختارين.

يشير عمود **Vector Dimension** إلى عدد الأبعاد في كل متجه تمثيلي. وقد أظهرت النتائج أن كلا النموذجين ينتجان متجهات ذات 1024 بُعدًا. على سبيل المثال، الشكل `(719, 1024)` يعني أن هناك 719 مقطعًا، وكل مقطع تم تمثيله بمتجه رقمي مكوّن من 1024 بُعدًا. وبالمثل، الشكل `(722, 1024)` يعني أن هناك 722 مقطعًا تم تحويلها إلى متجهات دلالية بنفس عدد الأبعاد.

كما يوضح عمود **Estimated Memory MB** الحجم التقريبي لمصفوفة التضمينات في الذاكرة. وقد بلغ الحجم التقريبي 2.81 ميغابايت لمصفوفات استراتيجية **Fixed-size Chunking**، و2.82 ميغابايت لمصفوفات استراتيجية **Structural Chunking**. ويُعد هذا الحجم مناسبًا جدًا للتجارب المحلية، كما أنه يؤكد أن حجم البيانات في هذه المرحلة قابل للإدارة بكفاءة داخل بيئة Jupyter وChromaDB.

يوضح عمود **Cache Saved** أن جميع ملفات التضمينات تم حفظها بنجاح بصيغة `.npy`. وتعد هذه الخطوة مهمة لأنها تسمح بإعادة استخدام التضمينات لاحقًا دون الحاجة إلى إعادة توليدها في كل مرة يتم فيها تشغيل النوتبوك. وهذا يقلل زمن التنفيذ، ويجعل التجارب اللاحقة أكثر استقرارًا وقابلية للتكرار.

كما يشير ظهور الحالة **OK** في جميع الصفوف إلى أن عملية توليد التضمينات تمت بنجاح لجميع التركيبات، دون وجود مشاكل في عدد المقاطع أو أبعاد المتجهات أو حفظ ملفات التضمينات.

بناءً على هذه النتائج، أصبحت التضمينات النصية جاهزة للاستخدام في المرحلة التالية، وهي بناء مجموعات ChromaDB وتنفيذ تجارب الاسترجاع. وستُستخدم هذه التضمينات لاحقًا لمقارنة أداء نماذج التضمين واستراتيجيات التقسيم باستخدام مقاييس تقييم مثل Recall@1 وRecall@3 وRecall@5 وMRR وnDCG@5 وزمن الاستجابة.

### الملفات الناتجة من هذه المرحلة

نتج عن هذه المرحلة حفظ ملفات التضمينات الخاصة بكل تركيبة من استراتيجية التقسيم ونموذج التضمين، إضافة إلى ملفات ملخص النتائج. وتشمل المخرجات الرئيسية:

* `embeddings_fixed_size_bge_m3.npy`
* `embeddings_fixed_size_e5_large.npy`
* `embeddings_structural_bge_m3.npy`
* `embeddings_structural_e5_large.npy`
* `embedding_generation_summary.csv`
* `embedding_generation_summary.xlsx`

تمثل هذه الملفات الأساس العملي للمرحلة التالية من المشروع، حيث سيتم استخدامها لبناء قاعدة المتجهات ChromaDB وتقييم جودة الاسترجاع الدلالي والهجين.


In [14]:

# =========================================================
# Stage 09 - Mandatory Persistent ChromaDB Vector Database
# =========================================================

# In this version, ChromaDB is not optional.
# The vector database is built and then used directly in dense retrieval experiments.

assert CHROMADB_AVAILABLE, "ChromaDB is mandatory but not available."
assert BUILD_CHROMA_DB, "BUILD_CHROMA_DB must be True in this ChromaDB-based stage."


def sanitize_metadata(row):
    allowed = {}
    for col in [
        "doc_index", "chunking_strategy_eval", "chunk_id", "parent_document_id", "source_type", "source_name",
        "article_number", "article_number_label", "article_number_int", "article_status",
        "article_title", "legal_category", "source_url", "rag_usage", "chunk_word_len", "chunk_char_len"
    ]:
        value = row.get(col, "")
        if pd.isna(value):
            value = ""
        # Chroma metadata supports str, int, float, bool. Store all as simple values.
        if col in ["doc_index", "chunk_word_len", "chunk_char_len"]:
            try:
                allowed[col] = int(value)
            except Exception:
                allowed[col] = 0
        else:
            allowed[col] = str(value)
    return allowed


def collection_name_for(corpus_name, model_key):
    # Keep name deterministic and short enough for Chroma.
    return f"rag_{safe_key(corpus_name)}_{safe_key(model_key)}"[:60]

client = chromadb.PersistentClient(path=str(VECTOR_DIR))
CHROMA_COLLECTIONS = {}
chroma_manifest_rows = []

for corpus_name, corpus_df in CORPORA.items():
    CHROMA_COLLECTIONS[corpus_name] = {}
    for model_key, emb in EMBEDDING_STORE[corpus_name].items():
        collection_name = collection_name_for(corpus_name, model_key)

        if REFRESH_CHROMA_COLLECTIONS:
            try:
                client.delete_collection(collection_name)
                print("Deleted old collection:", collection_name)
            except Exception:
                pass

        collection = client.get_or_create_collection(
            name=collection_name,
            metadata={
                "hnsw:space": "cosine",
                "project": "saudi_labor_law_voice_agent",
                "stage": "stage03_rag_retrieval",
                "corpus": corpus_name,
                "embedding_model": model_key,
            },
        )

        ids = corpus_df["chunk_id"].astype(str).tolist()
        assert len(ids) == len(set(ids)), f"Duplicate chunk_id values found in {corpus_name}. Chroma IDs must be unique."

        docs = corpus_df["retrieval_text"].fillna("").astype(str).tolist()
        metas = [sanitize_metadata(row) for _, row in corpus_df.iterrows()]
        vectors = np.asarray(emb, dtype="float32").tolist()

        batch_size = 128
        for start in range(0, len(ids), batch_size):
            end = start + batch_size
            collection.upsert(
                ids=ids[start:end],
                documents=docs[start:end],
                metadatas=metas[start:end],
                embeddings=vectors[start:end],
            )

        count_after = collection.count()
        assert count_after == len(ids), (
            f"Chroma count mismatch for {collection_name}: "
            f"expected {len(ids)}, got {count_after}"
        )

        CHROMA_COLLECTIONS[corpus_name][model_key] = collection
        chroma_manifest_rows.append({
            "corpus": corpus_name,
            "model_key": model_key,
            "collection_name": collection_name,
            "records_expected": len(ids),
            "records_in_chroma": count_after,
            "vector_db_path": str(VECTOR_DIR),
            "status": "OK",
        })
        print("✅ Chroma collection ready:", collection_name, "records:", count_after)

chroma_manifest = pd.DataFrame(chroma_manifest_rows)
assert not chroma_manifest.empty, "No ChromaDB collections were created."
assert (chroma_manifest["status"] == "OK").all(), "Some ChromaDB collections failed."

chroma_manifest.to_csv(STAGE03_DIR / "chroma_collections_manifest.csv", index=False, encoding="utf-8-sig")
chroma_manifest.to_excel(STAGE03_DIR / "chroma_collections_manifest.xlsx", index=False)
with open(STAGE03_DIR / "chroma_collections_manifest.json", "w", encoding="utf-8") as f:
    json.dump(chroma_manifest.to_dict(orient="records"), f, ensure_ascii=False, indent=2)

display(chroma_manifest)
print("Mandatory ChromaDB build completed successfully.")


Deleted old collection: rag_structural_e5_large
✅ Chroma collection ready: rag_structural_e5_large records: 722
Deleted old collection: rag_structural_bge_m3
✅ Chroma collection ready: rag_structural_bge_m3 records: 722
Deleted old collection: rag_fixed_size_e5_large
✅ Chroma collection ready: rag_fixed_size_e5_large records: 719
Deleted old collection: rag_fixed_size_bge_m3
✅ Chroma collection ready: rag_fixed_size_bge_m3 records: 719


,corpus,model_key,collection_name,records_expected,records_in_chroma,vector_db_path,status
0,structural,e5_large,rag_structural_e5_large,722,722,C:\Users\PC\Desktop\data collection code\saudi...,OK
1,structural,bge_m3,rag_structural_bge_m3,722,722,C:\Users\PC\Desktop\data collection code\saudi...,OK
2,fixed_size,e5_large,rag_fixed_size_e5_large,719,719,C:\Users\PC\Desktop\data collection code\saudi...,OK
3,fixed_size,bge_m3,rag_fixed_size_bge_m3,719,719,C:\Users\PC\Desktop\data collection code\saudi...,OK


Mandatory ChromaDB build completed successfully.


In [15]:

# =========================================================
# Stage 10 - ChromaDB Dense, BM25, Hybrid, and Reranker Retrieval
# =========================================================

class SimpleBM25:
    def __init__(self, k1=1.5, b=0.75):
        self.k1 = k1
        self.b = b
        self.corpus_tokens = []
        self.doc_len = []
        self.avgdl = 0.0
        self.idf = {}

    def fit(self, texts):
        self.corpus_tokens = [tokenize_arabic(t) for t in texts]
        self.doc_len = [len(toks) for toks in self.corpus_tokens]
        self.avgdl = float(np.mean(self.doc_len)) if self.doc_len else 1.0
        df = Counter()
        for toks in self.corpus_tokens:
            for term in set(toks):
                df[term] += 1
        n = len(self.corpus_tokens)
        self.idf = {term: math.log((n - freq + 0.5) / (freq + 0.5) + 1.0) for term, freq in df.items()}

    def score_one(self, query_tokens, doc_idx):
        toks = self.corpus_tokens[doc_idx]
        tf = Counter(toks)
        dl = self.doc_len[doc_idx] if self.doc_len else 0
        score = 0.0
        for term in query_tokens:
            if term not in self.idf:
                continue
            f = tf.get(term, 0)
            denom = f + self.k1 * (1 - self.b + self.b * dl / (self.avgdl or 1.0))
            score += self.idf[term] * (f * (self.k1 + 1)) / (denom or 1.0)
        return score

    def scores(self, query):
        q_tokens = tokenize_arabic(query)
        return np.array([self.score_one(q_tokens, i) for i in range(len(self.corpus_tokens))], dtype="float32")


def minmax_norm(scores):
    scores = np.asarray(scores, dtype="float32")
    if scores.size == 0:
        return scores
    mn, mx = float(scores.min()), float(scores.max())
    if mx - mn < 1e-9:
        return np.zeros_like(scores)
    return (scores - mn) / (mx - mn)


def minmax_dict(score_dict, keys):
    values = np.array([float(score_dict.get(k, 0.0)) for k in keys], dtype="float32")
    norm_values = minmax_norm(values)
    return {k: float(v) for k, v in zip(keys, norm_values)}


def build_bm25_indexes():
    bm25_indexes = {}
    for corpus_name, corpus_df in CORPORA.items():
        bm25 = SimpleBM25()
        bm25.fit(corpus_df["retrieval_text"].tolist())
        bm25_indexes[corpus_name] = bm25
    return bm25_indexes

BM25_INDEXES = build_bm25_indexes()


def make_result(corpus_df, idx, score, rank, extra=None):
    row = corpus_df.iloc[int(idx)]
    result = {
        "rank": int(rank),
        "doc_index": int(idx),
        "score": float(score),
        "chunk_id": str(row.get("chunk_id", "")),
        "parent_document_id": str(row.get("parent_document_id", "")),
        "source_type": str(row.get("source_type", "")),
        "article_number_int": str(row.get("article_number_int", "")),
        "article_number_label": str(row.get("article_number_label", "")),
        "article_title": str(row.get("article_title", "")),
        "question": str(row.get("question", "")),
        "text": str(row.get("retrieval_text", "")),
    }
    if extra:
        result.update(extra)
    return result


def chroma_distance_to_similarity(distance):
    # With hnsw:space='cosine', Chroma returns a cosine distance.
    # Higher score is better, so similarity = 1 - distance.
    try:
        return float(1.0 - float(distance))
    except Exception:
        return 0.0


def encode_query_for_model(query, model_key):
    model_info = loaded_embedding_models[model_key]
    cfg = model_info["config"]
    model = model_info["model"]
    q_text = format_query(query, cfg)
    return encode_texts(model, [q_text], batch_size=1, show_progress_bar=False)[0]


def chroma_dense_retrieve(query, corpus_name, model_key, top_k=5):
    assert corpus_name in CHROMA_COLLECTIONS, f"Missing Chroma corpus: {corpus_name}"
    assert model_key in CHROMA_COLLECTIONS[corpus_name], f"Missing Chroma collection for {corpus_name}/{model_key}"

    corpus_df = CORPORA[corpus_name]
    collection = CHROMA_COLLECTIONS[corpus_name][model_key]
    q_emb = encode_query_for_model(query, model_key)

    query_result = collection.query(
        query_embeddings=[q_emb.astype(float).tolist()],
        n_results=min(top_k, collection.count()),
        include=["documents", "metadatas", "distances"],
    )

    ids = query_result.get("ids", [[]])[0]
    docs = query_result.get("documents", [[]])[0]
    metas = query_result.get("metadatas", [[]])[0]
    distances = query_result.get("distances", [[]])[0]

    results = []
    for rank, (item_id, doc, meta, distance) in enumerate(zip(ids, docs, metas, distances), start=1):
        doc_index = int(meta.get("doc_index", -1)) if meta else -1
        if doc_index < 0 or doc_index >= len(corpus_df):
            # Rare fallback if metadata is not available.
            matches = corpus_df.index[corpus_df["chunk_id"].astype(str) == str(item_id)].tolist()
            doc_index = int(matches[0]) if matches else 0
        score = chroma_distance_to_similarity(distance)
        results.append(make_result(corpus_df, doc_index, score, rank, extra={
            "retrieval_backend": "chromadb",
            "chroma_collection": collection.name,
            "chroma_id": str(item_id),
            "chroma_distance": float(distance),
            "dense_score": float(score),
        }))
    return results


def dense_retrieve(query, corpus_name, model_key, top_k=5):
    # Mandatory dense retrieval through ChromaDB.
    return chroma_dense_retrieve(query, corpus_name, model_key, top_k=top_k)


def bm25_retrieve(query, corpus_name, top_k=5):
    corpus_df = CORPORA[corpus_name]
    scores = BM25_INDEXES[corpus_name].scores(query)
    idxs = np.argsort(-scores)[:top_k]
    return [make_result(corpus_df, i, scores[i], rank + 1, extra={"retrieval_backend": "bm25"}) for rank, i in enumerate(idxs)]


def hybrid_retrieve(query, corpus_name, model_key, alpha=0.80, top_k=5, candidate_k=50):
    corpus_df = CORPORA[corpus_name]

    # Dense candidates must come from ChromaDB.
    dense_candidates = chroma_dense_retrieve(
        query=query,
        corpus_name=corpus_name,
        model_key=model_key,
        top_k=min(candidate_k, len(corpus_df)),
    )
    dense_score_map = {int(r["doc_index"]): float(r.get("dense_score", r.get("score", 0.0))) for r in dense_candidates}

    # BM25 candidates are lexical candidates.
    bm25_scores = BM25_INDEXES[corpus_name].scores(query)
    bm25_top_idxs = np.argsort(-bm25_scores)[:candidate_k].astype(int).tolist()
    bm25_score_map = {int(i): float(bm25_scores[int(i)]) for i in bm25_top_idxs}

    candidate_idxs = sorted(set(dense_score_map.keys()) | set(bm25_score_map.keys()))
    dense_norm = minmax_dict(dense_score_map, candidate_idxs)
    bm25_norm = minmax_dict(bm25_score_map, candidate_idxs)

    fused_rows = []
    for i in candidate_idxs:
        fused = float(alpha) * dense_norm.get(i, 0.0) + (1.0 - float(alpha)) * bm25_norm.get(i, 0.0)
        fused_rows.append((i, fused))

    fused_rows = sorted(fused_rows, key=lambda x: x[1], reverse=True)[:top_k]

    return [
        make_result(corpus_df, i, fused, rank + 1, extra={
            "retrieval_backend": "chromadb+bm25",
            "dense_score": float(dense_score_map.get(i, 0.0)),
            "bm25_score": float(bm25_score_map.get(i, 0.0)),
            "dense_norm": float(dense_norm.get(i, 0.0)),
            "bm25_norm": float(bm25_norm.get(i, 0.0)),
            "alpha": float(alpha),
        })
        for rank, (i, fused) in enumerate(fused_rows)
    ]


def hybrid_rerank_retrieve(query, corpus_name, model_key, alpha=0.80, top_k=5, candidate_k=50):
    candidates = hybrid_retrieve(query, corpus_name, model_key, alpha=alpha, top_k=candidate_k, candidate_k=candidate_k)
    if reranker_model is None or len(candidates) == 0:
        return candidates[:top_k]
    pairs = [[query, r["text"]] for r in candidates]
    rerank_scores = reranker_model.predict(pairs)
    for r, s in zip(candidates, rerank_scores):
        r["rerank_score"] = float(s)
        r["retrieval_backend"] = "chromadb+bm25+reranker"
    candidates = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)[:top_k]
    for rank, r in enumerate(candidates, start=1):
        r["rank"] = rank
        r["score"] = r.get("rerank_score", r.get("score", 0.0))
    return candidates

print("Retrieval functions ready. Dense retrieval is executed through ChromaDB.")


Retrieval functions ready. Dense retrieval is executed through ChromaDB.


In [16]:
# =========================================================
# Stage 11 - Evaluation Metrics
# =========================================================

def result_is_relevant(eval_row, result):
    gold_doc_ids = parse_gold_document_ids(eval_row)
    gold_article_nums = parse_gold_article_numbers(eval_row)

    result_parent = str(result.get("parent_document_id", "")).strip()
    if result_parent and result_parent in gold_doc_ids:
        return True

    # For legal article questions, allow article-number matching as backup
    result_nums = set(extract_numbers(result.get("article_number_int", ""))) | set(extract_numbers(result.get("article_number_label", "")))
    if gold_article_nums and result_nums and len(gold_article_nums & result_nums) > 0:
        return True

    return False


def relevance_vector(eval_row, results, k=5):
    rel = []
    for r in results[:k]:
        rel.append(1 if result_is_relevant(eval_row, r) else 0)
    while len(rel) < k:
        rel.append(0)
    return rel


def reciprocal_rank(rel):
    for i, v in enumerate(rel, start=1):
        if v:
            return 1.0 / i
    return 0.0


def dcg_at_k(rel):
    return sum((2**v - 1) / math.log2(i + 2) for i, v in enumerate(rel))


def ndcg_at_k(rel):
    ideal = sorted(rel, reverse=True)
    ideal_dcg = dcg_at_k(ideal)
    if ideal_dcg == 0:
        return 0.0
    return dcg_at_k(rel) / ideal_dcg


def summarize_eval_rows(detail_df, label_cols):
    if detail_df.empty:
        return pd.DataFrame()
    group_cols = label_cols
    summary = (
        detail_df.groupby(group_cols)
        .agg(
            questions=("eval_id", "count"),
            recall_at_1=("hit_at_1", "mean"),
            recall_at_3=("hit_at_3", "mean"),
            recall_at_5=("hit_at_5", "mean"),
            mrr=("rr", "mean"),
            ndcg_at_5=("ndcg_at_5", "mean"),
            avg_latency_sec=("latency_sec", "mean"),
        )
        .reset_index()
    )
    for c in ["recall_at_1", "recall_at_3", "recall_at_5", "mrr", "ndcg_at_5", "avg_latency_sec"]:
        summary[c] = summary[c].round(4)
    return summary

print("Metric functions ready.")

Metric functions ready.


In [16]:
# =========================================================
# Stage 12 - Build Evaluation Subsets
# =========================================================

# In-scope retrieval evaluation only: FAQ + legal article retrieval.
df_eval_in_scope = df_eval[~df_eval["is_out_of_scope_bool"]].copy()
df_eval_in_scope = df_eval_in_scope[df_eval_in_scope["eval_type"].isin(["faq_retrieval", "legal_article_retrieval"])].copy()

def has_gold(row):
    return bool(parse_gold_document_ids(row) or parse_gold_article_numbers(row))

df_eval_in_scope["has_gold"] = df_eval_in_scope.apply(has_gold, axis=1)
missing_gold = df_eval_in_scope[~df_eval_in_scope["has_gold"]]

print("In-scope evaluation rows:", len(df_eval_in_scope))
print("Missing gold rows:", len(missing_gold))
display(df_eval_in_scope["eval_type"].value_counts().to_frame("count"))

assert len(missing_gold) == 0, "Some in-scope evaluation rows have no gold references."

if MAX_EVAL_ROWS is not None:
    df_eval_run = df_eval_in_scope.sample(n=min(MAX_EVAL_ROWS, len(df_eval_in_scope)), random_state=RANDOM_SEED).reset_index(drop=True)
else:
    df_eval_run = df_eval_in_scope.reset_index(drop=True)

print("Rows used in retrieval evaluation:", len(df_eval_run))

In-scope evaluation rows: 223
Missing gold rows: 0


,count
eval_type,
faq_retrieval,199
legal_article_retrieval,24


Rows used in retrieval evaluation: 223


In [17]:
# =========================================================
# GPU Acceleration Check before Stage 13 - Robust Version
# تجهيز GPU قبل تشغيل تجارب الاسترجاع
# =========================================================

import torch
import gc

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

print("CUDA available:", torch.cuda.is_available())
print("Selected device:", DEVICE)

if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))
    print("GPU memory allocated GB:", round(torch.cuda.memory_allocated(0) / 1024**3, 2))
    print("GPU memory reserved GB:", round(torch.cuda.memory_reserved(0) / 1024**3, 2))

# تنظيف ذاكرة الكرت قبل التجارب
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


def move_possible_model_to_device(obj, device, name="model"):
    """
    يحاول نقل النموذج إلى GPU سواء كان النموذج مباشرة أو داخل dict.
    """
    # الحالة 1: الكائن نفسه موديل وله .to()
    if hasattr(obj, "to"):
        obj.to(device)
        print(f"✅ {name} moved directly to {device}")
        return True

    # الحالة 2: الكائن dict وفيه موديل داخلي
    if isinstance(obj, dict):
        possible_keys = ["model", "embedding_model", "sentence_transformer", "encoder"]

        for key in possible_keys:
            if key in obj and hasattr(obj[key], "to"):
                obj[key].to(device)
                print(f"✅ {name}['{key}'] moved to {device}")
                return True

        print(f"⚠️ {name} is a dict, but no movable model object was found inside it.")
        print("Available keys:", list(obj.keys()))
        return False

    print(f"⚠️ Could not move {name}. Object type:", type(obj))
    return False


# نقل نماذج التضمين إلى GPU إذا كانت موجودة
if "loaded_embedding_models" in globals():
    for model_key, model_obj in loaded_embedding_models.items():
        move_possible_model_to_device(
            model_obj,
            DEVICE,
            name=f"embedding model [{model_key}]"
        )
else:
    print("⚠️ loaded_embedding_models not found.")


# نقل reranker إلى GPU
if "reranker_model" in globals() and reranker_model is not None:
    try:
        if hasattr(reranker_model, "model"):
            reranker_model.model.to(DEVICE)
            print("✅ Reranker internal model moved to", DEVICE)
        elif hasattr(reranker_model, "to"):
            reranker_model.to(DEVICE)
            print("✅ Reranker moved directly to", DEVICE)
        else:
            print("⚠️ Reranker object has no .to() method.")
    except Exception as e:
        print("⚠️ Could not move reranker to GPU:", e)
else:
    print("⚠️ reranker_model not found or is None.")

print("\nGPU preparation completed.")

if torch.cuda.is_available():
    print("GPU memory allocated GB after move:", round(torch.cuda.memory_allocated(0) / 1024**3, 2))
    print("GPU memory reserved GB after move:", round(torch.cuda.memory_reserved(0) / 1024**3, 2))

CUDA available: True
Selected device: cuda
GPU name: NVIDIA GeForce RTX 5090
GPU memory allocated GB: 6.32
GPU memory reserved GB: 6.33
✅ embedding model [e5_large]['model'] moved to cuda
✅ embedding model [bge_m3]['model'] moved to cuda
✅ Reranker internal model moved to cuda

GPU preparation completed.
GPU memory allocated GB after move: 6.32
GPU memory reserved GB after move: 6.33


In [18]:
# =========================================================
# ٍstage 12 Fix FAQ Retrieval Evaluation Set
# إصلاح تقييم FAQ بحيث تكون المراجع الذهبية موجودة داخل ChromaDB
# مع تقليل المطابقة النصية المباشرة في أسئلة FAQ
# =========================================================

import pandas as pd
import numpy as np
import re
from pathlib import Path
from IPython.display import display, HTML

# تحسين عرض الجداول داخل النوتبوك
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

# ---------------------------------------------------------
# 0) Helper functions
# ---------------------------------------------------------

def first_existing_col(df, candidates):
    """
    Return the first existing column from a list of possible names.
    """
    for col in candidates:
        if col in df.columns:
            return col
    return None


def normalize_arabic_for_match(text):
    """
    Normalize Arabic text for simple exact-match diagnostics.
    """
    text = "" if pd.isna(text) else str(text)
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)
    text = re.sub(r"[^\u0600-\u06FFa-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip().lower()
    return text


def simple_arabic_paraphrase_question(q):
    """
    Rewrite FAQ evaluation questions to reduce exact-match leakage.

    Important:
    - The indexed FAQ answer remains unchanged.
    - Only the evaluation query is rewritten.
    - This helps evaluate semantic retrieval instead of direct text matching.
    """
    q = "" if pd.isna(q) else str(q).strip()
    q = re.sub(r"\s+", " ", q)
    q_clean = q.strip(" ؟?.")

    if not q_clean:
        return q_clean

    replacements = [
        ("هل يحق", "هل مسموح نظاماً"),
        ("هل يجوز", "هل مسموح"),
        ("هل يمكن", "هل يستطيع"),
        ("هل يسمح", "هل مسموح"),
        ("هل توفر", "أريد معرفة ما إذا كانت توفر"),
        ("هل يوجد", "أريد معرفة ما إذا كان يوجد"),
        ("هل", "أريد معرفة ما إذا كان"),
        ("ما هي", "أريد معرفة"),
        ("ما هو", "أريد معرفة"),
        ("متى", "في أي حالة"),
        ("كيف", "ما طريقة"),
        ("كم", "ما مقدار"),
        ("أين", "في أي مكان"),
    ]

    new_q = q_clean

    for old, new in replacements:
        if new_q.startswith(old):
            new_q = new_q.replace(old, new, 1)
            break
    else:
        new_q = "أريد توضيح الحكم النظامي بخصوص: " + q_clean

    return new_q.strip() + "؟"


# ---------------------------------------------------------
# 1) Collect indexed parent document IDs from CORPORA
# ---------------------------------------------------------

indexed_parent_ids = set()

for corpus_name, corpus_df in CORPORA.items():
    if "parent_document_id" in corpus_df.columns:
        indexed_parent_ids |= set(
            corpus_df["parent_document_id"]
            .astype(str)
            .str.strip()
        )

print("Indexed parent document IDs:", len(indexed_parent_ids))

assert len(indexed_parent_ids) > 0, "No indexed parent_document_id values found in CORPORA."


# ---------------------------------------------------------
# 2) Load FAQ indexing set from Stage 02
# ---------------------------------------------------------

faq_indexing_path = FINAL_DIR / "hrsd_faq_indexing_set.csv"

assert faq_indexing_path.exists(), f"FAQ indexing file not found: {faq_indexing_path}"

df_faq_indexed_source = pd.read_csv(faq_indexing_path)

print("FAQ indexing source shape:", df_faq_indexed_source.shape)
display(df_faq_indexed_source.head(3))


# ---------------------------------------------------------
# 3) Detect important columns safely
# ---------------------------------------------------------

faq_id_col = first_existing_col(
    df_faq_indexed_source,
    ["document_unit_id", "parent_document_id", "doc_id", "id"]
)

faq_question_col = first_existing_col(
    df_faq_indexed_source,
    ["question", "faq_question", "title", "user_question", "retrieval_text", "text_for_indexing"]
)

assert faq_id_col is not None, "Could not find FAQ document ID column."
assert faq_question_col is not None, "Could not find FAQ question/text column."

print("FAQ ID column:", faq_id_col)
print("FAQ question column:", faq_question_col)


# ---------------------------------------------------------
# 4) Keep only FAQ records that actually exist in indexed corpus
# ---------------------------------------------------------

df_faq_indexed_source[faq_id_col] = (
    df_faq_indexed_source[faq_id_col]
    .astype(str)
    .str.strip()
)

df_faq_eval_fixed_source = df_faq_indexed_source[
    df_faq_indexed_source[faq_id_col].isin(indexed_parent_ids)
].copy()

df_faq_eval_fixed_source = df_faq_eval_fixed_source.drop_duplicates(
    subset=[faq_id_col]
)

print("FAQ records available in indexed corpus:", len(df_faq_eval_fixed_source))

assert len(df_faq_eval_fixed_source) > 0, "No FAQ indexed records found inside ChromaDB corpus."


# ---------------------------------------------------------
# 5) Match previous FAQ evaluation size if possible
# ---------------------------------------------------------

TARGET_FAQ_EVAL_SIZE = 131

if len(df_faq_eval_fixed_source) >= TARGET_FAQ_EVAL_SIZE:
    df_faq_eval_fixed_source = df_faq_eval_fixed_source.sample(
        n=TARGET_FAQ_EVAL_SIZE,
        random_state=42
    ).reset_index(drop=True)
else:
    df_faq_eval_fixed_source = df_faq_eval_fixed_source.reset_index(drop=True)

print("Final fixed FAQ retrieval evaluation size:", len(df_faq_eval_fixed_source))


# ---------------------------------------------------------
# 6) Build fixed FAQ retrieval evaluation rows
# ---------------------------------------------------------

faq_eval_fixed_rows = []

for i, row in df_faq_eval_fixed_source.iterrows():
    gold_id = str(row[faq_id_col]).strip()
    original_question = str(row[faq_question_col]).strip()

    faq_eval_fixed_rows.append({
        "eval_id": f"FAQ_INDEXED_RETRIEVAL_{i+1:04d}",
        "eval_type": "faq_retrieval",
        "question": original_question,
        "gold_document_unit_id": gold_id,
        "gold_document_unit_ids": gold_id,
        "gold_article_number": "",
        "gold_article_numbers": "",
        "evaluation_note": "FAQ retrieval evaluation uses FAQ documents that are indexed in ChromaDB."
    })

df_faq_eval_fixed = pd.DataFrame(faq_eval_fixed_rows)

print("Fixed FAQ evaluation rows:", df_faq_eval_fixed.shape)


# ---------------------------------------------------------
# 7) Keep legal retrieval questions from existing evaluation
# ---------------------------------------------------------

if "df_eval" in globals():
    evaluation_source_df = df_eval.copy()
elif "df_eval_run" in globals():
    evaluation_source_df = df_eval_run.copy()
else:
    raise ValueError("Neither df_eval nor df_eval_run found. Please load evaluation dataset first.")

legal_eval_fixed = evaluation_source_df[
    evaluation_source_df["eval_type"]
    .astype(str)
    .str.contains("legal", case=False, na=False)
].copy()

print("Legal retrieval evaluation size:", len(legal_eval_fixed))

assert len(legal_eval_fixed) > 0, "No legal retrieval evaluation questions found."


# ---------------------------------------------------------
# 8) Build FAIR FAQ paraphrase evaluation set
# استخدام FAQ بصياغة معدلة بدل السؤال المطابق نصياً
# ---------------------------------------------------------

df_faq_eval_paraphrased = df_faq_eval_fixed.copy()

df_faq_eval_paraphrased["original_question"] = df_faq_eval_paraphrased["question"]

df_faq_eval_paraphrased["question"] = df_faq_eval_paraphrased["question"].apply(
    simple_arabic_paraphrase_question
)

df_faq_eval_paraphrased["eval_type"] = "faq_paraphrase_retrieval"

df_faq_eval_paraphrased["same_as_original_after_normalization"] = (
    df_faq_eval_paraphrased["original_question"].apply(normalize_arabic_for_match)
    == df_faq_eval_paraphrased["question"].apply(normalize_arabic_for_match)
)

faq_exact_match_count = int(
    df_faq_eval_paraphrased["same_as_original_after_normalization"].sum()
)

faq_exact_match_ratio = round(
    df_faq_eval_paraphrased["same_as_original_after_normalization"].mean(), 4
) if len(df_faq_eval_paraphrased) else 0

print("FAQ paraphrased questions:", len(df_faq_eval_paraphrased))
print("FAQ still exact match after normalization:", faq_exact_match_count)
print("FAQ exact match ratio:", faq_exact_match_ratio)


# ---------------------------------------------------------
# 9) Build final FAIR retrieval evaluation dataset for Stage 13
# ---------------------------------------------------------

df_eval_run = pd.concat(
    [
        df_faq_eval_paraphrased,
        legal_eval_fixed
    ],
    ignore_index=True
)

print("Updated df_eval_run shape:", df_eval_run.shape)
display(df_eval_run["eval_type"].value_counts().to_frame("count"))


# ---------------------------------------------------------
# 10) Verify that all FAQ gold IDs are now available in indexed corpus
# ---------------------------------------------------------

faq_gold_ids = set(
    df_faq_eval_fixed["gold_document_unit_id"]
    .astype(str)
    .str.strip()
)

available_faq_gold_ids = faq_gold_ids & indexed_parent_ids
missing_faq_gold_ids = faq_gold_ids - indexed_parent_ids

print("FAQ gold IDs:", len(faq_gold_ids))
print("FAQ gold IDs available in indexed corpus:", len(available_faq_gold_ids))
print("FAQ gold IDs missing from indexed corpus:", len(missing_faq_gold_ids))
print("FAQ availability ratio:", round(len(available_faq_gold_ids) / max(len(faq_gold_ids), 1), 4))

assert len(missing_faq_gold_ids) == 0, "Some FAQ gold IDs are still missing from indexed corpus."


# ---------------------------------------------------------
# 11) Save fixed and fair retrieval evaluation dataset
# ---------------------------------------------------------

fixed_eval_path = STAGE03_DIR / "stage03_fixed_faq_and_legal_retrieval_eval_dataset.csv"
fair_eval_path = STAGE03_DIR / "stage03_fair_faq_paraphrase_and_legal_retrieval_eval_dataset.csv"

df_eval_run.to_csv(
    fixed_eval_path,
    index=False,
    encoding="utf-8-sig"
)

df_eval_run.to_csv(
    fair_eval_path,
    index=False,
    encoding="utf-8-sig"
)


# ---------------------------------------------------------
# 12) Display academic explanation and sample
# ---------------------------------------------------------

display(HTML("""
<div dir="rtl" style="text-align:right; line-height:2; font-size:17px;">
<h3>تم إصلاح مجموعة تقييم FAQ للاسترجاع</h3>
<p>
تم إنشاء مجموعة تقييم FAQ جديدة بحيث تكون المراجع الذهبية موجودة فعلياً داخل قاعدة الفهرسة.
كما تم تعديل صياغة أسئلة FAQ في التقييم فقط، مع الإبقاء على الإجابات الأصلية داخل قاعدة المعرفة.
</p>
<p>
هذا التعديل يقلل من مشكلة المطابقة النصية المباشرة، ويجعل التقييم أقرب إلى قياس الاسترجاع الدلالي
بدلاً من اختبار استرجاع سؤال محفوظ بنفس الصياغة.
</p>
</div>
"""))

diagnostic_summary = pd.DataFrame([{
    "indexed_parent_ids": len(indexed_parent_ids),
    "faq_eval_questions": len(df_faq_eval_paraphrased),
    "legal_eval_questions": len(legal_eval_fixed),
    "total_eval_questions": len(df_eval_run),
    "faq_gold_ids_available_ratio": round(len(available_faq_gold_ids) / max(len(faq_gold_ids), 1), 4),
    "faq_exact_match_after_paraphrase_ratio": faq_exact_match_ratio
}])

display(diagnostic_summary)

print("Fixed retrieval evaluation dataset saved to:")
print(fixed_eval_path)

print("Fair retrieval evaluation dataset saved to:")
print(fair_eval_path)

display(
    df_faq_eval_paraphrased[
        [
            "eval_id",
            "original_question",
            "question",
            "gold_document_unit_id",
            "eval_type",
            "same_as_original_after_normalization"
        ]
    ].head(15)
)

Indexed parent document IDs: 714
FAQ indexing source shape: (503, 17)


,faq_id,question,answer,beneficiaries,sector,subsite,page_number,page_url,source,searchable_text,source_type,source_name,legal_category,document_unit_id,text_for_indexing,normalized_text,is_eval_record
0,1,هل تستخدم البوابة والانظمة التابعة لها الحلول الوطنية eSignature؟,"تقدم الوزارة خدمة التوقيع الالكتروني في اطار ادارة وتوثيق عقود العمل عبر منصة ""قوى"" يمكن للمنشآت انشاء وتوثيق وانهاء...",أصحاب عمل | أفراد | العمالة المنزلية | كبار السن,قطاع العمل,مركز الرياض للسياسات السلوكية,1,https://www.hrsd.gov.sa/contact-us/faq,HRSD FAQ,التصنيف: أصحاب عمل | أفراد | العمالة المنزلية | كبار السن قطاع العمل مركز الرياض للسياسات السلوكية السؤال: هل تستخدم...,faq,HRSD FAQ,أصحاب عمل | أفراد | العمالة المنزلية | كبار السن | قطاع العمل | مركز الرياض للسياسات السلوكية,87948be8261685e657502e8c023d087a27fb13927373f98568f379a8d20ff856,المصدر: HRSD FAQ التصنيف: أصحاب عمل | أفراد | العمالة المنزلية | كبار السن | قطاع العمل | مركز الرياض للسياسات السلو...,المصدر: HRSD FAQ التصنيف: اصحاب عمل افراد العماله المنزليه كبار السن قطاع العمل مركز الرياض للسياسات السلوكيه السوال...,False
1,2,ماذا أفعل إذا تم إرجاع الطلب للمراجعة؟,الدخول إلى “مهامي” ← اختيار مهمة طلب مراجعة ← تعديل البيانات ← إعادة الإرسال.,التخصصات الاجتماعية | العمالة المنزلية | كبار السن,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية,1,https://www.hrsd.gov.sa/contact-us/faq,HRSD FAQ,التصنيف: التخصصات الاجتماعية | العمالة المنزلية | كبار السن قطاع التنمية الاجتماعية مركز الرياض للسياسات السلوكية ال...,faq,HRSD FAQ,التخصصات الاجتماعية | العمالة المنزلية | كبار السن | قطاع التنمية الاجتماعية | مركز الرياض للسياسات السلوكية,45ba092b75bd47f574e08441bf1e9c90f26407759e87244a21cbec0393a0b80b,المصدر: HRSD FAQ التصنيف: التخصصات الاجتماعية | العمالة المنزلية | كبار السن | قطاع التنمية الاجتماعية | مركز الرياض...,المصدر: HRSD FAQ التصنيف: التخصصات الاجتماعيه العماله المنزليه كبار السن قطاع التنميه الاجتماعيه مركز الرياض للسياسا...,False
2,3,ماذا يحدث بعد إرسال الطلب؟,يتم تحويل الطلب إلى اللجنة للمراجعة ويتم إشعار المتقدم بحالة الطلب.,التخصصات الاجتماعية | العمالة المنزلية | كبار السن,قطاع التنمية الاجتماعية,مركز الرياض للسياسات السلوكية,1,https://www.hrsd.gov.sa/contact-us/faq,HRSD FAQ,التصنيف: التخصصات الاجتماعية | العمالة المنزلية | كبار السن قطاع التنمية الاجتماعية مركز الرياض للسياسات السلوكية ال...,faq,HRSD FAQ,التخصصات الاجتماعية | العمالة المنزلية | كبار السن | قطاع التنمية الاجتماعية | مركز الرياض للسياسات السلوكية,892e3d1d59a9e5b0888f0ee43a62b3a57facd5b2c367f9d7a2982b9c020f85b2,المصدر: HRSD FAQ التصنيف: التخصصات الاجتماعية | العمالة المنزلية | كبار السن | قطاع التنمية الاجتماعية | مركز الرياض...,المصدر: HRSD FAQ التصنيف: التخصصات الاجتماعيه العماله المنزليه كبار السن قطاع التنميه الاجتماعيه مركز الرياض للسياسا...,False


FAQ ID column: document_unit_id
FAQ question column: question
FAQ records available in indexed corpus: 503
Final fixed FAQ retrieval evaluation size: 131
Fixed FAQ evaluation rows: (131, 8)
Legal retrieval evaluation size: 24
FAQ paraphrased questions: 131
FAQ still exact match after normalization: 0
FAQ exact match ratio: 0.0
Updated df_eval_run shape: (155, 22)


,count
eval_type,
faq_paraphrase_retrieval,131
legal_article_retrieval,24


FAQ gold IDs: 131
FAQ gold IDs available in indexed corpus: 131
FAQ gold IDs missing from indexed corpus: 0
FAQ availability ratio: 1.0


,indexed_parent_ids,faq_eval_questions,legal_eval_questions,total_eval_questions,faq_gold_ids_available_ratio,faq_exact_match_after_paraphrase_ratio
0,714,131,24,155,1.0,0.0


Fixed retrieval evaluation dataset saved to:
C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\stage03_fixed_faq_and_legal_retrieval_eval_dataset.csv
Fair retrieval evaluation dataset saved to:
C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\stage03_fair_faq_paraphrase_and_legal_retrieval_eval_dataset.csv


,eval_id,original_question,question,gold_document_unit_id,eval_type,same_as_original_after_normalization
0,FAQ_INDEXED_RETRIEVAL_0001,هل توفر البوابة المعلومات ومجموعة الخدمات الخاصة بالمرأة؟,أريد معرفة ما إذا كانت توفر البوابة المعلومات ومجموعة الخدمات الخاصة بالمرأة؟,1d2d4109d0423311606722e82dfd41c20a7f7742c83169619385ec8d2a4c2155,faq_paraphrase_retrieval,False
1,FAQ_INDEXED_RETRIEVAL_0002,ماهي أوقات العمل الرسمية للفروع؟,أريد توضيح الحكم النظامي بخصوص: ماهي أوقات العمل الرسمية للفروع؟,271c55cb0f0f2d621adfb634fce4722dc68011cfcef490eec0337fb3104fcedb,faq_paraphrase_retrieval,False
2,FAQ_INDEXED_RETRIEVAL_0003,شاهدتُ حالة عنف وقعت أمامي، ماذا أفعل؟,أريد توضيح الحكم النظامي بخصوص: شاهدتُ حالة عنف وقعت أمامي، ماذا أفعل؟,8f9d950f458be122538bbb4a6e889159595c47b5b21a723af28c00e7166a1468,faq_paraphrase_retrieval,False
3,FAQ_INDEXED_RETRIEVAL_0004,هل يتم احتساب الفرد الذي يعمل لدى أكثر من منشأة في نفس الوقت ضمن نسبة التوطين؟,أريد معرفة ما إذا كان يتم احتساب الفرد الذي يعمل لدى أكثر من منشأة في نفس الوقت ضمن نسبة التوطين؟,a78f94c86dd92b762fcc3c865abb5182e6facb506a9d5512a0f44e5b9c30e184,faq_paraphrase_retrieval,False
4,FAQ_INDEXED_RETRIEVAL_0005,هل يمكن إصدار تأشيرة لجنسية أو مهنة غير مدرجة في مساند؟,هل يستطيع إصدار تأشيرة لجنسية أو مهنة غير مدرجة في مساند؟,ac0493e4a60d1923d3e181202d0bfd389b31023b65ec159be502aee177c1a7a3,faq_paraphrase_retrieval,False
5,FAQ_INDEXED_RETRIEVAL_0006,هل يتم إيواء المرأة التي تعرضت للعنف؟,أريد معرفة ما إذا كان يتم إيواء المرأة التي تعرضت للعنف؟,4b4e32fe41f1745800e5617e787e755d1bd2198efdceb01bb0f43a814b09205a,faq_paraphrase_retrieval,False
6,FAQ_INDEXED_RETRIEVAL_0007,ما مدى امكانية الجمع بين المكافأة المقطوعة للمستشارين غير المتفرغين ومكافأة العمل خارج وقت الدوام الرسمي وأيام العطل...,أريد توضيح الحكم النظامي بخصوص: ما مدى امكانية الجمع بين المكافأة المقطوعة للمستشارين غير المتفرغين ومكافأة العمل خا...,486f8265889bb1cb71b039ce24e5d65ba171764ac0fd47ef3693ce9d9b64c9b6,faq_paraphrase_retrieval,False
7,FAQ_INDEXED_RETRIEVAL_0008,هل الخدمة تتيح تحديث بيانات العامل.,أريد معرفة ما إذا كان الخدمة تتيح تحديث بيانات العامل؟,efee6aee87704b6999b8578223366ee7497784934576b50a92d7e4493d8133d9,faq_paraphrase_retrieval,False
8,FAQ_INDEXED_RETRIEVAL_0009,ماهي خدمة قرار مكافأة الطلبة الجامعين لذوي الإعاقة ؟,أريد توضيح الحكم النظامي بخصوص: ماهي خدمة قرار مكافأة الطلبة الجامعين لذوي الإعاقة؟,79cfa8fd63a7d91b82f3805b9e15f894c8ee5ab0e56ef344a0c328a6455fdf45,faq_paraphrase_retrieval,False
9,FAQ_INDEXED_RETRIEVAL_0010,العديد يطالبون بشرح البرنامج وشرح للدليل الإجرائي بشكل مبسط وواضح.,أريد توضيح الحكم النظامي بخصوص: العديد يطالبون بشرح البرنامج وشرح للدليل الإجرائي بشكل مبسط وواضح؟,88c5634d5ce67afabafadfa5ac0f6e59e58a20ee6ddd2a52ea39b4f1bd7eaa7b,faq_paraphrase_retrieval,False


In [20]:
# =========================================================
# Stage 12.6 - Add Extra Legal Questions for Manual Review
# إضافة 80 سؤال قانوني طبيعي للمراجعة اليدوية وربطها لاحقاً مع Golden ID
# =========================================================

import pandas as pd
from IPython.display import display, HTML

extra_legal_questions = [
    {"eval_id": "LEGAL_EXTRA_001", "eval_type": "legal_article_retrieval", "question": "كم أقصى مدة ممكن تكون فترة التجربة في عقد العمل؟", "legal_topic": "فترة التجربة"},
    {"eval_id": "LEGAL_EXTRA_002", "eval_type": "legal_article_retrieval", "question": "هل يحق للشركة تمدد فترة التجربة بعد ما بدأت؟", "legal_topic": "فترة التجربة"},
    {"eval_id": "LEGAL_EXTRA_003", "eval_type": "legal_article_retrieval", "question": "إذا كنت في فترة التجربة هل أقدر أترك العمل بدون إنذار؟", "legal_topic": "فترة التجربة"},
    {"eval_id": "LEGAL_EXTRA_004", "eval_type": "legal_article_retrieval", "question": "هل لازم تكون فترة التجربة مكتوبة في العقد؟", "legal_topic": "فترة التجربة"},
    {"eval_id": "LEGAL_EXTRA_005", "eval_type": "legal_article_retrieval", "question": "لو انتهت فترة التجربة وسكتت الشركة هل أعتبر مثبت؟", "legal_topic": "فترة التجربة"},
    {"eval_id": "LEGAL_EXTRA_006", "eval_type": "legal_article_retrieval", "question": "هل يجوز لصاحب العمل يجربني أكثر من مرة على نفس الوظيفة؟", "legal_topic": "فترة التجربة"},
    {"eval_id": "LEGAL_EXTRA_007", "eval_type": "legal_article_retrieval", "question": "هل يحق للموظف يرفض تمديد فترة التجربة؟", "legal_topic": "فترة التجربة"},
    {"eval_id": "LEGAL_EXTRA_008", "eval_type": "legal_article_retrieval", "question": "إذا وقعت عقد جديد مع نفس الشركة هل تبدأ فترة تجربة جديدة؟", "legal_topic": "فترة التجربة"},

    {"eval_id": "LEGAL_EXTRA_009", "eval_type": "legal_article_retrieval", "question": "متى يعتبر عقد العمل منتهي نظامياً؟", "legal_topic": "انتهاء عقد العمل"},
    {"eval_id": "LEGAL_EXTRA_010", "eval_type": "legal_article_retrieval", "question": "هل انتهاء مدة العقد يعني أن العقد انتهى تلقائياً؟", "legal_topic": "انتهاء عقد العمل"},
    {"eval_id": "LEGAL_EXTRA_011", "eval_type": "legal_article_retrieval", "question": "إذا استمر الموظف بالعمل بعد انتهاء العقد هل يتجدد العقد؟", "legal_topic": "انتهاء عقد العمل"},
    {"eval_id": "LEGAL_EXTRA_012", "eval_type": "legal_article_retrieval", "question": "هل يحق للشركة تنهي عقدي بدون سبب واضح؟", "legal_topic": "انتهاء عقد العمل"},
    {"eval_id": "LEGAL_EXTRA_013", "eval_type": "legal_article_retrieval", "question": "متى يحق للعامل إنهاء العقد بدون موافقة صاحب العمل؟", "legal_topic": "انتهاء عقد العمل"},
    {"eval_id": "LEGAL_EXTRA_014", "eval_type": "legal_article_retrieval", "question": "إذا الشركة أغلقت الفرع هل يحق لها إنهاء عقد الموظفين؟", "legal_topic": "انتهاء عقد العمل"},
    {"eval_id": "LEGAL_EXTRA_015", "eval_type": "legal_article_retrieval", "question": "هل وفاة صاحب العمل تنهي عقد العامل؟", "legal_topic": "انتهاء عقد العمل"},
    {"eval_id": "LEGAL_EXTRA_016", "eval_type": "legal_article_retrieval", "question": "هل بلوغ العامل سن التقاعد ينهي عقد العمل؟", "legal_topic": "انتهاء عقد العمل"},

    {"eval_id": "LEGAL_EXTRA_017", "eval_type": "legal_article_retrieval", "question": "متى يستحق الموظف مكافأة نهاية الخدمة؟", "legal_topic": "مكافأة نهاية الخدمة"},
    {"eval_id": "LEGAL_EXTRA_018", "eval_type": "legal_article_retrieval", "question": "كيف تحسب مكافأة نهاية الخدمة إذا خدمت خمس سنوات؟", "legal_topic": "مكافأة نهاية الخدمة"},
    {"eval_id": "LEGAL_EXTRA_019", "eval_type": "legal_article_retrieval", "question": "إذا استقلت هل آخذ مكافأة نهاية الخدمة كاملة؟", "legal_topic": "مكافأة نهاية الخدمة"},
    {"eval_id": "LEGAL_EXTRA_020", "eval_type": "legal_article_retrieval", "question": "هل الموظف يستحق مكافأة نهاية الخدمة إذا تم فصله؟", "legal_topic": "مكافأة نهاية الخدمة"},
    {"eval_id": "LEGAL_EXTRA_021", "eval_type": "legal_article_retrieval", "question": "هل تحسب البدلات ضمن مكافأة نهاية الخدمة؟", "legal_topic": "مكافأة نهاية الخدمة"},
    {"eval_id": "LEGAL_EXTRA_022", "eval_type": "legal_article_retrieval", "question": "إذا اشتغلت أقل من سنتين هل لي مكافأة نهاية خدمة؟", "legal_topic": "مكافأة نهاية الخدمة"},
    {"eval_id": "LEGAL_EXTRA_023", "eval_type": "legal_article_retrieval", "question": "هل تختلف مكافأة نهاية الخدمة بين الاستقالة وإنهاء العقد؟", "legal_topic": "مكافأة نهاية الخدمة"},
    {"eval_id": "LEGAL_EXTRA_024", "eval_type": "legal_article_retrieval", "question": "متى تسقط مكافأة نهاية الخدمة عن العامل؟", "legal_topic": "مكافأة نهاية الخدمة"},

    {"eval_id": "LEGAL_EXTRA_025", "eval_type": "legal_article_retrieval", "question": "كم عدد ساعات العمل اليومية في النظام؟", "legal_topic": "ساعات العمل"},
    {"eval_id": "LEGAL_EXTRA_026", "eval_type": "legal_article_retrieval", "question": "كم الحد الأقصى لساعات العمل الأسبوعية؟", "legal_topic": "ساعات العمل"},
    {"eval_id": "LEGAL_EXTRA_027", "eval_type": "legal_article_retrieval", "question": "هل تختلف ساعات العمل في رمضان؟", "legal_topic": "ساعات العمل"},
    {"eval_id": "LEGAL_EXTRA_028", "eval_type": "legal_article_retrieval", "question": "هل يحق لصاحب العمل يطلب مني أشتغل وقت إضافي؟", "legal_topic": "ساعات العمل الإضافية"},
    {"eval_id": "LEGAL_EXTRA_029", "eval_type": "legal_article_retrieval", "question": "كيف يتم حساب أجر الساعة الإضافية؟", "legal_topic": "ساعات العمل الإضافية"},
    {"eval_id": "LEGAL_EXTRA_030", "eval_type": "legal_article_retrieval", "question": "هل يجوز تشغيل العامل أكثر من الحد النظامي لساعات العمل؟", "legal_topic": "ساعات العمل"},
    {"eval_id": "LEGAL_EXTRA_031", "eval_type": "legal_article_retrieval", "question": "هل وقت الراحة داخل الدوام يحسب من ساعات العمل؟", "legal_topic": "فترات الراحة"},
    {"eval_id": "LEGAL_EXTRA_032", "eval_type": "legal_article_retrieval", "question": "كم أقل مدة راحة لازم تعطى للعامل خلال اليوم؟", "legal_topic": "فترات الراحة"},

    {"eval_id": "LEGAL_EXTRA_033", "eval_type": "legal_article_retrieval", "question": "هل يحق للموظف إجازة سنوية مدفوعة؟", "legal_topic": "الإجازة السنوية"},
    {"eval_id": "LEGAL_EXTRA_034", "eval_type": "legal_article_retrieval", "question": "كم عدد أيام الإجازة السنوية للعامل؟", "legal_topic": "الإجازة السنوية"},
    {"eval_id": "LEGAL_EXTRA_035", "eval_type": "legal_article_retrieval", "question": "هل تزيد الإجازة السنوية بعد سنوات خدمة معينة؟", "legal_topic": "الإجازة السنوية"},
    {"eval_id": "LEGAL_EXTRA_036", "eval_type": "legal_article_retrieval", "question": "هل يحق للشركة تأجيل إجازتي السنوية؟", "legal_topic": "الإجازة السنوية"},
    {"eval_id": "LEGAL_EXTRA_037", "eval_type": "legal_article_retrieval", "question": "إذا تركت العمل هل آخذ بدل رصيد الإجازات؟", "legal_topic": "الإجازة السنوية"},
    {"eval_id": "LEGAL_EXTRA_038", "eval_type": "legal_article_retrieval", "question": "هل أقدر أتنازل عن إجازتي السنوية مقابل مبلغ؟", "legal_topic": "الإجازة السنوية"},
    {"eval_id": "LEGAL_EXTRA_039", "eval_type": "legal_article_retrieval", "question": "هل إجازات الأعياد تكون مدفوعة الأجر؟", "legal_topic": "الإجازات الرسمية"},
    {"eval_id": "LEGAL_EXTRA_040", "eval_type": "legal_article_retrieval", "question": "إذا صادفت الإجازة الرسمية يوم راحتي الأسبوعية هل أعوض عنها؟", "legal_topic": "الإجازات الرسمية"},

    {"eval_id": "LEGAL_EXTRA_041", "eval_type": "legal_article_retrieval", "question": "كم مدة الإجازة المرضية المسموحة للعامل؟", "legal_topic": "الإجازة المرضية"},
    {"eval_id": "LEGAL_EXTRA_042", "eval_type": "legal_article_retrieval", "question": "هل الإجازة المرضية تكون بأجر كامل؟", "legal_topic": "الإجازة المرضية"},
    {"eval_id": "LEGAL_EXTRA_043", "eval_type": "legal_article_retrieval", "question": "متى تكون الإجازة المرضية بدون أجر؟", "legal_topic": "الإجازة المرضية"},
    {"eval_id": "LEGAL_EXTRA_044", "eval_type": "legal_article_retrieval", "question": "هل يحق لصاحب العمل فصل الموظف بسبب المرض؟", "legal_topic": "الإجازة المرضية"},
    {"eval_id": "LEGAL_EXTRA_045", "eval_type": "legal_article_retrieval", "question": "هل يجب تقديم تقرير طبي للحصول على إجازة مرضية؟", "legal_topic": "الإجازة المرضية"},
    {"eval_id": "LEGAL_EXTRA_046", "eval_type": "legal_article_retrieval", "question": "إذا مرضت أثناء الإجازة السنوية هل تتحول لإجازة مرضية؟", "legal_topic": "الإجازة المرضية"},

    {"eval_id": "LEGAL_EXTRA_047", "eval_type": "legal_article_retrieval", "question": "متى يحق لصاحب العمل فصل العامل بدون مكافأة؟", "legal_topic": "الفصل التأديبي"},
    {"eval_id": "LEGAL_EXTRA_048", "eval_type": "legal_article_retrieval", "question": "هل الغياب بدون عذر يعطي الشركة حق الفصل؟", "legal_topic": "الغياب"},
    {"eval_id": "LEGAL_EXTRA_049", "eval_type": "legal_article_retrieval", "question": "كم يوم غياب يسبب فصل الموظف؟", "legal_topic": "الغياب"},
    {"eval_id": "LEGAL_EXTRA_050", "eval_type": "legal_article_retrieval", "question": "هل لازم الشركة تنذر الموظف قبل فصله بسبب الغياب؟", "legal_topic": "الغياب"},
    {"eval_id": "LEGAL_EXTRA_051", "eval_type": "legal_article_retrieval", "question": "إذا رفض الموظف تنفيذ العمل هل يحق فصله؟", "legal_topic": "الفصل التأديبي"},
    {"eval_id": "LEGAL_EXTRA_052", "eval_type": "legal_article_retrieval", "question": "هل يحق للشركة فصل الموظف بسبب إفشاء أسرار العمل؟", "legal_topic": "الفصل التأديبي"},
    {"eval_id": "LEGAL_EXTRA_053", "eval_type": "legal_article_retrieval", "question": "هل الاعتداء على المدير أو الزملاء يؤدي للفصل؟", "legal_topic": "الفصل التأديبي"},
    {"eval_id": "LEGAL_EXTRA_054", "eval_type": "legal_article_retrieval", "question": "هل استخدام مستندات مزورة في التوظيف يسبب الفصل؟", "legal_topic": "الفصل التأديبي"},

    {"eval_id": "LEGAL_EXTRA_055", "eval_type": "legal_article_retrieval", "question": "هل يجوز نقل العامل من مدينة إلى مدينة بدون موافقته؟", "legal_topic": "نقل العامل"},
    {"eval_id": "LEGAL_EXTRA_056", "eval_type": "legal_article_retrieval", "question": "هل يحق للشركة تغيير طبيعة عملي بعد توقيع العقد؟", "legal_topic": "شروط العمل"},
    {"eval_id": "LEGAL_EXTRA_057", "eval_type": "legal_article_retrieval", "question": "إذا كلفتني الشركة بعمل غير المتفق عليه هل هذا نظامي؟", "legal_topic": "شروط العمل"},
    {"eval_id": "LEGAL_EXTRA_058", "eval_type": "legal_article_retrieval", "question": "هل يحق لصاحب العمل تخفيض راتبي بدون موافقتي؟", "legal_topic": "الأجر"},
    {"eval_id": "LEGAL_EXTRA_059", "eval_type": "legal_article_retrieval", "question": "متى يجب دفع راتب العامل؟", "legal_topic": "الأجر"},
    {"eval_id": "LEGAL_EXTRA_060", "eval_type": "legal_article_retrieval", "question": "هل تأخير الراتب يعتبر مخالفة؟", "legal_topic": "الأجر"},
    {"eval_id": "LEGAL_EXTRA_061", "eval_type": "legal_article_retrieval", "question": "هل يجوز الخصم من راتب الموظف؟", "legal_topic": "الخصم من الأجر"},
    {"eval_id": "LEGAL_EXTRA_062", "eval_type": "legal_article_retrieval", "question": "كم الحد الأعلى للخصم من الراتب؟", "legal_topic": "الخصم من الأجر"},
    {"eval_id": "LEGAL_EXTRA_063", "eval_type": "legal_article_retrieval", "question": "هل يحق للشركة تخصم من راتبي بسبب تلف جهاز؟", "legal_topic": "الخصم من الأجر"},
    {"eval_id": "LEGAL_EXTRA_064", "eval_type": "legal_article_retrieval", "question": "هل يحق للعامل الاعتراض على الخصم من راتبه؟", "legal_topic": "الخصم من الأجر"},

    {"eval_id": "LEGAL_EXTRA_065", "eval_type": "legal_article_retrieval", "question": "هل يجب أن يكون عقد العمل مكتوب؟", "legal_topic": "عقد العمل"},
    {"eval_id": "LEGAL_EXTRA_066", "eval_type": "legal_article_retrieval", "question": "ما البيانات الأساسية التي لازم تكون في عقد العمل؟", "legal_topic": "عقد العمل"},
    {"eval_id": "LEGAL_EXTRA_067", "eval_type": "legal_article_retrieval", "question": "إذا ما عندي عقد مكتوب هل تضيع حقوقي؟", "legal_topic": "عقد العمل"},
    {"eval_id": "LEGAL_EXTRA_068", "eval_type": "legal_article_retrieval", "question": "هل يحق للعامل طلب نسخة من عقد العمل؟", "legal_topic": "عقد العمل"},
    {"eval_id": "LEGAL_EXTRA_069", "eval_type": "legal_article_retrieval", "question": "هل العقد الإلكتروني يعتبر عقد عمل نظامي؟", "legal_topic": "عقد العمل"},
    {"eval_id": "LEGAL_EXTRA_070", "eval_type": "legal_article_retrieval", "question": "إذا اختلف الراتب الفعلي عن المكتوب في العقد ماذا أفعل؟", "legal_topic": "عقد العمل"},

    {"eval_id": "LEGAL_EXTRA_071", "eval_type": "legal_article_retrieval", "question": "هل يحق للعامل الحصول على شهادة خبرة بعد ترك العمل؟", "legal_topic": "شهادة الخدمة"},
    {"eval_id": "LEGAL_EXTRA_072", "eval_type": "legal_article_retrieval", "question": "هل يجوز لصاحب العمل يمنعني من أخذ شهادة خدمة؟", "legal_topic": "شهادة الخدمة"},
    {"eval_id": "LEGAL_EXTRA_073", "eval_type": "legal_article_retrieval", "question": "هل يحق للشركة تحتفظ بجواز العامل؟", "legal_topic": "حقوق العامل"},
    {"eval_id": "LEGAL_EXTRA_074", "eval_type": "legal_article_retrieval", "question": "هل يحق لصاحب العمل تحميل العامل رسوم الاستقدام؟", "legal_topic": "حقوق العامل"},
    {"eval_id": "LEGAL_EXTRA_075", "eval_type": "legal_article_retrieval", "question": "هل يجوز تشغيل العامل في يوم الراحة الأسبوعية؟", "legal_topic": "الراحة الأسبوعية"},
    {"eval_id": "LEGAL_EXTRA_076", "eval_type": "legal_article_retrieval", "question": "ما هو يوم الراحة الأسبوعية في نظام العمل؟", "legal_topic": "الراحة الأسبوعية"},
    {"eval_id": "LEGAL_EXTRA_077", "eval_type": "legal_article_retrieval", "question": "إذا اشتغلت في يوم الراحة هل أستحق تعويض؟", "legal_topic": "الراحة الأسبوعية"},
    {"eval_id": "LEGAL_EXTRA_078", "eval_type": "legal_article_retrieval", "question": "هل يحق للعامل ترك العمل إذا لم يدفع صاحب العمل الراتب؟", "legal_topic": "ترك العمل"},
    {"eval_id": "LEGAL_EXTRA_079", "eval_type": "legal_article_retrieval", "question": "هل يحق للعامل ترك العمل إذا تعرض للإهانة أو المعاملة السيئة؟", "legal_topic": "ترك العمل"},
    {"eval_id": "LEGAL_EXTRA_080", "eval_type": "legal_article_retrieval", "question": "هل يحق للعامل ترك العمل إذا كلف بعمل خطر بدون حماية؟", "legal_topic": "السلامة المهنية"},
]

df_extra_questions = pd.DataFrame(extra_legal_questions)

display(HTML("""
<div dir="rtl" style="text-align:right; line-height:2; font-size:16px;">
<h3>تم تحميل الأسئلة القانونية الإضافية</h3>
<p>
هذه الأسئلة لم تدخل التقييم النهائي بعد، لأنها تحتاج إلى ربط يدوي مع Golden ID الصحيح.
</p>
</div>
"""))

print("Extra legal questions:", len(df_extra_questions))
display(df_extra_questions.head(10))

Extra legal questions: 80


,eval_id,eval_type,question,legal_topic
0,LEGAL_EXTRA_001,legal_article_retrieval,كم أقصى مدة ممكن تكون فترة التجربة في عقد العمل؟,فترة التجربة
1,LEGAL_EXTRA_002,legal_article_retrieval,هل يحق للشركة تمدد فترة التجربة بعد ما بدأت؟,فترة التجربة
2,LEGAL_EXTRA_003,legal_article_retrieval,إذا كنت في فترة التجربة هل أقدر أترك العمل بدون إنذار؟,فترة التجربة
3,LEGAL_EXTRA_004,legal_article_retrieval,هل لازم تكون فترة التجربة مكتوبة في العقد؟,فترة التجربة
4,LEGAL_EXTRA_005,legal_article_retrieval,لو انتهت فترة التجربة وسكتت الشركة هل أعتبر مثبت؟,فترة التجربة
5,LEGAL_EXTRA_006,legal_article_retrieval,هل يجوز لصاحب العمل يجربني أكثر من مرة على نفس الوظيفة؟,فترة التجربة
6,LEGAL_EXTRA_007,legal_article_retrieval,هل يحق للموظف يرفض تمديد فترة التجربة؟,فترة التجربة
7,LEGAL_EXTRA_008,legal_article_retrieval,إذا وقعت عقد جديد مع نفس الشركة هل تبدأ فترة تجربة جديدة؟,فترة التجربة
8,LEGAL_EXTRA_009,legal_article_retrieval,متى يعتبر عقد العمل منتهي نظامياً؟,انتهاء عقد العمل
9,LEGAL_EXTRA_010,legal_article_retrieval,هل انتهاء مدة العقد يعني أن العقد انتهى تلقائياً؟,انتهاء عقد العمل


# To Add New Leagel Qoutsion 

In [21]:
# =========================================================
# Stage 12.7 - Suggest Golden IDs for Extra Legal Questions
# اقتراح Golden IDs للأسئلة القانونية الجديدة
# =========================================================

import pandas as pd
import numpy as np
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from IPython.display import display, HTML

def first_existing_col(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None

def clean_for_search(text):
    text = "" if pd.isna(text) else str(text)
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"ؤ", "و", text)
    text = re.sub(r"ئ", "ي", text)
    text = re.sub(r"[^\u0600-\u06FFa-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# ---------------------------------------------------------
# Build legal candidate documents from CORPORA
# ---------------------------------------------------------

legal_docs = []

for corpus_name, corpus_df in CORPORA.items():
    temp = corpus_df.copy()

    id_col = first_existing_col(temp, ["parent_document_id", "document_unit_id", "doc_id", "id"])
    text_col = first_existing_col(temp, ["retrieval_text", "chunk_text", "text", "article_text", "content"])
    article_col = first_existing_col(temp, ["article_number", "gold_article_number", "article_id", "article_number_int"])

    if id_col is None or text_col is None:
        continue

    if "source_type" in temp.columns:
        temp = temp[
            temp["source_type"].astype(str).str.contains("law|legal|article", case=False, na=False)
        ].copy()

    for _, row in temp.iterrows():
        doc_id = str(row[id_col]).strip()
        text = str(row[text_col]).strip()

        if not doc_id or not text:
            continue

        article_number = ""
        if article_col is not None:
            article_number = str(row[article_col]).strip()

        legal_docs.append({
            "corpus_name": corpus_name,
            "candidate_id": doc_id,
            "candidate_article_number": article_number,
            "candidate_text": text
        })

df_legal_docs = pd.DataFrame(legal_docs)

df_legal_docs = (
    df_legal_docs
    .drop_duplicates(subset=["candidate_id"])
    .reset_index(drop=True)
)

assert len(df_legal_docs) > 0, "No legal candidate documents found."
assert "df_extra_questions" in globals(), "Run Stage 12.6 first."

df_extra = df_extra_questions.copy()

df_extra["clean_question"] = df_extra["question"].apply(clean_for_search)
df_legal_docs["clean_text"] = df_legal_docs["candidate_text"].apply(clean_for_search)

# ---------------------------------------------------------
# TF-IDF similarity for candidate suggestion only
# ---------------------------------------------------------

all_texts = df_legal_docs["clean_text"].tolist() + df_extra["clean_question"].tolist()

vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=1
)

tfidf_matrix = vectorizer.fit_transform(all_texts)

doc_vectors = tfidf_matrix[:len(df_legal_docs)]
question_vectors = tfidf_matrix[len(df_legal_docs):]

similarity_matrix = cosine_similarity(question_vectors, doc_vectors)

# ---------------------------------------------------------
# Build review table
# ---------------------------------------------------------

TOP_N = 5
review_rows = []

for q_idx, q_row in df_extra.reset_index(drop=True).iterrows():
    sims = similarity_matrix[q_idx]
    top_indices = np.argsort(sims)[::-1][:TOP_N]

    base = {
        "eval_id": q_row["eval_id"],
        "eval_type": "legal_article_retrieval",
        "question": q_row["question"],
        "legal_topic": q_row.get("legal_topic", ""),
        "manual_gold_article_number": "",
        "manual_gold_document_unit_id": "",
        "review_status": "PENDING_MANUAL_REVIEW"
    }

    for rank, doc_idx in enumerate(top_indices, start=1):
        cand = df_legal_docs.iloc[doc_idx]
        base[f"candidate_{rank}_score"] = round(float(sims[doc_idx]), 4)
        base[f"candidate_{rank}_article"] = cand["candidate_article_number"]
        base[f"candidate_{rank}_id"] = cand["candidate_id"]
        base[f"candidate_{rank}_text_preview"] = cand["candidate_text"][:500]

    review_rows.append(base)

df_gold_review = pd.DataFrame(review_rows)

review_path = STAGE03_DIR / "legal_extra_questions_gold_id_review_candidates.csv"

df_gold_review.to_csv(
    review_path,
    index=False,
    encoding="utf-8-sig"
)

display(HTML("""
<div dir="rtl" style="text-align:right; line-height:2; font-size:16px;">
<h3>تم إنشاء ملف مراجعة Golden IDs</h3>
<p>
افتح الملف، وراجع أفضل 5 مواد مرشحة لكل سؤال، ثم املأ manual_gold_article_number و manual_gold_document_unit_id يدوياً.
</p>
</div>
"""))

print("Review file saved to:")
print(review_path)

display(df_gold_review.head())

Review file saved to:
C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\legal_extra_questions_gold_id_review_candidates.csv


,eval_id,eval_type,question,legal_topic,manual_gold_article_number,manual_gold_document_unit_id,review_status,candidate_1_score,candidate_1_article,candidate_1_id,...,candidate_3_id,candidate_3_text_preview,candidate_4_score,candidate_4_article,candidate_4_id,candidate_4_text_preview,candidate_5_score,candidate_5_article,candidate_5_id,candidate_5_text_preview
0,LEGAL_EXTRA_001,legal_article_retrieval,كم أقصى مدة ممكن تكون فترة التجربة في عقد العمل؟,فترة التجربة,,,PENDING_MANUAL_REVIEW,0.2556,الرابعة والخمسون,ab8af4e6ecab46b0bf5b588cc55664b9d0a008dc9073912d3417c1f04d083142,...,b56e9c99f1c551928c8727dae90850186c14efc78d6272f429f0c448add3f3dc,التصنيف القانوني: علاقات العمل\nالباب: علاقات العمل\nالفصل: الفصل الأول\nرقم المادة: الثالثة والخمسون\nعنوان المادة:...,0.1002,الثانية,4f2961046b5fffdc637e2b7de4dfb1fd862ccc7b9f3069349681b04ff17df638,التصنيف القانوني: التعريفات / الأحكام العامة\nالباب: التعريفات / الأحكام العامة\nالفصل: الفصل الأول\nرقم المادة: الث...,0.0950,التاسعة والسبعون بعد المائة,cf4e4b2806a0c863c377af5a46b2d45d628aa531ca1024270da12c93a51c2097,التصنيف القانوني: عقد العمل البحري\nالباب: عقد العمل البحري\nالفصل: nan\nرقم المادة: التاسعة والسبعون بعد المائة\nعن...
1,LEGAL_EXTRA_002,legal_article_retrieval,هل يحق للشركة تمدد فترة التجربة بعد ما بدأت؟,فترة التجربة,,,PENDING_MANUAL_REVIEW,0.1790,الرابعة والخمسون,ab8af4e6ecab46b0bf5b588cc55664b9d0a008dc9073912d3417c1f04d083142,...,dd858095f7b7989c6920172b61ccf992fb378afd325c06fb4eda55d1ceadaf9c,التصنيف القانوني: شروط العمل وظروفه\nالباب: شروط العمل وظروفه\nالفصل: الفصل الثالث\nرقم المادة: الثالثة بعد المائة\n...,0.0518,الثالثة والخمسون,b56e9c99f1c551928c8727dae90850186c14efc78d6272f429f0c448add3f3dc,التصنيف القانوني: علاقات العمل\nالباب: علاقات العمل\nالفصل: الفصل الأول\nرقم المادة: الثالثة والخمسون\nعنوان المادة:...,0.0484,الستون بعد المائة,f3b2d548d622040188c6ebcfda67f959c540fe61dbb6be0a130d3cd508041879,التصنيف القانوني: تشغيل النساء\nالباب: تشغيل النساء\nالفصل: nan\nرقم المادة: الستون بعد المائة\nعنوان المادة: المادة...
2,LEGAL_EXTRA_003,legal_article_retrieval,إذا كنت في فترة التجربة هل أقدر أترك العمل بدون إنذار؟,فترة التجربة,,,PENDING_MANUAL_REVIEW,0.1924,الرابعة والخمسون,ab8af4e6ecab46b0bf5b588cc55664b9d0a008dc9073912d3417c1f04d083142,...,93106f92e2e463b2afeee9b680ab418b16cc3907a7b7f1d8308274a5081c8839,التصنيف القانوني: علاقات العمل\nالباب: علاقات العمل\nالفصل: الفصل الثالث\nرقم المادة: الحادية والثمانون\nعنوان الماد...,0.0849,الثالثة بعد المائة,dd858095f7b7989c6920172b61ccf992fb378afd325c06fb4eda55d1ceadaf9c,التصنيف القانوني: شروط العمل وظروفه\nالباب: شروط العمل وظروفه\nالفصل: الفصل الثالث\nرقم المادة: الثالثة بعد المائة\n...,0.0824,الثانية,4f2961046b5fffdc637e2b7de4dfb1fd862ccc7b9f3069349681b04ff17df638,التصنيف القانوني: التعريفات / الأحكام العامة\nالباب: التعريفات / الأحكام العامة\nالفصل: الفصل الأول\nرقم المادة: الث...
3,LEGAL_EXTRA_004,legal_article_retrieval,هل لازم تكون فترة التجربة مكتوبة في العقد؟,فترة التجربة,,,PENDING_MANUAL_REVIEW,0.2511,الرابعة والخمسون,ab8af4e6ecab46b0bf5b588cc55664b9d0a008dc9073912d3417c1f04d083142,...,4f2961046b5fffdc637e2b7de4dfb1fd862ccc7b9f3069349681b04ff17df638,التصنيف القانوني: التعريفات / الأحكام العامة\nالباب: التعريفات / الأحكام العامة\nالفصل: الفصل الأول\nرقم المادة: الث...,0.0818,الثالثة والخمسون,b56e9c99f1c551928c8727dae90850186c14efc78d6272f429f0c448add3f3dc,التصنيف القانوني: علاقات العمل\nالباب: علاقات العمل\nالفصل: الفصل الأول\nرقم المادة: الثالثة والخمسون\nعنوان المادة:...,0.0785,السابعة والسبعون,1b19a4fb2b4d423df5c66d04b1a372b80a041caaf759a0b8ba32555c20a6f902,التصنيف القانوني: علاقات العمل\nالباب: علاقات العمل\nالفصل: الفصل الثالث\nرقم المادة: السابعة والسبعون\nعنوان المادة...
4,LEGAL_EXTRA_005,legal_article_retrieval,لو انتهت فترة التجربة وسكتت الشركة هل أعتبر مثبت؟,فترة التجربة,,,PENDING_MANUAL_REVIEW,0.1590,الرابعة والخمسون,ab8af4e6ecab46b0bf5b588cc55664b9d0a008dc9073912d3417c1f04d083142,...,596b788c22bea70c4b18299cbd376342009abee45436ad55214038a1b5e999fa,التصنيف القانوني: تشغيل النساء\nالباب: تشغيل النساء\nالفصل: na

In [25]:
# =========================================================
# Stage 12.7.5 - Export Clean Manual Review File as XLSX
# إنشاء ملف مراجعة يدوي نظيف بصيغة Excel تحفظ العربي
# =========================================================

import pandas as pd
from pathlib import Path
from IPython.display import display, HTML

candidate_path = STAGE03_DIR / "legal_extra_questions_gold_id_review_candidates.csv"
manual_xlsx_path = STAGE03_DIR / "legal_extra_questions_gold_id_reviewed.xlsx"

assert candidate_path.exists(), f"Candidate file not found: {candidate_path}"

# اقرأ ملف المرشحين بترميز صحيح
df_candidates = pd.read_csv(candidate_path, encoding="utf-8-sig")

# تأكد من وجود أعمدة المراجعة اليدوية
if "manual_gold_article_number" not in df_candidates.columns:
    df_candidates["manual_gold_article_number"] = ""

if "manual_gold_document_unit_id" not in df_candidates.columns:
    df_candidates["manual_gold_document_unit_id"] = ""

if "review_status" not in df_candidates.columns:
    df_candidates["review_status"] = "PENDING_MANUAL_REVIEW"

# احفظ كـ XLSX وليس CSV
with pd.ExcelWriter(manual_xlsx_path, engine="openpyxl") as writer:
    df_candidates.to_excel(writer, index=False, sheet_name="manual_review")

print("Clean manual review Excel file saved to:")
print(manual_xlsx_path)

display(HTML("""
<div dir="rtl" style="text-align:right; line-height:2; font-size:16px;">
<h3>تم إنشاء ملف Excel نظيف للمراجعة اليدوية</h3>
<p>
افتح ملف XLSX وليس CSV، ثم عبئ فقط:
</p>
<ul>
<li>manual_gold_article_number</li>
<li>manual_gold_document_unit_id</li>
</ul>
<p>
بعد التعبئة احفظ الملف بنفس الاسم والصيغة XLSX.
</p>
</div>
"""))

display(df_candidates[[
    "eval_id",
    "question",
    "legal_topic",
    "manual_gold_article_number",
    "manual_gold_document_unit_id"
]].head(5))

Clean manual review Excel file saved to:
C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\legal_extra_questions_gold_id_reviewed.xlsx


,eval_id,question,legal_topic,manual_gold_article_number,manual_gold_document_unit_id
0,LEGAL_EXTRA_001,كم أقصى مدة ممكن تكون فترة التجربة في عقد العمل؟,فترة التجربة,NaN,NaN
1,LEGAL_EXTRA_002,هل يحق للشركة تمدد فترة التجربة بعد ما بدأت؟,فترة التجربة,NaN,NaN
2,LEGAL_EXTRA_003,إذا كنت في فترة التجربة هل أقدر أترك العمل بدون إنذار؟,فترة التجربة,NaN,NaN
3,LEGAL_EXTRA_004,هل لازم تكون فترة التجربة مكتوبة في العقد؟,فترة التجربة,NaN,NaN
4,LEGAL_EXTRA_005,لو انتهت فترة التجربة وسكتت الشركة هل أعتبر مثبت؟,فترة التجربة,NaN,NaN


# after fill the file now merge it with Merge Reviewed Extra Legal Questions into df_eval_run

In [26]:
# =========================================================
# Stage 12.8 - Merge Reviewed Extra Legal Questions into df_eval_run
# دمج الأسئلة القانونية الإضافية بعد مراجعة Golden IDs يدوياً من ملف XLSX
# =========================================================

import pandas as pd
from IPython.display import display, HTML

# ---------------------------------------------------------
# 1) Load manually reviewed Golden ID Excel file
# قراءة ملف المراجعة اليدوية بصيغة XLSX للحفاظ على النص العربي
# ---------------------------------------------------------

review_path = STAGE03_DIR / "legal_extra_questions_gold_id_reviewed.xlsx"

assert review_path.exists(), (
    f"Manual reviewed Excel file not found:\n{review_path}\n\n"
    "Please create/fill the file and save it as:\n"
    "legal_extra_questions_gold_id_reviewed.xlsx"
)

df_reviewed = pd.read_excel(review_path)

print("Reviewed Excel file shape:", df_reviewed.shape)
display(df_reviewed.head(3))


# ---------------------------------------------------------
# 2) Validate required columns
# التحقق من وجود الأعمدة المطلوبة
# ---------------------------------------------------------

required_cols = [
    "eval_id",
    "eval_type",
    "question",
    "manual_gold_article_number",
    "manual_gold_document_unit_id"
]

missing_cols = [c for c in required_cols if c not in df_reviewed.columns]

assert not missing_cols, f"Missing required columns in reviewed Excel file: {missing_cols}"


# ---------------------------------------------------------
# 3) Clean manual fields
# تنظيف الحقول التي تم تعبئتها يدوياً
# ---------------------------------------------------------

df_reviewed["eval_id"] = (
    df_reviewed["eval_id"]
    .astype(str)
    .str.strip()
)

df_reviewed["question"] = (
    df_reviewed["question"]
    .astype(str)
    .str.strip()
)

df_reviewed["manual_gold_article_number"] = (
    df_reviewed["manual_gold_article_number"]
    .astype(str)
    .str.strip()
)

df_reviewed["manual_gold_document_unit_id"] = (
    df_reviewed["manual_gold_document_unit_id"]
    .astype(str)
    .str.strip()
)


# ---------------------------------------------------------
# 4) Check corrupted Arabic questions
# فحص هل الأسئلة العربية انضربت وتحولت إلى علامات استفهام
# ---------------------------------------------------------

corrupted_questions = df_reviewed[
    df_reviewed["question"].astype(str).str.contains(r"\?{3,}", regex=True, na=False)
].copy()

print("Corrupted question rows:", len(corrupted_questions))

if len(corrupted_questions) > 0:
    display(
        corrupted_questions[
            [
                "eval_id",
                "question",
                "manual_gold_article_number",
                "manual_gold_document_unit_id"
            ]
        ].head(20)
    )

assert len(corrupted_questions) == 0, (
    "Some questions are corrupted and appear as question marks. "
    "Do not use CSV/XLS. Recreate the reviewed file as XLSX from the original candidates file."
)


# ---------------------------------------------------------
# 5) Check missing manual Golden IDs
# التأكد من عدم وجود أسئلة بدون Golden ID أو رقم مادة
# ---------------------------------------------------------

invalid_values = ["", "nan", "none", "null", "na", "n/a"]

missing_gold = df_reviewed[
    df_reviewed["manual_gold_document_unit_id"].str.lower().isin(invalid_values)
    | df_reviewed["manual_gold_article_number"].str.lower().isin(invalid_values)
].copy()

print("Questions missing manual Golden ID:", len(missing_gold))

if len(missing_gold) > 0:
    display(
        missing_gold[
            [
                "eval_id",
                "question",
                "manual_gold_article_number",
                "manual_gold_document_unit_id"
            ]
        ]
    )

assert len(missing_gold) == 0, (
    "Some questions still have missing manual_gold_article_number or "
    "manual_gold_document_unit_id. Please complete them in "
    "legal_extra_questions_gold_id_reviewed.xlsx first."
)


# ---------------------------------------------------------
# 6) Detect common wrong entries
# كشف الأخطاء الشائعة مثل وضع text_preview بدلاً من candidate_id
# ---------------------------------------------------------

wrong_id_rows = df_reviewed[
    df_reviewed["manual_gold_document_unit_id"].str.contains(
        r"text_preview|candidate_\d+_text|preview",
        case=False,
        na=False,
        regex=True
    )
].copy()

print("Wrong ID format rows:", len(wrong_id_rows))

if len(wrong_id_rows) > 0:
    display(
        wrong_id_rows[
            [
                "eval_id",
                "question",
                "manual_gold_article_number",
                "manual_gold_document_unit_id"
            ]
        ]
    )

assert len(wrong_id_rows) == 0, (
    "Some rows contain text_preview instead of the real candidate ID. "
    "Use the value from candidate_1_id, candidate_2_id, candidate_3_id, etc."
)


# ---------------------------------------------------------
# 7) Optional check: manual ID should be one of candidate IDs
# فحص اختياري: هل الـ ID اليدوي موجود ضمن المرشحين الخمسة؟
# ---------------------------------------------------------

candidate_id_cols = [
    c for c in df_reviewed.columns
    if c.startswith("candidate_") and c.endswith("_id")
]

if candidate_id_cols:
    def id_exists_in_candidates(row):
        manual_id = str(row["manual_gold_document_unit_id"]).strip()
        candidate_ids = [str(row[c]).strip() for c in candidate_id_cols]
        return manual_id in candidate_ids

    df_reviewed["manual_id_found_in_candidates"] = df_reviewed.apply(
        id_exists_in_candidates,
        axis=1
    )

    not_in_candidates = df_reviewed[
        df_reviewed["manual_id_found_in_candidates"] == False
    ].copy()

    print("Manual IDs not found in top candidate IDs:", len(not_in_candidates))

    if len(not_in_candidates) > 0:
        display(HTML("""
        <div dir="rtl" style="text-align:right; line-height:2; font-size:16px; color:#8a6d3b;">
        <b>تنبيه:</b> توجد بعض الـ IDs غير موجودة ضمن أول 5 مرشحين.
        هذا ليس خطأ بالضرورة إذا اخترت ID صحيحاً من خارج المرشحين،
        لكنه يحتاج مراجعة يدوية دقيقة.
        </div>
        """))

        display(
            not_in_candidates[
                [
                    "eval_id",
                    "question",
                    "manual_gold_article_number",
                    "manual_gold_document_unit_id"
                ]
            ].head(20)
        )


# ---------------------------------------------------------
# 8) Build final extra legal evaluation set
# بناء مجموعة التقييم النهائية للأسئلة القانونية الإضافية
# ---------------------------------------------------------

df_extra_legal_eval = df_reviewed.copy()

df_extra_legal_eval["eval_type"] = "legal_article_retrieval"

df_extra_legal_eval["gold_article_number"] = (
    df_extra_legal_eval["manual_gold_article_number"]
)

df_extra_legal_eval["gold_article_numbers"] = (
    df_extra_legal_eval["manual_gold_article_number"]
)

df_extra_legal_eval["gold_document_unit_id"] = (
    df_extra_legal_eval["manual_gold_document_unit_id"]
)

df_extra_legal_eval["gold_document_unit_ids"] = (
    df_extra_legal_eval["manual_gold_document_unit_id"]
)

df_extra_legal_eval["evaluation_note"] = (
    "Additional manually reviewed legal retrieval question."
)

df_extra_legal_eval = df_extra_legal_eval[
    [
        "eval_id",
        "eval_type",
        "question",
        "gold_document_unit_id",
        "gold_document_unit_ids",
        "gold_article_number",
        "gold_article_numbers",
        "evaluation_note"
    ]
].copy()

print("Extra reviewed legal questions:", len(df_extra_legal_eval))
display(df_extra_legal_eval.head(5))


# ---------------------------------------------------------
# 9) Merge with existing df_eval_run from Cell 28
# دمج الأسئلة الجديدة مع مجموعة التقييم الحالية
# ---------------------------------------------------------

assert "df_eval_run" in globals(), (
    "df_eval_run not found. Please run Cell 28 / Fix FAQ Retrieval Evaluation Set first."
)

before_counts = df_eval_run["eval_type"].value_counts().to_frame("before_count")

df_eval_run_extended = pd.concat(
    [
        df_eval_run,
        df_extra_legal_eval
    ],
    ignore_index=True
)

# Remove duplicate eval_id if any
df_eval_run_extended = (
    df_eval_run_extended
    .drop_duplicates(subset=["eval_id"], keep="last")
    .reset_index(drop=True)
)

df_eval_run = df_eval_run_extended.copy()

after_counts = df_eval_run["eval_type"].value_counts().to_frame("after_count")

print("Updated df_eval_run shape:", df_eval_run.shape)
display(before_counts.join(after_counts, how="outer").fillna(0).astype(int))


# ---------------------------------------------------------
# 10) Save extended evaluation dataset
# حفظ مجموعة التقييم الموسعة
# ---------------------------------------------------------

extended_eval_csv_path = STAGE03_DIR / "stage03_fair_faq_legal_plus_extra_reviewed_eval_dataset.csv"
extended_eval_xlsx_path = STAGE03_DIR / "stage03_fair_faq_legal_plus_extra_reviewed_eval_dataset.xlsx"

df_eval_run.to_csv(
    extended_eval_csv_path,
    index=False,
    encoding="utf-8-sig"
)

with pd.ExcelWriter(extended_eval_xlsx_path, engine="openpyxl") as writer:
    df_eval_run.to_excel(writer, index=False, sheet_name="extended_eval")


display(HTML("""
<div dir="rtl" style="text-align:right; line-height:2; font-size:16px;">
<h3>تم دمج الأسئلة القانونية الإضافية بنجاح</h3>
<p>
الآن أصبح <b>df_eval_run</b> يحتوي على:
</p>
<ul>
<li>أسئلة FAQ المعاد صياغتها.</li>
<li>الأسئلة القانونية الأصلية.</li>
<li>الأسئلة القانونية الجديدة بعد مراجعة Golden IDs يدوياً.</li>
</ul>
<p>
الخطوة التالية: تشغيل <b>Stage 13 - Run Retrieval Experiments</b>.
</p>
</div>
"""))

print("Extended evaluation CSV saved to:")
print(extended_eval_csv_path)

print("\nExtended evaluation Excel saved to:")
print(extended_eval_xlsx_path)

display(df_eval_run["eval_type"].value_counts().to_frame("count"))

Reviewed Excel file shape: (80, 27)


,eval_id,eval_type,question,legal_topic,manual_gold_article_number,manual_gold_document_unit_id,review_status,candidate_1_score,candidate_1_article,candidate_1_id,...,candidate_3_id,candidate_3_text_preview,candidate_4_score,candidate_4_article,candidate_4_id,candidate_4_text_preview,candidate_5_score,candidate_5_article,candidate_5_id,candidate_5_text_preview
0,LEGAL_EXTRA_001,legal_article_retrieval,كم أقصى مدة ممكن تكون فترة التجربة في عقد العمل؟,فترة التجربة,53,b56e9c99f1c551928c8727dae90850186c14efc78d6272f429f0c448add3f3dc,PENDING_MANUAL_REVIEW,0.2556,الرابعة والخمسون,ab8af4e6ecab46b0bf5b588cc55664b9d0a008dc9073912d3417c1f04d083142,...,b56e9c99f1c551928c8727dae90850186c14efc78d6272f429f0c448add3f3dc,التصنيف القانوني: علاقات العمل\nالباب: علاقات العمل\nالفصل: الفصل الأول\nرقم المادة: الثالثة والخمسون\nعنوان المادة:...,0.1002,الثانية,4f2961046b5fffdc637e2b7de4dfb1fd862ccc7b9f3069349681b04ff17df638,التصنيف القانوني: التعريفات / الأحكام العامة\nالباب: التعريفات / الأحكام العامة\nالفصل: الفصل الأول\nرقم المادة: الث...,0.0950,التاسعة والسبعون بعد المائة,cf4e4b2806a0c863c377af5a46b2d45d628aa531ca1024270da12c93a51c2097,التصنيف القانوني: عقد العمل البحري\nالباب: عقد العمل البحري\nالفصل: nan\nرقم المادة: التاسعة والسبعون بعد المائة\nعن...
1,LEGAL_EXTRA_002,legal_article_retrieval,هل يحق للشركة تمدد فترة التجربة بعد ما بدأت؟,فترة التجربة,53,b56e9c99f1c551928c8727dae90850186c14efc78d6272f429f0c448add3f3dc,PENDING_MANUAL_REVIEW,0.1790,الرابعة والخمسون,ab8af4e6ecab46b0bf5b588cc55664b9d0a008dc9073912d3417c1f04d083142,...,dd858095f7b7989c6920172b61ccf992fb378afd325c06fb4eda55d1ceadaf9c,التصنيف القانوني: شروط العمل وظروفه\nالباب: شروط العمل وظروفه\nالفصل: الفصل الثالث\nرقم المادة: الثالثة بعد المائة\n...,0.0518,الثالثة والخمسون,b56e9c99f1c551928c8727dae90850186c14efc78d6272f429f0c448add3f3dc,التصنيف القانوني: علاقات العمل\nالباب: علاقات العمل\nالفصل: الفصل الأول\nرقم المادة: الثالثة والخمسون\nعنوان المادة:...,0.0484,الستون بعد المائة,f3b2d548d622040188c6ebcfda67f959c540fe61dbb6be0a130d3cd508041879,التصنيف القانوني: تشغيل النساء\nالباب: تشغيل النساء\nالفصل: nan\nرقم المادة: الستون بعد المائة\nعنوان المادة: المادة...
2,LEGAL_EXTRA_003,legal_article_retrieval,إذا كنت في فترة التجربة هل أقدر أترك العمل بدون إنذار؟,فترة التجربة,54,ab8af4e6ecab46b0bf5b588cc55664b9d0a008dc9073912d3417c1f04d083142,PENDING_MANUAL_REVIEW,0.1924,الرابعة والخمسون,ab8af4e6ecab46b0bf5b588cc55664b9d0a008dc9073912d3417c1f04d083142,...,93106f92e2e463b2afeee9b680ab418b16cc3907a7b7f1d8308274a5081c8839,التصنيف القانوني: علاقات العمل\nالباب: علاقات العمل\nالفصل: الفصل الثالث\nرقم المادة: الحادية والثمانون\nعنوان الماد...,0.0849,الثالثة بعد المائة,dd858095f7b7989c6920172b61ccf992fb378afd325c06fb4eda55d1ceadaf9c,التصنيف القانوني: شروط العمل وظروفه\nالباب: شروط العمل وظروفه\nالفصل: الفصل الثالث\nرقم المادة: الثالثة بعد المائة\n...,0.0824,الثانية,4f2961046b5fffdc637e2b7de4dfb1fd862ccc7b9f3069349681b04ff17df638,التصنيف القانوني: التعريفات / الأحكام العامة\nالباب: التعريفات / الأحكام العامة\nالفصل: الفصل الأول\nرقم المادة: الث...


Corrupted question rows: 0
Questions missing manual Golden ID: 0
Wrong ID format rows: 0
Manual IDs not found in top candidate IDs: 0
Extra reviewed legal questions: 80


,eval_id,eval_type,question,gold_document_unit_id,gold_document_unit_ids,gold_article_number,gold_article_numbers,evaluation_note
0,LEGAL_EXTRA_001,legal_article_retrieval,كم أقصى مدة ممكن تكون فترة التجربة في عقد العمل؟,b56e9c99f1c551928c8727dae90850186c14efc78d6272f429f0c448add3f3dc,b56e9c99f1c551928c8727dae90850186c14efc78d6272f429f0c448add3f3dc,53,53,Additional manually reviewed legal retrieval question.
1,LEGAL_EXTRA_002,legal_article_retrieval,هل يحق للشركة تمدد فترة التجربة بعد ما بدأت؟,b56e9c99f1c551928c8727dae90850186c14efc78d6272f429f0c448add3f3dc,b56e9c99f1c551928c8727dae90850186c14efc78d6272f429f0c448add3f3dc,53,53,Additional manually reviewed legal retrieval question.
2,LEGAL_EXTRA_003,legal_article_retrieval,إذا كنت في فترة التجربة هل أقدر أترك العمل بدون إنذار؟,ab8af4e6ecab46b0bf5b588cc55664b9d0a008dc9073912d3417c1f04d083142,ab8af4e6ecab46b0bf5b588cc55664b9d0a008dc9073912d3417c1f04d083142,54,54,Additional manually reviewed legal retrieval question.
3,LEGAL_EXTRA_004,legal_article_retrieval,هل لازم تكون فترة التجربة مكتوبة في العقد؟,b56e9c99f1c551928c8727dae90850186c14efc78d6272f429f0c448add3f3dc,b56e9c99f1c551928c8727dae90850186c14efc78d6272f429f0c448add3f3dc,53,53,Additional manually reviewed legal retrieval question.
4,LEGAL_EXTRA_005,legal_article_retrieval,لو انتهت فترة التجربة وسكتت الشركة هل أعتبر مثبت؟,ab8af4e6ecab46b0bf5b588cc55664b9d0a008dc9073912d3417c1f04d083142,ab8af4e6ecab46b0bf5b588cc55664b9d0a008dc9073912d3417c1f04d083142,53,53,Additional manually reviewed legal retrieval question.


Updated df_eval_run shape: (235, 22)


,before_count,after_count
eval_type,,
faq_paraphrase_retrieval,131,131
legal_article_retrieval,104,104


Extended evaluation CSV saved to:
C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\stage03_fair_faq_legal_plus_extra_reviewed_eval_dataset.csv

Extended evaluation Excel saved to:
C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\stage03_fair_faq_legal_plus_extra_reviewed_eval_dataset.xlsx


,count
eval_type,
faq_paraphrase_retrieval,131
legal_article_retrieval,104


In [27]:
# =========================================================
# Stage 13 - Run Retrieval Experiments
# =========================================================

def retrieve_with_config(query, corpus_name, retriever_name, model_key=None, alpha=None, top_k=5):
    start = time.time()
    if retriever_name == "dense":
        results = dense_retrieve(query, corpus_name, model_key, top_k=top_k)
    elif retriever_name == "bm25":
        results = bm25_retrieve(query, corpus_name, top_k=top_k)
    elif retriever_name == "hybrid":
        results = hybrid_retrieve(query, corpus_name, model_key, alpha=alpha, top_k=top_k, candidate_k=CANDIDATE_K)
    elif retriever_name == "hybrid_reranker":
        results = hybrid_rerank_retrieve(query, corpus_name, model_key, alpha=alpha, top_k=top_k, candidate_k=CANDIDATE_K)
    else:
        raise ValueError(f"Unknown retriever: {retriever_name}")
    latency = time.time() - start
    return results, latency

experiment_configs = []

for corpus_name in CORPORA.keys():
    # BM25 does not depend on embedding model
    experiment_configs.append({
        "chunking_strategy": corpus_name,
        "embedding_model": "none",
        "retriever": "bm25",
        "alpha": None,
    })

    for model_key in loaded_embedding_models.keys():
        experiment_configs.append({
            "chunking_strategy": corpus_name,
            "embedding_model": model_key,
            "retriever": "dense",
            "alpha": None,
        })
        for alpha in HYBRID_ALPHAS:
            experiment_configs.append({
                "chunking_strategy": corpus_name,
                "embedding_model": model_key,
                "retriever": "hybrid",
                "alpha": alpha,
            })
        if reranker_model is not None:
            experiment_configs.append({
                "chunking_strategy": corpus_name,
                "embedding_model": model_key,
                "retriever": "hybrid_reranker",
                "alpha": RERANKER_ALPHA,
            })

print("Total experiment configurations:", len(experiment_configs))
display(pd.DataFrame(experiment_configs))

all_detail_rows = []

for exp_idx, cfg in enumerate(experiment_configs, start=1):
    print("=" * 100)
    print(f"Experiment {exp_idx}/{len(experiment_configs)}:", cfg)
    print("=" * 100)

    for _, eval_row in df_eval_run.iterrows():
        query = str(eval_row["question"])
        results, latency = retrieve_with_config(
            query=query,
            corpus_name=cfg["chunking_strategy"],
            retriever_name=cfg["retriever"],
            model_key=None if cfg["embedding_model"] == "none" else cfg["embedding_model"],
            alpha=cfg["alpha"],
            top_k=TOP_K,
        )

        rel = relevance_vector(eval_row, results, k=TOP_K)
        rr = reciprocal_rank(rel)
        ndcg5 = ndcg_at_k(rel)
        hit_at_1 = int(any(rel[:1]))
        hit_at_3 = int(any(rel[:3]))
        hit_at_5 = int(any(rel[:5]))

        top1 = results[0] if results else {}
        all_detail_rows.append({
            "eval_id": eval_row.get("eval_id", ""),
            "eval_type": eval_row.get("eval_type", ""),
            "question": query,
            "gold_document_unit_ids": eval_row.get("gold_document_unit_ids", eval_row.get("gold_document_unit_id", "")),
            "gold_article_numbers": eval_row.get("gold_article_numbers", eval_row.get("gold_article_number", "")),
            "chunking_strategy": cfg["chunking_strategy"],
            "embedding_model": cfg["embedding_model"],
            "retriever": cfg["retriever"],
            "alpha": cfg["alpha"],
            "hit_at_1": hit_at_1,
            "hit_at_3": hit_at_3,
            "hit_at_5": hit_at_5,
            "rr": rr,
            "ndcg_at_5": ndcg5,
            "latency_sec": latency,
            "top1_chunk_id": top1.get("chunk_id", ""),
            "top1_parent_document_id": top1.get("parent_document_id", ""),
            "top1_source_type": top1.get("source_type", ""),
            "top1_article_number": top1.get("article_number_int", ""),
            "top1_article_title": top1.get("article_title", ""),
            "top1_score": top1.get("score", np.nan),
            "top1_retrieval_backend": top1.get("retrieval_backend", ""),
            "top5_parent_document_ids": "|".join([r.get("parent_document_id", "") for r in results[:5]]),
            "top5_article_numbers": "|".join([str(r.get("article_number_int", "")) for r in results[:5]]),
            "top5_relevance": "".join(map(str, rel[:5])),
        })

retrieval_detail = pd.DataFrame(all_detail_rows)
retrieval_detail.to_csv(STAGE03_DIR / "retrieval_evaluation_detailed_results.csv", index=False, encoding="utf-8-sig")
retrieval_detail.to_excel(STAGE03_DIR / "retrieval_evaluation_detailed_results.xlsx", index=False)

print("Detailed results shape:", retrieval_detail.shape)
display(retrieval_detail.head(10))

Total experiment configurations: 22


,chunking_strategy,embedding_model,retriever,alpha
0,structural,none,bm25,NaN
1,structural,e5_large,dense,NaN
2,structural,e5_large,hybrid,0.65
3,structural,e5_large,hybrid,0.80
4,structural,e5_large,hybrid,0.90
5,structural,e5_large,hybrid_reranker,0.80
6,structural,bge_m3,dense,NaN
7,structural,bge_m3,hybrid,0.65
8,structural,bge_m3,hybrid,0.80
9,structural,bge_m3,hybrid,0.90


Experiment 1/22: {'chunking_strategy': 'structural', 'embedding_model': 'none', 'retriever': 'bm25', 'alpha': None}
Experiment 2/22: {'chunking_strategy': 'structural', 'embedding_model': 'e5_large', 'retriever': 'dense', 'alpha': None}
Experiment 3/22: {'chunking_strategy': 'structural', 'embedding_model': 'e5_large', 'retriever': 'hybrid', 'alpha': 0.65}
Experiment 4/22: {'chunking_strategy': 'structural', 'embedding_model': 'e5_large', 'retriever': 'hybrid', 'alpha': 0.8}
Experiment 5/22: {'chunking_strategy': 'structural', 'embedding_model': 'e5_large', 'retriever': 'hybrid', 'alpha': 0.9}
Experiment 6/22: {'chunking_strategy': 'structural', 'embedding_model': 'e5_large', 'retriever': 'hybrid_reranker', 'alpha': 0.8}
Experiment 7/22: {'chunking_strategy': 'structural', 'embedding_model': 'bge_m3', 'retriever': 'dense', 'alpha': None}
Experiment 8/22: {'chunking_strategy': 'structural', 'embedding_model': 'bge_m3', 'retriever': 'hybrid', 'alpha': 0.65}
Experiment 9/22: {'chunking_st

,eval_id,eval_type,question,gold_document_unit_ids,gold_article_numbers,chunking_strategy,embedding_model,retriever,alpha,hit_at_1,...,top1_chunk_id,top1_parent_document_id,top1_source_type,top1_article_number,top1_article_title,top1_score,top1_retrieval_backend,top5_parent_document_ids,top5_article_numbers,top5_relevance
0,FAQ_INDEXED_RETRIEVAL_0001,faq_paraphrase_retrieval,أريد معرفة ما إذا كانت توفر البوابة المعلومات ومجموعة الخدمات الخاصة بالمرأة؟,1d2d4109d0423311606722e82dfd41c20a7f7742c83169619385ec8d2a4c2155,,structural,none,bm25,NaN,1,...,STRUCT_000488,1d2d4109d0423311606722e82dfd41c20a7f7742c83169619385ec8d2a4c2155,faq,nan,nan,48.934719,bm25,1d2d4109d0423311606722e82dfd41c20a7f7742c83169619385ec8d2a4c2155|5d4b007a1cabc286063fd6b2e27c8fdf7ff5173befb7906c8dd...,nan|nan|nan|nan|nan,10000
1,FAQ_INDEXED_RETRIEVAL_0002,faq_paraphrase_retrieval,أريد توضيح الحكم النظامي بخصوص: ماهي أوقات العمل الرسمية للفروع؟,271c55cb0f0f2d621adfb634fce4722dc68011cfcef490eec0337fb3104fcedb,,structural,none,bm25,NaN,1,...,STRUCT_000292,271c55cb0f0f2d621adfb634fce4722dc68011cfcef490eec0337fb3104fcedb,faq,nan,nan,32.355991,bm25,271c55cb0f0f2d621adfb634fce4722dc68011cfcef490eec0337fb3104fcedb|3601b726fee9518b82fa8eec6bbc14b207e6eebc11f8d239d14...,nan|nan|nan|nan|nan,10000
2,FAQ_INDEXED_RETRIEVAL_0003,faq_paraphrase_retrieval,أريد توضيح الحكم النظامي بخصوص: شاهدتُ حالة عنف وقعت أمامي، ماذا أفعل؟,8f9d950f458be122538bbb4a6e889159595c47b5b21a723af28c00e7166a1468,,structural,none,bm25,NaN,1,...,STRUCT_000509,8f9d950f458be122538bbb4a6e889159595c47b5b21a723af28c00e7166a1468,faq,nan,nan,47.533714,bm25,8f9d950f458be122538bbb4a6e889159595c47b5b21a723af28c00e7166a1468|f2850b21dfa09eddda39fa687fb317959b4517d80a68226dd55...,nan|nan|nan|nan|nan,10000
3,FAQ_INDEXED_RETRIEVAL_0004,faq_paraphrase_retrieval,أريد معرفة ما إذا كان يتم احتساب الفرد الذي يعمل لدى أكثر من منشأة في نفس الوقت ضمن نسبة التوطين؟,a78f94c86dd92b762fcc3c865abb5182e6facb506a9d5512a0f44e5b9c30e184,,structural,none,bm25,NaN,1,...,STRUCT_000374,a78f94c86dd92b762fcc3c865abb5182e6facb506a9d5512a0f44e5b9c30e184,faq,nan,nan,78.836662,bm25,a78f94c86dd92b762fcc3c865abb5182e6facb506a9d5512a0f44e5b9c30e184|5e8999400c09e0a2277f41f0e97cff194b05c8cf35263e18cbe...,nan|nan|nan|nan|nan,10000
4,FAQ_INDEXED_RETRIEVAL_0005,faq_paraphrase_retrieval,هل يستطيع إصدار تأشيرة لجنسية أو مهنة غير مدرجة في مساند؟,ac0493e4a60d1923d3e181202d0bfd389b31023b65ec159be502aee177c1a7a3,,structural,none,bm25,NaN,1,...,STRUCT_000323,ac0493e4a60d1923d3e181202d0bfd389b31023b65ec159be502aee177c1a7a3,faq,nan,nan,60.362209,bm25,ac0493e4a60d1923d3e181202d0bfd389b31023b65ec159be502aee177c1a7a3|146fb1672fe7cc7a24fca0cb4eb594ee9b1ec3edd95367b957a...,nan|38.0|nan|nan|45.0,10000
5,FAQ_INDEXED_RETRIEVAL_0006,faq_paraphrase_retrieval,أريد معرفة ما إذا كان يتم إيواء المرأة التي تعرضت للعنف؟,4b4e32fe41f1745800e5617e787e755d1bd2198efdceb01bb0f43a814b09205a,,structural,none,bm25,NaN,1,...,STRUCT_000500,4b4e32fe41f1745800e5617e787e755d1bd2198efdceb01bb0f43a814b09205a,faq,nan,nan,47.889797,bm25,4b4e32fe41f1745800e5617e787e755d1bd2198efdceb01bb0f43a814b09205a|5eba454702f0e063612cb5d6d44774a791231781c240e5f90e4...,nan|nan|nan|nan|nan,10000
6,FAQ_INDEXED_RETRIEVAL_0007,faq_paraphrase_retrieval,أريد توضيح الحكم النظامي بخصوص: ما مدى امكانية الجمع بين المكافأة المقطوعة للمستشارين غير المتفرغين ومكافأة العمل خا...,486f8265889bb1cb71b039ce24e5d65ba171764ac0fd47ef3693ce9d9b64c9b6,,structural,none,bm25,NaN,1,...,STRUCT_000612,486f8265889bb1cb71b039ce24e5d65ba171764ac0fd47ef3693ce9d9b64c9b6,faq,nan,nan,110.166412,bm25,486f8265889bb1cb71b039ce24e5d65ba171764ac0fd47ef3693ce9d9b64c9b6|3601b726fee9518b82fa8eec6bbc14b207e6eebc11f8d239d14...,nan|nan|nan|nan|nan,10000
7,FAQ_INDEXED_RETRIEVAL_0008,faq_paraphrase_retrieval,أريد معرفة ما إذا كان الخدمة تتيح تحديث بيانات العامل؟,efee6aee87704b6999b8578223366ee7497784934576b50a92d7e4493d8133d9,,structural,none,bm25,NaN,1,...,STRUCT_000343,efee6aee87704b6999b8578223366ee7497784934576b50a92d7e4493d8133d9,faq,nan,nan,26.83593

In [28]:
# =========================================================
# Check Retrieval Detail Before Summary - Professional View
# فحص نتائج الاسترجاع التفصيلية قبل بناء الملخص
# =========================================================

import pandas as pd
from IPython.display import display, HTML

# ---------------------------------------------------------
# 1) Professional display helper
# ---------------------------------------------------------

def display_professional_table(df, title="Table", max_rows=10):
    """
    Display a wide, scrollable, professional-looking DataFrame in Jupyter.
    """
    if df is None or len(df) == 0:
        display(HTML(f"""
        <div style="
            direction: rtl;
            text-align: right;
            font-family: Tahoma, Arial;
            border: 1px solid #f5c2c7;
            background: #f8d7da;
            color: #842029;
            padding: 14px;
            border-radius: 8px;
            margin: 10px 0;
        ">
            <b>{title}</b><br>
            لا توجد بيانات لعرضها.
        </div>
        """))
        return

    view_df = df.head(max_rows).copy()

    html_table = view_df.to_html(
        index=False,
        escape=True,
        border=0
    )

    display(HTML(f"""
    <div style="
        direction: rtl;
        text-align: right;
        font-family: Tahoma, Arial;
        margin: 18px 0 8px 0;
        font-size: 18px;
        font-weight: 700;
        color: #17324d;
    ">
        {title}
    </div>

    <div style="
        width: 100%;
        overflow-x: auto;
        overflow-y: hidden;
        border: 1px solid #d9e2ec;
        border-radius: 10px;
        background: #ffffff;
        padding: 0;
        margin-bottom: 20px;
    ">
        <style>
            .professional-wide-table table {{
                border-collapse: collapse;
                width: max-content;
                min-width: 1400px;
                font-family: Tahoma, Arial, sans-serif;
                font-size: 13px;
            }}

            .professional-wide-table th {{
                background-color: #1f4e79;
                color: white;
                text-align: center;
                padding: 10px 12px;
                border: 1px solid #d9e2ec;
                white-space: nowrap;
                font-weight: 700;
            }}

            .professional-wide-table td {{
                padding: 9px 12px;
                border: 1px solid #e5eaf0;
                vertical-align: top;
                white-space: normal;
                max-width: 420px;
                word-wrap: break-word;
                text-align: left;
                direction: ltr;
            }}

            .professional-wide-table tr:nth-child(even) {{
                background-color: #f8fbfd;
            }}

            .professional-wide-table tr:hover {{
                background-color: #eef6ff;
            }}

            .professional-wide-table td:nth-child(3),
            .professional-wide-table td:nth-child(4),
            .professional-wide-table td:nth-child(5) {{
                direction: rtl;
                text-align: right;
                min-width: 300px;
            }}
        </style>

        <div class="professional-wide-table">
            {html_table}
        </div>
    </div>
    """))


def display_status_card(title, message, status="info"):
    """
    Display a professional status card.
    """
    colors = {
        "success": ("#d1e7dd", "#0f5132", "#badbcc"),
        "warning": ("#fff3cd", "#664d03", "#ffecb5"),
        "danger":  ("#f8d7da", "#842029", "#f5c2c7"),
        "info":    ("#cff4fc", "#055160", "#b6effb"),
    }

    bg, color, border = colors.get(status, colors["info"])

    display(HTML(f"""
    <div style="
        direction: rtl;
        text-align: right;
        font-family: Tahoma, Arial;
        background: {bg};
        color: {color};
        border: 1px solid {border};
        padding: 14px 16px;
        border-radius: 10px;
        margin: 12px 0;
        line-height: 1.8;
        font-size: 15px;
    ">
        <b>{title}</b><br>
        {message}
    </div>
    """))


# ---------------------------------------------------------
# 2) Check retrieval_detail
# ---------------------------------------------------------

if "retrieval_detail" not in globals():
    display_status_card(
        title="retrieval_detail غير موجود",
        message="يجب تشغيل خلية Stage 13 - Run Retrieval Experiments أولاً قبل بناء الملخص.",
        status="danger"
    )

else:
    display_status_card(
        title="تم العثور على retrieval_detail",
        message="تم العثور على نتائج الاسترجاع التفصيلية، ويمكن الآن مراجعتها قبل بناء الجداول النهائية.",
        status="success"
    )

    # -----------------------------------------------------
    # 3) Summary cards
    # -----------------------------------------------------

    total_rows = len(retrieval_detail)
    total_cols = retrieval_detail.shape[1]

    unique_questions = (
        retrieval_detail["eval_id"].nunique()
        if "eval_id" in retrieval_detail.columns
        else "N/A"
    )

    unique_configs = (
        retrieval_detail["experiment_name"].nunique()
        if "experiment_name" in retrieval_detail.columns
        else (
            retrieval_detail["retriever_name"].nunique()
            if "retriever_name" in retrieval_detail.columns
            else "N/A"
        )
    )

    display(HTML(f"""
    <div style="
        display: flex;
        gap: 14px;
        flex-wrap: wrap;
        margin: 14px 0 22px 0;
        font-family: Tahoma, Arial;
        direction: rtl;
    ">
        <div style="flex:1; min-width:180px; background:#f8fbfd; border:1px solid #d9e2ec; border-radius:10px; padding:14px; text-align:center;">
            <div style="font-size:13px; color:#52616b;">عدد الصفوف التفصيلية</div>
            <div style="font-size:24px; font-weight:700; color:#1f4e79;">{total_rows:,}</div>
        </div>

        <div style="flex:1; min-width:180px; background:#f8fbfd; border:1px solid #d9e2ec; border-radius:10px; padding:14px; text-align:center;">
            <div style="font-size:13px; color:#52616b;">عدد الأعمدة</div>
            <div style="font-size:24px; font-weight:700; color:#1f4e79;">{total_cols}</div>
        </div>

        <div style="flex:1; min-width:180px; background:#f8fbfd; border:1px solid #d9e2ec; border-radius:10px; padding:14px; text-align:center;">
            <div style="font-size:13px; color:#52616b;">عدد أسئلة التقييم</div>
            <div style="font-size:24px; font-weight:700; color:#1f4e79;">{unique_questions}</div>
        </div>

        <div style="flex:1; min-width:180px; background:#f8fbfd; border:1px solid #d9e2ec; border-radius:10px; padding:14px; text-align:center;">
            <div style="font-size:13px; color:#52616b;">عدد إعدادات الاسترجاع</div>
            <div style="font-size:24px; font-weight:700; color:#1f4e79;">{unique_configs}</div>
        </div>
    </div>
    """))

    # -----------------------------------------------------
    # 4) Display eval type distribution if available
    # -----------------------------------------------------

    if "eval_type" in retrieval_detail.columns:
        eval_type_summary = (
            retrieval_detail["eval_type"]
            .value_counts()
            .reset_index()
        )

        eval_type_summary.columns = ["eval_type", "rows_count"]

        display_professional_table(
            eval_type_summary,
            title="توزيع النتائج التفصيلية حسب نوع التقييم",
            max_rows=20
        )

    # -----------------------------------------------------
    # 5) Display selected important columns first
    # -----------------------------------------------------

    preferred_cols = [
        "experiment_name",
        "retriever_name",
        "chunking_strategy",
        "embedding_model",
        "retrieval_method",
        "reranker_model",
        "alpha",
        "eval_id",
        "eval_type",
        "question",
        "gold_document_unit_id",
        "gold_article_number",
        "rank",
        "retrieved_document_unit_id",
        "retrieved_article_number",
        "score",
        "is_hit",
        "hit_at_1",
        "hit_at_3",
        "hit_at_5",
        "reciprocal_rank",
        "ndcg_at_5",
        "latency_sec",
    ]

    existing_preferred_cols = [
        col for col in preferred_cols
        if col in retrieval_detail.columns
    ]

    if len(existing_preferred_cols) > 0:
        retrieval_detail_view = retrieval_detail[existing_preferred_cols].copy()
    else:
        retrieval_detail_view = retrieval_detail.copy()

    display_professional_table(
        retrieval_detail_view,
        title="عينة احترافية من retrieval_detail بعد تشغيل تجارب الاسترجاع",
        max_rows=10
    )

    # -----------------------------------------------------
    # 6) Academic note
    # -----------------------------------------------------

    display(HTML("""
    <div dir="rtl" style="
        text-align:right;
        font-family:Tahoma, Arial;
        line-height:2;
        font-size:15px;
        background:#f8fbfd;
        border-right:5px solid #1f4e79;
        padding:14px 18px;
        border-radius:8px;
        margin-top:15px;
    ">
        <b>ملاحظة تفسيرية:</b><br>
        هذا الجدول يمثل النتائج التفصيلية لكل سؤال ولكل إعداد من إعدادات الاسترجاع.
        لذلك قد يكون عدد الصفوف كبيراً لأنه يساوي تقريباً عدد أسئلة التقييم مضروباً في عدد تجارب الاسترجاع.
        يتم استخدام هذا الجدول لاحقاً لبناء ملخصات الأداء مثل Recall@1 و Recall@5 و MRR و nDCG@5.
    </div>
    """))

eval_type,rows_count
faq_paraphrase_retrieval,2882
legal_article_retrieval,2288


chunking_strategy,embedding_model,alpha,eval_id,eval_type,question,hit_at_1,hit_at_3,hit_at_5,ndcg_at_5,latency_sec
structural,none,NaN,FAQ_INDEXED_RETRIEVAL_0001,faq_paraphrase_retrieval,أريد معرفة ما إذا كانت توفر البوابة المعلومات ومجموعة الخدمات الخاصة بالمرأة؟,1,1,1,1.0,0.009485
structural,none,NaN,FAQ_INDEXED_RETRIEVAL_0002,faq_paraphrase_retrieval,أريد توضيح الحكم النظامي بخصوص: ماهي أوقات العمل الرسمية للفروع؟,1,1,1,1.0,0.010098
structural,none,NaN,FAQ_INDEXED_RETRIEVAL_0003,faq_paraphrase_retrieval,أريد توضيح الحكم النظامي بخصوص: شاهدتُ حالة عنف وقعت أمامي، ماذا أفعل؟,1,1,1,1.0,0.011588
structural,none,NaN,FAQ_INDEXED_RETRIEVAL_0004,faq_paraphrase_retrieval,أريد معرفة ما إذا كان يتم احتساب الفرد الذي يعمل لدى أكثر من منشأة في نفس الوقت ضمن نسبة التوطين؟,1,1,1,1.0,0.008569
structural,none,NaN,FAQ_INDEXED_RETRIEVAL_0005,faq_paraphrase_retrieval,هل يستطيع إصدار تأشيرة لجنسية أو مهنة غير مدرجة في مساند؟,1,1,1,1.0,0.009011
structural,none,NaN,FAQ_INDEXED_RETRIEVAL_0006,faq_paraphrase_retrieval,أريد معرفة ما إذا كان يتم إيواء المرأة التي تعرضت للعنف؟,1,1,1,1.0,0.010011
structural,none,NaN,FAQ_INDEXED_RETRIEVAL_0007,faq_paraphrase_retrieval,أريد توضيح الحكم النظامي بخصوص: ما مدى امكانية الجمع بين المكافأة المقطوعة للمستشارين غير المتفرغين ومكافأة العمل خارج وقت الدوام الرسمي وأيام العطل الرسمية والأعياد؟,1,1,1,1.0,0.011218
structural,none,NaN,FAQ_INDEXED_RETRIEVAL_0008,faq_paraphrase_retrieval,أريد معرفة ما إذا كان الخدمة تتيح تحديث بيانات العامل؟,1,1,1,1.0,0.009228
structural,none,NaN,FAQ_INDEXED_RETRIEVAL_0009,faq_paraphrase_retrieval,أريد توضيح الحكم النظامي بخصوص: ماهي خدمة قرار مكافأة الطلبة الجامعين لذوي الإعاقة؟,1,1,1,1.0,0.007488
structural,none,NaN,FAQ_INDEXED_RETRIEVAL_0010,faq_paraphrase_retrieval,أريد توضيح الحكم النظامي بخصوص: العديد يطالبون بشرح البرنامج وشرح للدليل الإجرائي بشكل مبسط وواضح؟,1,1,1,1.0,0.007526


In [29]:
print("retrieval_detail shape:", retrieval_detail.shape)

quick_check = retrieval_detail.groupby(
    ["chunking_strategy", "embedding_model", "retriever", "alpha"],
    dropna=False
)[["hit_at_1", "hit_at_3", "hit_at_5", "rr", "ndcg_at_5"]].mean().reset_index()

quick_check = quick_check.sort_values(
    ["rr", "hit_at_5", "hit_at_1"],
    ascending=[False, False, False]
)

display(quick_check.head(10))

retrieval_detail shape: (5170, 25)


,chunking_strategy,embedding_model,retriever,alpha,hit_at_1,hit_at_3,hit_at_5,rr,ndcg_at_5
15,structural,bge_m3,hybrid_reranker,0.8,0.761702,0.927660,0.974468,0.847234,0.877259
4,fixed_size,bge_m3,hybrid_reranker,0.8,0.757447,0.931915,0.974468,0.845461,0.876042
9,fixed_size,e5_large,hybrid_reranker,0.8,0.740426,0.914894,0.953191,0.828794,0.858371
20,structural,e5_large,hybrid_reranker,0.8,0.740426,0.906383,0.948936,0.824681,0.854341
0,fixed_size,bge_m3,dense,NaN,0.706383,0.876596,0.914894,0.789078,0.819602
3,fixed_size,bge_m3,hybrid,0.9,0.685106,0.851064,0.906383,0.766667,0.802040
11,structural,bge_m3,dense,NaN,0.680851,0.851064,0.910638,0.766525,0.801081
14,structural,bge_m3,hybrid,0.9,0.668085,0.825532,0.910638,0.753617,0.792710
2,fixed_size,bge_m3,hybrid,0.8,0.646809,0.800000,0.855319,0.724326,0.758882
5,fixed_size,e5_large,dense,NaN,0.634043,0.791489,0.855319,0.717518,0.753317


## فحص سريع لنتائج تجارب الاسترجاع

بعد تشغيل تجارب الاسترجاع، تم إنشاء جدول النتائج التفصيلية `retrieval_detail` بالشكل التالي:

```text
retrieval_detail shape: (5170, 25)
```

وهذا يعني أن مرحلة التقييم أنتجت **5,170 صفًا تفصيليًا** و **25 عمودًا**. هذا الرقم منطقي لأن مجموعة التقييم النهائية تحتوي على:

```text
235 سؤال تقييم × 22 إعداد استرجاع = 5,170 نتيجة تفصيلية
```

تم تجميع النتائج حسب طريقة التقسيم، ونموذج التضمين، وطريقة الاسترجاع، وقيمة `alpha`. ثم تم حساب متوسط المقاييس التالية:

* `hit_at_1`
* `hit_at_3`
* `hit_at_5`
* `rr`
* `ndcg_at_5`

أظهرت النتائج أن أفضل إعداد في هذا الفحص السريع هو:

```text
structural + bge_m3 + hybrid_reranker + alpha 0.8
```

حيث حقق النتائج التالية:

| المقياس         | النتيجة |
| --------------- | ------: |
| Hit@1           |  0.7617 |
| Hit@3           |  0.9277 |
| Hit@5           |  0.9745 |
| Reciprocal Rank |  0.8472 |
| nDCG@5          |  0.8773 |

تعني هذه النتائج أن المرجع الصحيح ظهر في المرتبة الأولى في حوالي **76.17%** من الأسئلة، وظهر ضمن أفضل ثلاث نتائج في حوالي **92.77%** من الأسئلة، وضمن أفضل خمس نتائج في حوالي **97.45%** من الأسئلة.

وتعد نتيجة `Hit@5` مهمة جدًا في أنظمة RAG، لأن نموذج اللغة لا يعتمد فقط على أول نتيجة مسترجعة، بل يمكنه استخدام عدة سياقات مسترجعة لتوليد الإجابة النهائية.

كما أظهرت النتائج أن استخدام `hybrid_reranker` أعطى أفضل أداء مقارنة بالاسترجاع الكثيف فقط أو الاسترجاع الهجين بدون إعادة ترتيب. وهذا يدل على أن طبقة إعادة الترتيب تساعد في رفع الوثيقة الصحيحة إلى مراتب أعلى.

كذلك أظهر نموذج `bge_m3` أداءً أفضل من `e5_large` في أفضل الإعدادات، مما يجعله الخيار الأنسب كطبقة تضمين في هذا المشروع.

بناءً على هذا الفحص، يمكن اعتماد الإعداد التالي كأفضل إعداد مبدئي للمرحلة التالية:

```text
structural + bge_m3 + hybrid_reranker + alpha 0.8
```

وسيتم استخدام هذا الإعداد كطبقة استرجاع أساسية قبل ربط النظام بطبقة توليد الإجابة بواسطة نموذج اللغة الكبير.


In [30]:
# =========================================================
# Stage 14 - Retrieval Evaluation Summary and Best Model Selection
# نسخة أسرع وأكثر استقراراً
# =========================================================

import json
import pandas as pd
from IPython.display import display, HTML

print("Building retrieval evaluation summary...")
print("retrieval_detail rows:", len(retrieval_detail))

summary_cols = ["chunking_strategy", "embedding_model", "retriever", "alpha"]

retrieval_summary = summarize_eval_rows(retrieval_detail, summary_cols)

# Sort by primary academic metric: MRR, then Recall@5, then Recall@1
retrieval_summary = retrieval_summary.sort_values(
    ["mrr", "recall_at_5", "recall_at_1"],
    ascending=[False, False, False]
).reset_index(drop=True)

retrieval_summary.insert(0, "rank", range(1, len(retrieval_summary) + 1))

# ---------------------------------------------------------
# Save CSV first
# ---------------------------------------------------------
retrieval_summary_csv = STAGE03_DIR / "retrieval_evaluation_summary.csv"
retrieval_summary.to_csv(
    retrieval_summary_csv,
    index=False,
    encoding="utf-8-sig"
)

# ---------------------------------------------------------
# Save Excel safely
# ---------------------------------------------------------
retrieval_summary_xlsx = STAGE03_DIR / "retrieval_evaluation_summary.xlsx"

try:
    retrieval_summary.to_excel(
        retrieval_summary_xlsx,
        index=False
    )
    excel_status = "Saved"
except Exception as e:
    excel_status = f"Excel save skipped/error: {e}"

# ---------------------------------------------------------
# Best configuration
# ---------------------------------------------------------
best_config = retrieval_summary.iloc[0].to_dict()

with open(STAGE03_DIR / "best_retrieval_config.json", "w", encoding="utf-8") as f:
    json.dump(best_config, f, ensure_ascii=False, indent=2)

# ---------------------------------------------------------
# Display professionally
# ---------------------------------------------------------

display(HTML("""
<div dir="rtl" style="text-align:right; line-height:2; font-size:17px;">
<h3>أفضل إعداد للاسترجاع</h3>
<p>
يعرض الجدول التالي أفضل إعداد تم اختياره بناءً على أعلى قيمة 
<span dir="ltr" style="display:inline-block;">MRR</span>
ثم 
<span dir="ltr" style="display:inline-block;">Recall@5</span>
ثم 
<span dir="ltr" style="display:inline-block;">Recall@1</span>.
</p>
</div>
"""))

display(pd.DataFrame([best_config]))

display(HTML("""
<div dir="rtl" style="text-align:right; line-height:2; font-size:17px;">
<h3>أفضل 10 إعدادات للاسترجاع</h3>
<p>
لتحسين سرعة العرض داخل النوتبوك، يتم عرض أفضل 10 إعدادات فقط، بينما يتم حفظ الجدول الكامل في ملفات خارجية.
</p>
</div>
"""))

top10_summary = retrieval_summary.head(10).copy()
display(top10_summary)

print("Saved files:")
print("CSV:", retrieval_summary_csv)
print("Excel:", retrieval_summary_xlsx, "|", excel_status)
print("Best config:", STAGE03_DIR / "best_retrieval_config.json")

print("\nSummary completed successfully.")

Building retrieval evaluation summary...
retrieval_detail rows: 5170


,rank,chunking_strategy,embedding_model,retriever,alpha,questions,recall_at_1,recall_at_3,recall_at_5,mrr,ndcg_at_5,avg_latency_sec
0,1,structural,bge_m3,hybrid_reranker,0.8,235,0.7617,0.9277,0.9745,0.8472,0.8773,0.407


,rank,chunking_strategy,embedding_model,retriever,alpha,questions,recall_at_1,recall_at_3,recall_at_5,mrr,ndcg_at_5,avg_latency_sec
0,1,structural,bge_m3,hybrid_reranker,0.80,235,0.7617,0.9277,0.9745,0.8472,0.8773,0.4070
1,2,fixed_size,bge_m3,hybrid_reranker,0.80,235,0.7574,0.9319,0.9745,0.8455,0.8760,0.4157
2,3,fixed_size,e5_large,hybrid_reranker,0.80,235,0.7404,0.9149,0.9532,0.8288,0.8584,0.3705
3,4,structural,e5_large,hybrid_reranker,0.80,235,0.7404,0.9064,0.9489,0.8247,0.8543,0.3644
4,5,fixed_size,bge_m3,hybrid,0.90,235,0.6851,0.8511,0.9064,0.7667,0.8020,0.0229
5,6,structural,bge_m3,hybrid,0.90,235,0.6681,0.8255,0.9106,0.7536,0.7927,0.0237
6,7,fixed_size,bge_m3,hybrid,0.80,235,0.6468,0.8000,0.8553,0.7243,0.7589,0.0252
7,8,structural,bge_m3,hybrid,0.80,235,0.6426,0.7830,0.8426,0.7162,0.7489,0.0249
8,9,fixed_size,bge_m3,hybrid,0.65,235,0.6255,0.7532,0.8128,0.6965,0.7276,0.0234
9,10,structural,bge_m3,hybrid,0.65,235,0.6213,0.7574,0.8043,0.6896,0.7206,0.0228


Saved files:
CSV: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\retrieval_evaluation_summary.csv
Excel: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\retrieval_evaluation_summary.xlsx | Saved
Best config: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\best_retrieval_config.json

Summary completed successfully.


## ملخص نتائج Stage 14

في هذه المرحلة تم تلخيص نتائج `retrieval_detail` الناتجة من Stage 13 لاختيار أفضل إعداد للاسترجاع. تم تقييم **235 سؤالًا** عبر **22 إعدادًا مختلفًا**، مما أنتج **5,170 نتيجة تفصيلية**.

تم ترتيب الإعدادات بناءً على أعلى قيمة `MRR`، ثم `Recall@5`، ثم `Recall@1`.

أفضل إعداد كان:

```text
structural + bge_m3 + hybrid_reranker + alpha 0.8
```

وحقق النتائج التالية:

| المقياس         |        النتيجة |
| --------------- | -------------: |
| Recall@1        |         0.7617 |
| Recall@3        |         0.9277 |
| Recall@5        |         0.9745 |
| MRR             |         0.8472 |
| nDCG@5          |         0.8773 |
| Average Latency | 0.4070 seconds |

تدل هذه النتائج على أن النظام استرجع المرجع الصحيح في المرتبة الأولى في حوالي **76.17%** من الأسئلة، وضمن أفضل خمس نتائج في حوالي **97.45%** من الأسئلة. وتعد نتيجة `Recall@5` مهمة جدًا في أنظمة RAG، لأن نموذج اللغة يمكنه استخدام أكثر من سياق مسترجع عند توليد الإجابة النهائية.

كما أظهرت النتائج أن استخدام `hybrid_reranker` أعطى أداءً أفضل من الاسترجاع الهجين بدون إعادة ترتيب، مما يؤكد أهمية طبقة إعادة الترتيب في تحسين موقع الوثيقة الصحيحة ضمن النتائج.

بناءً على ذلك، سيتم اعتماد الإعداد التالي كأفضل إعداد للمرحلة التالية:

```text
structural + bge_m3 + hybrid_reranker + alpha 0.8
```

وسيتم استخدامه كطبقة الاسترجاع الأساسية قبل ربط النظام بطبقة توليد الإجابة بواسطة نموذج اللغة الكبير.


In [31]:
# تشخيص: توزيع الأسئلة حسب النوع (نسخة محمية)
qcol = "query" if "query" in retrieval_detail.columns else \
       ("question" if "question" in retrieval_detail.columns else None)

print("أعمدة retrieval_detail المتاحة:")
print(list(retrieval_detail.columns))

if "eval_type" in retrieval_detail.columns and qcol is not None:
    uniq = retrieval_detail.drop_duplicates(qcol)
    display(uniq["eval_type"].value_counts(dropna=False).to_frame("count"))
elif "eval_type" in retrieval_detail.columns:
    display(retrieval_detail["eval_type"].value_counts(dropna=False).to_frame("count"))
else:
    print("لا يوجد عمود eval_type في retrieval_detail.")
    print("لكن النتائج موجودة أصلاً في جدول performance_by_type السابق.")

أعمدة retrieval_detail المتاحة:
['eval_id', 'eval_type', 'question', 'gold_document_unit_ids', 'gold_article_numbers', 'chunking_strategy', 'embedding_model', 'retriever', 'alpha', 'hit_at_1', 'hit_at_3', 'hit_at_5', 'rr', 'ndcg_at_5', 'latency_sec', 'top1_chunk_id', 'top1_parent_document_id', 'top1_source_type', 'top1_article_number', 'top1_article_title', 'top1_score', 'top1_retrieval_backend', 'top5_parent_document_ids', 'top5_article_numbers', 'top5_relevance']


,count
eval_type,
faq_paraphrase_retrieval,131
legal_article_retrieval,104


In [32]:
# =========================================================
# Diagnostic - Performance by Evaluation Type
# =========================================================

best = retrieval_summary.iloc[0]

best_rows = retrieval_detail[
    (retrieval_detail["chunking_strategy"] == best["chunking_strategy"]) &
    (retrieval_detail["embedding_model"] == best["embedding_model"]) &
    (retrieval_detail["retriever"] == best["retriever"])
].copy()

if pd.isna(best["alpha"]):
    best_rows = best_rows[best_rows["alpha"].isna()]
else:
    best_rows = best_rows[best_rows["alpha"].astype(float).round(4) == float(best["alpha"])]

performance_by_type = (
    best_rows
    .groupby("eval_type")
    .agg(
        questions=("question", "count"),
        recall_at_1=("hit_at_1", "mean"),
        recall_at_3=("hit_at_3", "mean"),
        recall_at_5=("hit_at_5", "mean"),
        mrr=("rr", "mean"),
        ndcg_at_5=("ndcg_at_5", "mean"),
        avg_latency_sec=("latency_sec", "mean"),
    )
    .reset_index()
)

display(performance_by_type)

,eval_type,questions,recall_at_1,recall_at_3,recall_at_5,mrr,ndcg_at_5,avg_latency_sec
0,faq_paraphrase_retrieval,131,0.977099,0.992366,1.000000,0.986641,0.990019,0.416611
1,legal_article_retrieval,104,0.490385,0.846154,0.942308,0.671635,0.735224,0.394814


## التحليل التشخيصي للأداء حسب نوع السؤال

بعد اختيار أفضل إعداد للاسترجاع في Stage 14، تم إجراء تحليل تشخيصي إضافي لقياس أداء الإعداد الأفضل حسب نوع السؤال. هذه الخطوة مهمة لأن المتوسط العام قد يخفي الفرق بين أداء النظام في أسئلة FAQ وأدائه في الأسئلة القانونية.

تم استخدام أفضل إعداد تم اختياره، ثم تم تجميع النتائج حسب:

```text
eval_type
```

والهدف من هذه الخطوة هو معرفة ما إذا كان النظام يؤدي بنفس القوة في جميع أنواع الأسئلة، أم أن هناك فرقًا بين استرجاع FAQ واسترجاع المواد القانونية.

---

### أنواع الأسئلة في التقييم

تتكون مجموعة التقييم النهائية من نوعين رئيسيين:

| نوع التقييم                | الوصف                                                        |
| -------------------------- | ------------------------------------------------------------ |
| `faq_paraphrase_retrieval` | أسئلة FAQ تمت إعادة صياغتها لتقليل المطابقة النصية المباشرة. |
| `legal_article_retrieval`  | أسئلة قانونية طبيعية يجب ربطها بالمادة القانونية الصحيحة.    |

بعد إضافة الأسئلة القانونية الجديدة، أصبح توزيع الأسئلة كالتالي:

| نوع التقييم              | عدد الأسئلة |
| ------------------------ | ----------: |
| FAQ Paraphrase Retrieval |         131 |
| Legal Article Retrieval  |         104 |

وهذا جعل التقييم أكثر توازنًا من السابق، حيث كانت الأسئلة القانونية في البداية 24 سؤالًا فقط.

---

### النتائج حسب نوع السؤال

أظهر أفضل إعداد للاسترجاع النتائج التالية:

| نوع التقييم              | عدد الأسئلة | Recall@1 | Recall@3 | Recall@5 |    MRR | nDCG@5 |  متوسط الزمن |
| ------------------------ | ----------: | -------: | -------: | -------: | -----: | -----: | -----------: |
| FAQ Paraphrase Retrieval |         131 |   0.9771 |   0.9924 |   1.0000 | 0.9866 | 0.9900 | 0.4166 ثانية |
| Legal Article Retrieval  |         104 |   0.4904 |   0.8462 |   0.9423 | 0.6716 | 0.7352 | 0.3948 ثانية |

---

### تفسير نتائج FAQ

حقق النظام أداءً عاليًا جدًا في أسئلة FAQ، حيث بلغت قيمة:

```text
Recall@1 = 0.9771
Recall@5 = 1.0000
MRR      = 0.9866
```

وهذا يعني أن وثيقة FAQ الصحيحة ظهرت في المرتبة الأولى في أغلب الأسئلة، وظهرت ضمن أفضل خمس نتائج في جميع أسئلة FAQ.

هذه النتيجة متوقعة لأن أسئلة FAQ تكون عادة قصيرة ومباشرة وقريبة دلاليًا من النصوص المفهرسة داخل قاعدة المعرفة. وحتى بعد إعادة صياغة أسئلة FAQ، تظل مهمة استرجاع FAQ أسهل من استرجاع المادة القانونية.

---

### تفسير نتائج الأسئلة القانونية

كانت مهمة استرجاع المواد القانونية أصعب، حيث حقق النظام:

```text
Recall@1 = 0.4904
Recall@3 = 0.8462
Recall@5 = 0.9423
MRR      = 0.6716
```

وهذا يعني أن المادة القانونية الصحيحة ظهرت في المرتبة الأولى في حوالي **49.04%** من الأسئلة القانونية. لكنها ظهرت ضمن أفضل ثلاث نتائج في حوالي **84.62%** من الأسئلة، وضمن أفضل خمس نتائج في حوالي **94.23%** من الأسئلة.

هذه نتيجة مهمة في نظام RAG، لأن نموذج اللغة لا يعتمد عادة على أول نتيجة فقط، بل يمكنه استخدام أكثر من سياق مسترجع لتوليد الإجابة النهائية. لذلك فإن ارتفاع `Recall@5` يدل على أن المادة القانونية الصحيحة غالبًا تكون متاحة ضمن السياقات التي سيتم تمريرها إلى نموذج اللغة.

---

### لماذا الأسئلة القانونية أصعب؟

استرجاع المواد القانونية أصعب من FAQ لعدة أسباب:

1. سؤال المستخدم القانوني قد لا يستخدم نفس ألفاظ المادة القانونية.
2. المادة القانونية قد تكون طويلة وتغطي أكثر من حالة.
3. بعض المواد القانونية متقاربة في الألفاظ، مثل الإنهاء، الاستقالة، الغياب، الفصل، الأجر، والإجازات.
4. الربط بين السؤال والمادة الصحيحة يحتاج فهمًا دلاليًا وليس مجرد تطابق كلمات.

لذلك من الطبيعي أن تكون نتائج FAQ أعلى من نتائج Legal Article Retrieval.

---

### الأهمية الأكاديمية للنتيجة

يوضح هذا التحليل أن النظام قوي جدًا في استرجاع FAQ، بينما يقدم أداءً أكثر واقعية في استرجاع المواد القانونية. وهذا الفرق مهم أكاديميًا لأنه يثبت أن تقييم المواد القانونية هو التحدي الحقيقي في المشروع.

كما أن توسيع الأسئلة القانونية من 24 إلى 104 سؤال جعل التقييم أكثر عدالة وتمثيلًا لأسئلة المستخدمين الحقيقيين.

---

### الخلاصة

يوضح هذا التحليل أن أفضل إعداد للاسترجاع يعمل بكفاءة عالية، لكنه يختلف في الأداء حسب نوع السؤال. فقد حقق النظام نتائج شبه كاملة في FAQ، بينما كانت الأسئلة القانونية أكثر تحديًا.

تعد قيمة `Recall@5 = 0.9423` في الأسئلة القانونية نتيجة مهمة، لأنها تعني أن المادة القانونية الصحيحة تظهر ضمن أفضل خمس نتائج في معظم الحالات. وهذا يجعل طبقة الاسترجاع مناسبة للاستخدام في المرحلة التالية من نظام RAG، حيث سيتم تمرير السياقات المسترجعة إلى نموذج اللغة الكبير لتوليد الإجابة النهائية.

ومع ذلك، فإن التحسين المستقبلي يجب أن يركز على رفع `Recall@1` و `MRR` في الأسئلة القانونية، بحيث تظهر المادة الصحيحة في المرتبة الأولى بشكل أكبر.


In [33]:
# =========================================================
# Diagnostic - Are gold document IDs available in indexed corpus?
# =========================================================

indexed_parent_ids = set()

for corpus_name, corpus_df in CORPORA.items():
    if "parent_document_id" in corpus_df.columns:
        indexed_parent_ids |= set(corpus_df["parent_document_id"].astype(str))

eval_gold_ids = set()

for _, row in df_eval_run.iterrows():
    ids = parse_gold_document_ids(row)
    eval_gold_ids |= set([str(x) for x in ids])

available_gold_ids = eval_gold_ids & indexed_parent_ids
missing_gold_ids = eval_gold_ids - indexed_parent_ids

print("Total gold document ids:", len(eval_gold_ids))
print("Gold ids available in indexed corpus:", len(available_gold_ids))
print("Gold ids missing from indexed corpus:", len(missing_gold_ids))

print("Availability ratio:", round(len(available_gold_ids) / max(len(eval_gold_ids), 1), 4))

Total gold document ids: 177
Gold ids available in indexed corpus: 177
Gold ids missing from indexed corpus: 0
Availability ratio: 1.0


## فحص توفر المراجع الذهبية داخل قاعدة الفهرسة

تهدف هذه الخلية إلى التحقق من أن جميع المراجع الذهبية المستخدمة في مجموعة التقييم موجودة فعليًا داخل قاعدة البيانات المفهرسة. ويعد هذا الفحص خطوة مهمة قبل اعتماد نتائج التقييم، لأن مقاييس الاسترجاع مثل `Recall@1` و `Recall@5` و `MRR` و `nDCG@5` لا تكون عادلة أو صحيحة إذا كانت بعض المراجع الصحيحة غير موجودة أصلًا داخل قاعدة الفهرسة.

في هذه الخطوة تم استخراج جميع معرفات الوثائق المفهرسة من الـ `CORPORA`، ثم تمت مقارنتها مع جميع الـ Gold Document IDs الموجودة في مجموعة التقييم النهائية `df_eval_run`.

أظهرت نتيجة الفحص ما يلي:

| المؤشر                                       | القيمة |
| -------------------------------------------- | -----: |
| إجمالي عدد المراجع الذهبية الفريدة           |    177 |
| عدد المراجع الذهبية الموجودة داخل الفهرس     |    177 |
| عدد المراجع الذهبية غير الموجودة داخل الفهرس |      0 |
| نسبة توفر المراجع الذهبية                    |    1.0 |

تشير هذه النتيجة إلى أن جميع المراجع الذهبية التي سيتم استخدامها في التقييم موجودة داخل قاعدة الفهرسة، أي أن النظام لديه فرصة عادلة لاسترجاع الوثيقة الصحيحة لكل سؤال.

من المهم ملاحظة أن عدد المراجع الذهبية الفريدة هو **177**، بينما عدد أسئلة التقييم الكلي هو **235**. وهذا طبيعي، لأن أكثر من سؤال قد يشير إلى نفس المادة القانونية أو نفس وثيقة FAQ. لذلك فإن هذا الرقم لا يمثل عدد الأسئلة، بل يمثل عدد الوثائق أو المراجع الفريدة المطلوبة في التقييم.

بما أن عدد المراجع المفقودة يساوي **0** ونسبة التوفر تساوي **1.0**، فهذا يؤكد أن إعداد التقييم صحيح، وأن نتائج الاسترجاع لن تتأثر بوجود مراجع ذهبية غير مفهرسة.


In [34]:
# =========================================================
# Stage 15 - Evaluation by Question Type
# =========================================================

by_type_summary = summarize_eval_rows(
    retrieval_detail,
    ["eval_type", "chunking_strategy", "embedding_model", "retriever", "alpha"]
)

by_type_summary = by_type_summary.sort_values(
    ["eval_type", "mrr", "recall_at_5"],
    ascending=[True, False, False]
).reset_index(drop=True)

by_type_summary.to_csv(STAGE03_DIR / "retrieval_summary_by_eval_type.csv", index=False, encoding="utf-8-sig")
by_type_summary.to_excel(STAGE03_DIR / "retrieval_summary_by_eval_type.xlsx", index=False)

display(by_type_summary)

,eval_type,chunking_strategy,embedding_model,retriever,alpha,questions,recall_at_1,recall_at_3,recall_at_5,mrr,ndcg_at_5,avg_latency_sec
0,faq_paraphrase_retrieval,fixed_size,bge_m3,hybrid,0.65,131,0.9924,1.0000,1.0000,0.9962,0.9972,0.0240
1,faq_paraphrase_retrieval,fixed_size,bge_m3,hybrid,0.80,131,0.9924,1.0000,1.0000,0.9962,0.9972,0.0268
2,faq_paraphrase_retrieval,fixed_size,bge_m3,hybrid,0.90,131,0.9924,1.0000,1.0000,0.9962,0.9972,0.0235
3,faq_paraphrase_retrieval,structural,bge_m3,hybrid,0.65,131,0.9924,1.0000,1.0000,0.9962,0.9972,0.0237
4,faq_paraphrase_retrieval,structural,bge_m3,hybrid,0.80,131,0.9924,1.0000,1.0000,0.9962,0.9972,0.0259
5,faq_paraphrase_retrieval,structural,bge_m3,hybrid,0.90,131,0.9924,1.0000,1.0000,0.9962,0.9972,0.0242
6,faq_paraphrase_retrieval,fixed_size,e5_large,hybrid,0.80,131,0.9847,1.0000,1.0000,0.9924,0.9944,0.0248
7,faq_paraphrase_retrieval,fixed_size,e5_large,hybrid,0.90,131,0.9847,1.0000,1.0000,0.9924,0.9944,0.0268
8,faq_paraphrase_retrieval,structural,e5_large,hybrid,0.80,131,0.9847,1.0000,1.0000,0.9924,0.9944,0.0228
9,faq_paraphrase_retrieval,structural,e5_large,hybrid,0.90,131,0.9847,1.0000,1.0000,0.9924,0.9944,0.0228


<div dir="rtl" style="text-align:right; font-family:Tahoma, Arial; line-height:2; font-size:16px;">

<h2>تحليل مقارن للأداء حسب نوع التقييم</h2>

<p>
يعرض هذا الجدول أداء إعدادات الاسترجاع المختلفة بعد فصل النتائج حسب نوع التقييم. 
تم تنفيذ هذا التحليل بهدف عدم الاعتماد على المتوسط العام فقط، لأن المتوسط العام قد يخفي الفروقات الحقيقية بين أداء النظام في أسئلة 
<span dir="ltr">FAQ</span> وأدائه في الأسئلة القانونية.
</p>

<p>
تنقسم مجموعة التقييم النهائية إلى نوعين رئيسيين:
</p>

<table style="width:100%; border-collapse:collapse; text-align:right; direction:rtl;">
<thead>
<tr style="background-color:#1f4e79; color:white;">
<th style="padding:8px; border:1px solid #ddd;">نوع التقييم</th>
<th style="padding:8px; border:1px solid #ddd;">الوصف</th>
<th style="padding:8px; border:1px solid #ddd;">عدد الأسئلة</th>
</tr>
</thead>
<tbody>
<tr>
<td style="padding:8px; border:1px solid #ddd;"><span dir="ltr">faq_paraphrase_retrieval</span></td>
<td style="padding:8px; border:1px solid #ddd;">أسئلة FAQ تمت إعادة صياغتها لتقليل المطابقة النصية المباشرة.</td>
<td style="padding:8px; border:1px solid #ddd;">131</td>
</tr>
<tr>
<td style="padding:8px; border:1px solid #ddd;"><span dir="ltr">legal_article_retrieval</span></td>
<td style="padding:8px; border:1px solid #ddd;">أسئلة قانونية طبيعية يجب ربطها بالمادة القانونية الصحيحة.</td>
<td style="padding:8px; border:1px solid #ddd;">104</td>
</tr>
</tbody>
</table>

<hr>

<h3>أولًا: أداء أسئلة FAQ</h3>

<p>
أظهرت نتائج أسئلة <span dir="ltr">FAQ</span> أداءً مرتفعًا جدًا في معظم الإعدادات. 
أفضل أداء في هذا النوع ظهر مع إعدادات <span dir="ltr">hybrid</span> باستخدام نموذج 
<span dir="ltr">bge_m3</span>، حيث حققت النتائج التالية تقريبًا:
</p>

<table style="width:60%; border-collapse:collapse; text-align:right; direction:rtl;">
<thead>
<tr style="background-color:#1f4e79; color:white;">
<th style="padding:8px; border:1px solid #ddd;">المقياس</th>
<th style="padding:8px; border:1px solid #ddd;">النتيجة</th>
</tr>
</thead>
<tbody>
<tr><td style="padding:8px; border:1px solid #ddd;"><span dir="ltr">Recall@1</span></td><td style="padding:8px; border:1px solid #ddd;">0.9924</td></tr>
<tr><td style="padding:8px; border:1px solid #ddd;"><span dir="ltr">Recall@3</span></td><td style="padding:8px; border:1px solid #ddd;">1.0000</td></tr>
<tr><td style="padding:8px; border:1px solid #ddd;"><span dir="ltr">Recall@5</span></td><td style="padding:8px; border:1px solid #ddd;">1.0000</td></tr>
<tr><td style="padding:8px; border:1px solid #ddd;"><span dir="ltr">MRR</span></td><td style="padding:8px; border:1px solid #ddd;">0.9962</td></tr>
<tr><td style="padding:8px; border:1px solid #ddd;"><span dir="ltr">nDCG@5</span></td><td style="padding:8px; border:1px solid #ddd;">0.9972</td></tr>
</tbody>
</table>

<p>
تشير هذه النتائج إلى أن النظام استطاع استرجاع وثيقة FAQ الصحيحة في المرتبة الأولى في حوالي 
<b>99.24%</b> من الأسئلة، كما ظهرت الوثيقة الصحيحة ضمن أفضل خمس نتائج في 
<b>100%</b> من الحالات.
</p>

<p>
هذا الأداء العالي متوقع؛ لأن أسئلة FAQ تكون غالبًا قصيرة ومباشرة وقريبة دلاليًا من النصوص المفهرسة، حتى بعد إعادة صياغتها لتقليل المطابقة النصية المباشرة.
</p>

<hr>

<h3>ثانيًا: أداء الأسئلة القانونية</h3>

<p>
كانت الأسئلة القانونية أكثر تحديًا من أسئلة FAQ. أفضل إعداد في 
<span dir="ltr">legal_article_retrieval</span> كان:
</p>

<div dir="ltr" style="text-align:left; background:#f5f7fa; border:1px solid #d9e2ec; padding:12px; border-radius:8px; font-family:Consolas, monospace;">
structural + bge_m3 + hybrid_reranker + alpha 0.8
</div>

<p>
وقد حقق هذا الإعداد النتائج التالية:
</p>

<table style="width:70%; border-collapse:collapse; text-align:right; direction:rtl;">
<thead>
<tr style="background-color:#1f4e79; color:white;">
<th style="padding:8px; border:1px solid #ddd;">المقياس</th>
<th style="padding:8px; border:1px solid #ddd;">النتيجة</th>
</tr>
</thead>
<tbody>
<tr><td style="padding:8px; border:1px solid #ddd;">عدد الأسئلة</td><td style="padding:8px; border:1px solid #ddd;">104</td></tr>
<tr><td style="padding:8px; border:1px solid #ddd;"><span dir="ltr">Recall@1</span></td><td style="padding:8px; border:1px solid #ddd;">0.4904</td></tr>
<tr><td style="padding:8px; border:1px solid #ddd;"><span dir="ltr">Recall@3</span></td><td style="padding:8px; border:1px solid #ddd;">0.8462</td></tr>
<tr><td style="padding:8px; border:1px solid #ddd;"><span dir="ltr">Recall@5</span></td><td style="padding:8px; border:1px solid #ddd;">0.9423</td></tr>
<tr><td style="padding:8px; border:1px solid #ddd;"><span dir="ltr">MRR</span></td><td style="padding:8px; border:1px solid #ddd;">0.6716</td></tr>
<tr><td style="padding:8px; border:1px solid #ddd;"><span dir="ltr">nDCG@5</span></td><td style="padding:8px; border:1px solid #ddd;">0.7352</td></tr>
<tr><td style="padding:8px; border:1px solid #ddd;">متوسط زمن الاسترجاع</td><td style="padding:8px; border:1px solid #ddd;">0.3948 ثانية</td></tr>
</tbody>
</table>

<p>
تعني هذه النتائج أن المادة القانونية الصحيحة ظهرت في المرتبة الأولى في حوالي 
<b>49.04%</b> من الأسئلة القانونية. كما ظهرت ضمن أفضل ثلاث نتائج في حوالي 
<b>84.62%</b> من الأسئلة، وضمن أفضل خمس نتائج في حوالي 
<b>94.23%</b> من الأسئلة.
</p>

<p>
وتعد قيمة <span dir="ltr">Recall@5 = 0.9423</span> مهمة جدًا في سياق أنظمة 
<span dir="ltr">RAG</span>، لأن نموذج اللغة الكبير لا يعتمد غالبًا على أول نتيجة فقط، بل يمكنه استخدام أكثر من سياق مسترجع عند توليد الإجابة النهائية. لذلك فإن ظهور المادة القانونية الصحيحة ضمن أفضل خمس نتائج يدل على أن طبقة الاسترجاع قادرة على توفير المرجع القانوني المناسب في أغلب الحالات.
</p>

<hr>

<h3>مقارنة FAQ مع الأسئلة القانونية</h3>

<p>
يوضح الجدول أن أداء النظام في أسئلة FAQ أعلى بكثير من أدائه في الأسئلة القانونية، وهذا أمر طبيعي من الناحية البحثية. 
في FAQ، يكون السؤال عادة قريبًا من سجل مفهرس يحتوي سؤالًا أو إجابة مشابهة. 
أما في الأسئلة القانونية، فيحتاج النظام إلى ربط سؤال مستخدم طبيعي بمادة قانونية قد تكون طويلة أو تحتوي على أكثر من حالة أو شرط.
</p>

<p>
لذلك لا ينبغي تفسير انخفاض <span dir="ltr">Recall@1</span> في الأسئلة القانونية على أنه فشل للنظام، بل هو مؤشر على أن مهمة الاسترجاع القانوني أكثر صعوبة وتتطلب فهمًا دلاليًا أعمق.
</p>

<hr>

<h3>أثر طبقة إعادة الترتيب Reranking</h3>

<p>
تظهر النتائج أن استخدام <span dir="ltr">hybrid_reranker</span> كان فعالًا جدًا في تحسين أداء الأسئلة القانونية. 
فعند مقارنة أفضل إعداد قانوني باستخدام Reranker مع أفضل إعدادات <span dir="ltr">hybrid</span> بدون Reranker، يظهر أن Reranking رفع جودة الترتيب بشكل واضح.
</p>

<table style="width:70%; border-collapse:collapse; text-align:right; direction:rtl;">
<thead>
<tr style="background-color:#1f4e79; color:white;">
<th style="padding:8px; border:1px solid #ddd;">الإعداد</th>
<th style="padding:8px; border:1px solid #ddd;"><span dir="ltr">Recall@5</span></th>
<th style="padding:8px; border:1px solid #ddd;"><span dir="ltr">MRR</span></th>
</tr>
</thead>
<tbody>
<tr>
<td style="padding:8px; border:1px solid #ddd;"><span dir="ltr">hybrid_reranker</span></td>
<td style="padding:8px; border:1px solid #ddd;">0.9423</td>
<td style="padding:8px; border:1px solid #ddd;">0.6716</td>
</tr>
<tr>
<td style="padding:8px; border:1px solid #ddd;"><span dir="ltr">hybrid</span> بدون Reranker</td>
<td style="padding:8px; border:1px solid #ddd;">أقل بوضوح</td>
<td style="padding:8px; border:1px solid #ddd;">أقل بوضوح</td>
</tr>
</tbody>
</table>

<p>
وهذا يؤكد أن طبقة إعادة الترتيب تساعد في رفع المادة القانونية الصحيحة إلى مراتب أعلى ضمن النتائج المسترجعة.
</p>

<hr>

<h3>الخلاصة</h3>

<p>
يوضح هذا التحليل أن أداء النظام يختلف حسب نوع السؤال. فقد حققت أسئلة FAQ نتائج شبه كاملة، بينما كانت الأسئلة القانونية أكثر تحديًا وأعطت نتائج أكثر واقعية.
</p>

<p>
أفضل إعداد عام ومناسب للانتقال إلى المرحلة التالية هو:
</p>

<div dir="ltr" style="text-align:left; background:#f5f7fa; border:1px solid #d9e2ec; padding:12px; border-radius:8px; font-family:Consolas, monospace;">
structural + bge_m3 + hybrid_reranker + alpha 0.8
</div>

<p>
ورغم أن <span dir="ltr">Recall@1</span> في الأسئلة القانونية ما زال قابلًا للتحسين، إلا أن قيمة 
<span dir="ltr">Recall@5 = 0.9423</span> تعد نتيجة قوية في نظام RAG، لأنها تعني أن المادة القانونية الصحيحة تكون متاحة ضمن السياقات المسترجعة في معظم الحالات.
</p>

<p>
بناءً على ذلك، يمكن اعتماد هذا الإعداد كطبقة الاسترجاع الأساسية قبل ربط النظام بطبقة توليد الإجابة باستخدام نموذج اللغة الكبير.
</p>

</div>

In [35]:
# =========================================================
# Stage 16 - Error Analysis
# =========================================================

# Analyze failures for the best configuration
best_mask = (
    (retrieval_detail["chunking_strategy"] == best_config["chunking_strategy"]) &
    (retrieval_detail["embedding_model"] == best_config["embedding_model"]) &
    (retrieval_detail["retriever"] == best_config["retriever"])
)

if pd.isna(best_config.get("alpha")):
    best_mask = best_mask & retrieval_detail["alpha"].isna()
else:
    best_mask = best_mask & (retrieval_detail["alpha"].astype(float).round(4) == float(best_config["alpha"]))

best_detail = retrieval_detail[best_mask].copy()
failures_at_5 = best_detail[best_detail["hit_at_5"] == 0].copy()
failures_at_1 = best_detail[best_detail["hit_at_1"] == 0].copy()

error_summary = pd.DataFrame([{
    "best_chunking_strategy": best_config["chunking_strategy"],
    "best_embedding_model": best_config["embedding_model"],
    "best_retriever": best_config["retriever"],
    "alpha": best_config.get("alpha", ""),
    "total_questions": len(best_detail),
    "failures_at_1": len(failures_at_1),
    "failures_at_5": len(failures_at_5),
    "failure_rate_at_5": round(len(failures_at_5) / max(len(best_detail), 1), 4),
}])

display(error_summary)

audit_cols = [
    "eval_id", "eval_type", "question", "gold_article_numbers", "gold_document_unit_ids",
    "top1_source_type", "top1_article_number", "top1_article_title", "top1_score",
    "top5_article_numbers", "top5_relevance"
]

display(failures_at_5[audit_cols].head(30))

error_summary.to_csv(STAGE03_DIR / "best_config_error_summary.csv", index=False, encoding="utf-8-sig")
failures_at_5.to_csv(STAGE03_DIR / "best_config_failures_at_5.csv", index=False, encoding="utf-8-sig")
failures_at_1.to_csv(STAGE03_DIR / "best_config_failures_at_1.csv", index=False, encoding="utf-8-sig")

,best_chunking_strategy,best_embedding_model,best_retriever,alpha,total_questions,failures_at_1,failures_at_5,failure_rate_at_5
0,structural,bge_m3,hybrid_reranker,0.8,235,56,6,0.0255


,eval_id,eval_type,question,gold_article_numbers,gold_document_unit_ids,top1_source_type,top1_article_number,top1_article_title,top1_score,top5_article_numbers,top5_relevance
2534,LEGAL_EXTRA_030,legal_article_retrieval,هل يجوز تشغيل العامل أكثر من الحد النظامي لساعات العمل؟,106,e8b5ef7c86bc97c469eb6af936fc5aeafda9ce44bf0eed4b9c3c3a7130387fc6,faq,nan,nan,0.991117,nan|98.0|nan|nan|101.0,00000
2553,LEGAL_EXTRA_049,legal_article_retrieval,كم يوم غياب يسبب فصل الموظف؟,80,1b6eab8234c7882e95dd2d1aeb5b2f561e3c1fff20c3f73d3c76785fe09b13a8,faq,nan,nan,0.483819,nan|78.0|155.0|nan|nan,00000
2564,LEGAL_EXTRA_060,legal_article_retrieval,هل تأخير الراتب يعتبر مخالفة؟,90,7b1238ba5214ccbbc7005da1b248f0f5a9d2ae6c4477226e0da96531c85da50f,faq,nan,nan,0.226982,nan|94.0|nan|nan|nan,00000
2567,LEGAL_EXTRA_063,legal_article_retrieval,هل يحق للشركة تخصم من راتبي بسبب تلف جهاز؟,93,f32b99c0ffecdc340cd88f76cee2614ce380ae91b27c99ac6cf7abb903715faf,legal_article,91.0,المادة الحادية والتسعون:,0.085194,91.0|nan|nan|nan|137.0,00000
2568,LEGAL_EXTRA_064,legal_article_retrieval,هل يحق للعامل الاعتراض على الخصم من راتبه؟,93,2afdc1c866c773156b5b12edc0b8ed1696aa7265ca814192997a62f3e9bc86b6,faq,nan,nan,0.377022,nan|91.0|nan|72.0|94.0,00000
2573,LEGAL_EXTRA_069,legal_article_retrieval,هل العقد الإلكتروني يعتبر عقد عمل نظامي؟,51,b9ce6acac4d58bf08d0059d367c9cac542b97ebf4d3e98e0ae1e4caf3b2b3d16,faq,nan,nan,0.378755,nan|nan|2.0|nan|nan,00000


In [38]:
# =========================================================
# Stage 14.6 - Source-Aware Diagnostic using Article Numbers
# تحليل تشخيصي موجه حسب نوع المصدر باستخدام أرقام المواد
# =========================================================

import re
import json
import numpy as np
import pandas as pd
from IPython.display import display, HTML

# ---------------------------------------------------------
# 0) Explanation
# ---------------------------------------------------------

display(HTML("""
<div dir="rtl" style="
    text-align:right;
    font-family:Tahoma, Arial;
    line-height:2;
    font-size:16px;
    background:#f8fbfd;
    border-right:5px solid #1f4e79;
    padding:14px 18px;
    border-radius:8px;
    margin-bottom:14px;">
<h3>تحليل تشخيصي موجه حسب نوع المصدر</h3>
<p>
هذه الخلية لا تستبدل تقييم <span dir="ltr">Full Corpus Evaluation</span> الرسمي،
بل تضيف تحليلًا تشخيصيًا لفهم أثر خلط مصادر المعرفة بين
<span dir="ltr">FAQ</span> والمواد القانونية.
</p>
<p>
بما أن جدول <span dir="ltr">retrieval_detail</span> لا يحتوي على أعمدة
<span dir="ltr">top5_document_unit_ids</span> أو <span dir="ltr">top5_source_types</span>،
سيتم استخدام <span dir="ltr">top5_article_numbers</span> كتحليل تشخيصي للأسئلة القانونية.
</p>
</div>
"""))

# ---------------------------------------------------------
# 1) Basic checks
# ---------------------------------------------------------

assert "retrieval_detail" in globals(), "retrieval_detail not found. Run Stage 13 first."
assert "retrieval_summary" in globals(), "retrieval_summary not found. Run Stage 14 first."
assert "STAGE03_DIR" in globals(), "STAGE03_DIR not found."

print("retrieval_detail shape:", retrieval_detail.shape)
print("retrieval_summary shape:", retrieval_summary.shape)


# ---------------------------------------------------------
# 2) Helper functions
# ---------------------------------------------------------

def first_existing_col(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None


def parse_list_cell(value):
    """
    Parse values that may be stored as:
    - pipe-separated strings
    - comma-separated strings
    - Arabic comma-separated strings
    - lists
    - NaN
    """
    if value is None:
        return []

    if isinstance(value, (list, tuple, set)):
        raw_items = list(value)
    else:
        text = str(value).strip()

        if text == "" or text.lower() in ["nan", "none", "null", "na", "n/a"]:
            return []

        raw_items = re.split(r"\s*\|\s*|\s*,\s*|\s*،\s*|\s*;\s*|\n+", text)

    items = []

    for item in raw_items:
        s = str(item).strip()
        if s == "" or s.lower() in ["nan", "none", "null", "na", "n/a"]:
            continue
        items.append(s)

    return items


def normalize_article_number(x):
    """
    Normalize article numbers such as:
    91.0 -> 91
    'المادة 91' -> 91
    '91' -> 91
    """
    if x is None:
        return ""

    s = str(x).strip()

    if s == "" or s.lower() in ["nan", "none", "null", "na", "n/a"]:
        return ""

    m = re.search(r"\d+", s)

    if m:
        return str(int(m.group(0)))

    return s


def parse_article_numbers(value):
    items = parse_list_cell(value)
    normalized = []

    for item in items:
        n = normalize_article_number(item)
        if n:
            normalized.append(n)

    # Deduplicate while preserving order
    seen = set()
    clean = []

    for n in normalized:
        if n not in seen:
            seen.add(n)
            clean.append(n)

    return clean


def compute_rank_metrics(retrieved_items, gold_items, k=5):
    """
    Compute Hit@1, Hit@3, Hit@5, RR, and nDCG@5.
    """
    retrieved = [str(x).strip() for x in retrieved_items if str(x).strip()]
    gold_set = set([str(x).strip() for x in gold_items if str(x).strip()])

    if len(gold_set) == 0:
        return {
            "hit_at_1": np.nan,
            "hit_at_3": np.nan,
            "hit_at_5": np.nan,
            "rr": np.nan,
            "ndcg_at_5": np.nan,
            "first_match_rank": np.nan
        }

    match_positions = [
        idx + 1
        for idx, item in enumerate(retrieved[:k])
        if item in gold_set
    ]

    if len(match_positions) == 0:
        return {
            "hit_at_1": 0,
            "hit_at_3": 0,
            "hit_at_5": 0,
            "rr": 0.0,
            "ndcg_at_5": 0.0,
            "first_match_rank": np.nan
        }

    first_rank = min(match_positions)

    return {
        "hit_at_1": int(first_rank <= 1),
        "hit_at_3": int(first_rank <= 3),
        "hit_at_5": int(first_rank <= 5),
        "rr": round(1.0 / first_rank, 6),
        "ndcg_at_5": round(1.0 / np.log2(first_rank + 1), 6),
        "first_match_rank": first_rank
    }


def display_status_card(title, message, status="info"):
    colors = {
        "success": ("#d1e7dd", "#0f5132", "#badbcc"),
        "warning": ("#fff3cd", "#664d03", "#ffecb5"),
        "danger":  ("#f8d7da", "#842029", "#f5c2c7"),
        "info":    ("#cff4fc", "#055160", "#b6effb"),
    }

    bg, color, border = colors.get(status, colors["info"])

    display(HTML(f"""
    <div dir="rtl" style="
        text-align:right;
        font-family:Tahoma, Arial;
        line-height:2;
        font-size:15px;
        background:{bg};
        color:{color};
        border:1px solid {border};
        padding:14px 16px;
        border-radius:8px;
        margin:12px 0;">
        <b>{title}</b><br>
        {message}
    </div>
    """))


def display_professional_table(df, title="Table", max_rows=15, preferred_cols=None):
    """
    Display a professional wide scrollable table.
    """
    if df is None or len(df) == 0:
        display_status_card(title, "لا توجد بيانات لعرضها.", status="warning")
        return

    if preferred_cols:
        existing_cols = [c for c in preferred_cols if c in df.columns]
        if existing_cols:
            view_df = df[existing_cols].head(max_rows).copy()
        else:
            view_df = df.head(max_rows).copy()
    else:
        view_df = df.head(max_rows).copy()

    # Shorten very long text only for notebook display
    for col in view_df.columns:
        if view_df[col].dtype == "object":
            view_df[col] = view_df[col].astype(str).apply(
                lambda x: x[:240] + "..." if len(x) > 240 else x
            )

    html_table = view_df.to_html(index=False, escape=True, border=0)

    display(HTML(f"""
    <div dir="rtl" style="
        text-align:right;
        font-family:Tahoma, Arial;
        line-height:2;
        margin-top:18px;
        margin-bottom:8px;">
        <h3 style="color:#17324d; margin-bottom:4px;">{title}</h3>
        <p style="font-size:14px; color:#52616b;">
        يتم عرض عينة مختصرة ومنظمة داخل النوتبوك، بينما تم حفظ الجدول الكامل في ملفات خارجية.
        </p>
    </div>

    <style>
        .professional-wide-table-wrapper {{
            width: 100%;
            overflow-x: auto;
            overflow-y: hidden;
            border: 1px solid #d9e2ec;
            border-radius: 10px;
            background: #ffffff;
            margin: 10px 0 22px 0;
        }}

        .professional-wide-table-wrapper table {{
            border-collapse: collapse;
            width: max-content;
            min-width: 1600px;
            font-family: Tahoma, Arial, sans-serif;
            font-size: 13px;
        }}

        .professional-wide-table-wrapper th {{
            background-color: #1f4e79;
            color: white;
            text-align: center;
            padding: 10px 12px;
            border: 1px solid #d9e2ec;
            white-space: nowrap;
            font-weight: 700;
        }}

        .professional-wide-table-wrapper td {{
            padding: 9px 12px;
            border: 1px solid #e5eaf0;
            vertical-align: top;
            white-space: normal;
            word-wrap: break-word;
            max-width: 420px;
            text-align: left;
            direction: ltr;
        }}

        .professional-wide-table-wrapper tr:nth-child(even) {{
            background-color: #f8fbfd;
        }}

        .professional-wide-table-wrapper tr:hover {{
            background-color: #eef6ff;
        }}

        .professional-wide-table-wrapper td:nth-child(4) {{
            direction: rtl;
            text-align: right;
            min-width: 340px;
            max-width: 520px;
        }}
    </style>

    <div class="professional-wide-table-wrapper">
        {html_table}
    </div>
    """))


# ---------------------------------------------------------
# 3) Select best configuration from Stage 14
# ---------------------------------------------------------

best_config = retrieval_summary.iloc[0].to_dict()

display(HTML("""
<div dir="rtl" style="text-align:right; font-family:Tahoma, Arial; line-height:2; font-size:16px;">
<h3>أفضل إعداد مستخدم في التحليل التشخيصي</h3>
</div>
"""))

display(pd.DataFrame([best_config]))

best_mask = pd.Series(True, index=retrieval_detail.index)

for col in ["chunking_strategy", "embedding_model", "retriever"]:
    if col in retrieval_detail.columns and col in best_config:
        best_mask &= retrieval_detail[col].astype(str).eq(str(best_config[col]))

if "alpha" in retrieval_detail.columns and "alpha" in best_config:
    if pd.isna(best_config["alpha"]):
        best_mask &= retrieval_detail["alpha"].isna()
    else:
        best_mask &= retrieval_detail["alpha"].astype(float).round(4).eq(
            round(float(best_config["alpha"]), 4)
        )

best_detail = retrieval_detail[best_mask].copy()

print("Rows for best configuration:", len(best_detail))

assert len(best_detail) > 0, "No rows found for best configuration."


# ---------------------------------------------------------
# 4) Detect whether strict source-aware columns exist
# ---------------------------------------------------------

topk_id_col = first_existing_col(
    best_detail,
    [
        "top5_document_unit_ids",
        "top5_parent_document_ids",
        "top5_document_ids",
        "top5_ids",
        "retrieved_document_unit_ids",
        "retrieved_parent_document_ids",
        "retrieved_ids"
    ]
)

topk_source_col = first_existing_col(
    best_detail,
    [
        "top5_source_types",
        "top5_sources",
        "retrieved_source_types",
        "retrieved_sources"
    ]
)

topk_article_col = first_existing_col(
    best_detail,
    [
        "top5_article_numbers",
        "retrieved_article_numbers"
    ]
)

print("Detected top-k ID column:", topk_id_col)
print("Detected top-k source column:", topk_source_col)
print("Detected top-k article column:", topk_article_col)

if topk_id_col is None or topk_source_col is None:
    display_status_card(
        title="تنبيه منهجي",
        message=(
            "لم يتم العثور على أعمدة top-k document IDs أو source types. "
            "لذلك سيتم تنفيذ تحليل تشخيصي باستخدام أرقام المواد القانونية، "
            "وليس تقييم Source-Aware صارم بديل عن Full Corpus."
        ),
        status="warning"
    )


# ---------------------------------------------------------
# 5) Build diagnostic detail
# ---------------------------------------------------------

assert topk_article_col is not None, (
    "top5_article_numbers column not found. "
    "Please make sure Stage 13 saves top5_article_numbers."
)

diagnostic_rows = []

for _, row in best_detail.iterrows():
    eval_type = str(row.get("eval_type", "")).strip()

    gold_articles = parse_article_numbers(
        row.get("gold_article_numbers", row.get("gold_article_number", ""))
    )

    retrieved_articles = parse_article_numbers(
        row.get(topk_article_col, "")
    )

    # For FAQ, article-number matching is not meaningful.
    # Keep original full-corpus metrics and mark method as full_corpus_reference.
    if "faq" in eval_type.lower():
        diagnostic_method = "full_corpus_reference_for_faq"

        diag_hit_at_1 = row.get("hit_at_1", np.nan)
        diag_hit_at_3 = row.get("hit_at_3", np.nan)
        diag_hit_at_5 = row.get("hit_at_5", np.nan)
        diag_rr = row.get("rr", np.nan)
        diag_ndcg = row.get("ndcg_at_5", np.nan)
        diag_first_rank = np.nan

    else:
        diagnostic_method = "article_number_diagnostic_for_legal"

        metrics = compute_rank_metrics(
            retrieved_items=retrieved_articles,
            gold_items=gold_articles,
            k=5
        )

        diag_hit_at_1 = metrics["hit_at_1"]
        diag_hit_at_3 = metrics["hit_at_3"]
        diag_hit_at_5 = metrics["hit_at_5"]
        diag_rr = metrics["rr"]
        diag_ndcg = metrics["ndcg_at_5"]
        diag_first_rank = metrics["first_match_rank"]

    diagnostic_rows.append({
        "eval_id": row.get("eval_id", ""),
        "eval_type": eval_type,
        "diagnostic_method": diagnostic_method,
        "question": row.get("question", ""),
        "gold_article_numbers": row.get("gold_article_numbers", row.get("gold_article_number", "")),
        "parsed_gold_articles": "|".join(gold_articles),
        "original_top5_article_numbers": row.get(topk_article_col, ""),
        "parsed_top5_article_numbers": "|".join(retrieved_articles[:5]),
        "full_hit_at_1": row.get("hit_at_1", np.nan),
        "full_hit_at_3": row.get("hit_at_3", np.nan),
        "full_hit_at_5": row.get("hit_at_5", np.nan),
        "full_rr": row.get("rr", np.nan),
        "full_ndcg_at_5": row.get("ndcg_at_5", np.nan),
        "diagnostic_hit_at_1": diag_hit_at_1,
        "diagnostic_hit_at_3": diag_hit_at_3,
        "diagnostic_hit_at_5": diag_hit_at_5,
        "diagnostic_rr": diag_rr,
        "diagnostic_ndcg_at_5": diag_ndcg,
        "diagnostic_first_match_rank": diag_first_rank,
        "latency_sec": row.get("latency_sec", np.nan)
    })

source_aware_diagnostic_detail = pd.DataFrame(diagnostic_rows)


# ---------------------------------------------------------
# 6) Summary by eval_type
# ---------------------------------------------------------

source_aware_diagnostic_summary = (
    source_aware_diagnostic_detail
    .groupby("eval_type")
    .agg(
        questions=("eval_id", "count"),
        diagnostic_recall_at_1=("diagnostic_hit_at_1", "mean"),
        diagnostic_recall_at_3=("diagnostic_hit_at_3", "mean"),
        diagnostic_recall_at_5=("diagnostic_hit_at_5", "mean"),
        diagnostic_mrr=("diagnostic_rr", "mean"),
        diagnostic_ndcg_at_5=("diagnostic_ndcg_at_5", "mean"),
        avg_latency_sec=("latency_sec", "mean")
    )
    .reset_index()
)

source_aware_diagnostic_overall = pd.DataFrame([{
    "eval_type": "overall_diagnostic",
    "questions": len(source_aware_diagnostic_detail),
    "diagnostic_recall_at_1": source_aware_diagnostic_detail["diagnostic_hit_at_1"].mean(),
    "diagnostic_recall_at_3": source_aware_diagnostic_detail["diagnostic_hit_at_3"].mean(),
    "diagnostic_recall_at_5": source_aware_diagnostic_detail["diagnostic_hit_at_5"].mean(),
    "diagnostic_mrr": source_aware_diagnostic_detail["diagnostic_rr"].mean(),
    "diagnostic_ndcg_at_5": source_aware_diagnostic_detail["diagnostic_ndcg_at_5"].mean(),
    "avg_latency_sec": source_aware_diagnostic_detail["latency_sec"].mean()
}])

source_aware_diagnostic_summary = pd.concat(
    [source_aware_diagnostic_summary, source_aware_diagnostic_overall],
    ignore_index=True
)

for col in [
    "diagnostic_recall_at_1",
    "diagnostic_recall_at_3",
    "diagnostic_recall_at_5",
    "diagnostic_mrr",
    "diagnostic_ndcg_at_5",
    "avg_latency_sec"
]:
    source_aware_diagnostic_summary[col] = source_aware_diagnostic_summary[col].round(4)


# ---------------------------------------------------------
# 7) Full corpus comparison by eval_type
# ---------------------------------------------------------

full_corpus_by_type = (
    best_detail
    .groupby("eval_type")
    .agg(
        questions=("eval_id", "count"),
        full_recall_at_1=("hit_at_1", "mean"),
        full_recall_at_3=("hit_at_3", "mean"),
        full_recall_at_5=("hit_at_5", "mean"),
        full_mrr=("rr", "mean"),
        full_ndcg_at_5=("ndcg_at_5", "mean"),
        full_avg_latency_sec=("latency_sec", "mean")
    )
    .reset_index()
)

for col in [
    "full_recall_at_1",
    "full_recall_at_3",
    "full_recall_at_5",
    "full_mrr",
    "full_ndcg_at_5",
    "full_avg_latency_sec"
]:
    full_corpus_by_type[col] = full_corpus_by_type[col].round(4)

source_aware_diagnostic_comparison = source_aware_diagnostic_summary[
    source_aware_diagnostic_summary["eval_type"] != "overall_diagnostic"
].merge(
    full_corpus_by_type,
    on=["eval_type", "questions"],
    how="left"
)


# ---------------------------------------------------------
# 8) Save outputs
# ---------------------------------------------------------

detail_csv = STAGE03_DIR / "source_aware_diagnostic_detail_best_config.csv"
summary_csv = STAGE03_DIR / "source_aware_diagnostic_summary_by_type.csv"
comparison_csv = STAGE03_DIR / "source_aware_diagnostic_vs_full_corpus_comparison.csv"

detail_xlsx = STAGE03_DIR / "source_aware_diagnostic_detail_best_config.xlsx"
summary_xlsx = STAGE03_DIR / "source_aware_diagnostic_summary_by_type.xlsx"
comparison_xlsx = STAGE03_DIR / "source_aware_diagnostic_vs_full_corpus_comparison.xlsx"

source_aware_diagnostic_detail.to_csv(detail_csv, index=False, encoding="utf-8-sig")
source_aware_diagnostic_summary.to_csv(summary_csv, index=False, encoding="utf-8-sig")
source_aware_diagnostic_comparison.to_csv(comparison_csv, index=False, encoding="utf-8-sig")

with pd.ExcelWriter(detail_xlsx, engine="openpyxl") as writer:
    source_aware_diagnostic_detail.to_excel(writer, index=False, sheet_name="diagnostic_detail")

with pd.ExcelWriter(summary_xlsx, engine="openpyxl") as writer:
    source_aware_diagnostic_summary.to_excel(writer, index=False, sheet_name="summary_by_type")

with pd.ExcelWriter(comparison_xlsx, engine="openpyxl") as writer:
    source_aware_diagnostic_comparison.to_excel(writer, index=False, sheet_name="comparison")

with open(STAGE03_DIR / "source_aware_diagnostic_best_config_used.json", "w", encoding="utf-8") as f:
    json.dump(best_config, f, ensure_ascii=False, indent=2)


# ---------------------------------------------------------
# 9) Display outputs professionally
# ---------------------------------------------------------

display_status_card(
    title="تم تنفيذ التحليل التشخيصي بنجاح",
    message=(
        "تم إنشاء ملخص تشخيصي يوضح أداء أفضل إعداد عند استخدام أرقام المواد "
        "لفهم سلوك الأسئلة القانونية، مع الإبقاء على تقييم Full Corpus كالتقييم الرسمي."
    ),
    status="success"
)

display(HTML("""
<div dir="rtl" style="text-align:right; font-family:Tahoma, Arial; line-height:2; font-size:16px;">
<h3>ملخص التحليل التشخيصي حسب نوع السؤال</h3>
</div>
"""))
display(source_aware_diagnostic_summary)

display(HTML("""
<div dir="rtl" style="text-align:right; font-family:Tahoma, Arial; line-height:2; font-size:16px;">
<h3>مقارنة التحليل التشخيصي مع تقييم Full Corpus</h3>
</div>
"""))
display(source_aware_diagnostic_comparison)

preferred_sample_cols = [
    "eval_id",
    "eval_type",
    "diagnostic_method",
    "question",
    "gold_article_numbers",
    "parsed_top5_article_numbers",
    "full_hit_at_5",
    "diagnostic_hit_at_5",
    "full_rr",
    "diagnostic_rr",
    "diagnostic_first_match_rank",
    "latency_sec",
]

display_professional_table(
    source_aware_diagnostic_detail,
    title="عينة احترافية من تفاصيل التحليل التشخيصي",
    max_rows=15,
    preferred_cols=preferred_sample_cols
)

print("Saved files:")
print("Detail CSV:", detail_csv)
print("Summary CSV:", summary_csv)
print("Comparison CSV:", comparison_csv)
print("Detail Excel:", detail_xlsx)
print("Summary Excel:", summary_xlsx)
print("Comparison Excel:", comparison_xlsx)

retrieval_detail shape: (5170, 25)
retrieval_summary shape: (16, 12)


,rank,chunking_strategy,embedding_model,retriever,alpha,questions,recall_at_1,recall_at_3,recall_at_5,mrr,ndcg_at_5,avg_latency_sec
0,1,structural,bge_m3,hybrid_reranker,0.8,235,0.7617,0.9277,0.9745,0.8472,0.8773,0.407


Rows for best configuration: 235
Detected top-k ID column: top5_parent_document_ids
Detected top-k source column: None
Detected top-k article column: top5_article_numbers


,eval_type,questions,diagnostic_recall_at_1,diagnostic_recall_at_3,diagnostic_recall_at_5,diagnostic_mrr,diagnostic_ndcg_at_5,avg_latency_sec
0,faq_paraphrase_retrieval,131,0.9771,0.9924,1.0000,0.9866,0.9900,0.4166
1,legal_article_retrieval,104,0.6635,0.8365,0.8365,0.7452,0.7689,0.3948
2,overall_diagnostic,235,0.8383,0.9234,0.9277,0.8798,0.8922,0.4070


,eval_type,questions,diagnostic_recall_at_1,diagnostic_recall_at_3,diagnostic_recall_at_5,diagnostic_mrr,diagnostic_ndcg_at_5,avg_latency_sec,full_recall_at_1,full_recall_at_3,full_recall_at_5,full_mrr,full_ndcg_at_5,full_avg_latency_sec
0,faq_paraphrase_retrieval,131,0.9771,0.9924,1.0000,0.9866,0.9900,0.4166,0.9771,0.9924,1.0000,0.9866,0.9900,0.4166
1,legal_article_retrieval,104,0.6635,0.8365,0.8365,0.7452,0.7689,0.3948,0.4904,0.8462,0.9423,0.6716,0.7352,0.3948


eval_id,eval_type,diagnostic_method,question,gold_article_numbers,parsed_top5_article_numbers,full_hit_at_5,diagnostic_hit_at_5,full_rr,diagnostic_rr,diagnostic_first_match_rank,latency_sec
FAQ_INDEXED_RETRIEVAL_0001,faq_paraphrase_retrieval,full_corpus_reference_for_faq,أريد معرفة ما إذا كانت توفر البوابة المعلومات ومجموعة الخدمات الخاصة بالمرأة؟,,,1,1,1.00,1.00,NaN,0.380219
FAQ_INDEXED_RETRIEVAL_0002,faq_paraphrase_retrieval,full_corpus_reference_for_faq,أريد توضيح الحكم النظامي بخصوص: ماهي أوقات العمل الرسمية للفروع؟,,6|164|49|101,1,1,1.00,1.00,NaN,0.284749
FAQ_INDEXED_RETRIEVAL_0003,faq_paraphrase_retrieval,full_corpus_reference_for_faq,أريد توضيح الحكم النظامي بخصوص: شاهدتُ حالة عنف وقعت أمامي، ماذا أفعل؟,,2|230,1,1,1.00,1.00,NaN,0.690088
FAQ_INDEXED_RETRIEVAL_0004,faq_paraphrase_retrieval,full_corpus_reference_for_faq,أريد معرفة ما إذا كان يتم احتساب الفرد الذي يعمل لدى أكثر من منشأة في نفس الوقت ضمن نسبة التوطين؟,,,1,1,1.00,1.00,NaN,0.275060
FAQ_INDEXED_RETRIEVAL_0005,faq_paraphrase_retrieval,full_corpus_reference_for_faq,هل يستطيع إصدار تأشيرة لجنسية أو مهنة غير مدرجة في مساند؟,,34,1,1,1.00,1.00,NaN,0.452078
FAQ_INDEXED_RETRIEVAL_0006,faq_paraphrase_retrieval,full_corpus_reference_for_faq,أريد معرفة ما إذا كان يتم إيواء المرأة التي تعرضت للعنف؟,,,1,1,1.00,1.00,NaN,0.454952
FAQ_INDEXED_RETRIEVAL_0007,faq_paraphrase_retrieval,full_corpus_reference_for_faq,أريد توضيح الحكم النظامي بخصوص: ما مدى امكانية الجمع بين المكافأة المقطوعة للمستشارين غير المتفرغين ومكافأة العمل خارج وقت الدوام الرسمي وأيام العطل الرسمية والأعياد؟,,,1,1,1.00,1.00,NaN,0.302857
FAQ_INDEXED_RETRIEVAL_0008,faq_paraphrase_retrieval,full_corpus_reference_for_faq,أريد معرفة ما إذا كان الخدمة تتيح تحديث بيانات العامل؟,,,1,1,1.00,1.00,NaN,0.284237
FAQ_INDEXED_RETRIEVAL_0009,faq_paraphrase_retrieval,full_corpus_reference_for_faq,أريد توضيح الحكم النظامي بخصوص: ماهي خدمة قرار مكافأة الطلبة الجامعين لذوي الإعاقة؟,,,1,1,1.00,1.00,NaN,0.475348
FAQ_INDEXED_RETRIEVAL_0010,faq_paraphrase_retrieval,full_corpus_reference_for_faq,أريد توضيح الحكم النظامي بخصوص: العديد يطالبون بشرح البرنامج وشرح للدليل الإجرائي بشكل مبسط وواضح؟,,2|7,1,1,1.00,1.00,NaN,0.701527


Saved files:
Detail CSV: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\source_aware_diagnostic_detail_best_config.csv
Summary CSV: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\source_aware_diagnostic_summary_by_type.csv
Comparison CSV: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\source_aware_diagnostic_vs_full_corpus_comparison.csv
Detail Excel: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\source_aware_diagnostic_detail_best_config.xlsx
Summary Excel: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\source_aware_diagnostic_summary_by_type.xlsx
Comparison Excel: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\source_aware_diagnostic_vs_full_corpus_comparison.xlsx


<div dir="rtl" style="text-align:right; font-family:Tahoma, Arial; line-height:2; font-size:16px;">

<h2>تحليل تشخيصي موجه حسب نوع المصدر</h2>

<p>
تم في هذه الخلية تنفيذ تحليل تشخيصي إضافي بعد اختيار أفضل إعداد للاسترجاع. الهدف من هذا التحليل هو فهم أثر خلط مصادر المعرفة بين وثائق الأسئلة الشائعة <span dir="ltr">FAQ</span> والمواد القانونية <span dir="ltr">Legal Articles</span> على أداء طبقة الاسترجاع.
</p>

<p>
من المهم توضيح أن هذا التحليل لا يستبدل تقييم <span dir="ltr">Full Corpus Evaluation</span> الرسمي، بل يستخدم كتحليل مساعد لفهم سلوك النظام، خصوصًا في الحالات التي قد يسترجع فيها النظام وثائق <span dir="ltr">FAQ</span> قريبة دلاليًا بدل المادة القانونية المطلوبة.
</p>

<hr>

<h3>طبيعة التحليل</h3>

<p>
أظهرت الخلية أن جدول <span dir="ltr">retrieval_detail</span> لا يحتوي على أعمدة تفصيلية كافية مثل <span dir="ltr">top5_document_unit_ids</span> أو <span dir="ltr">top5_source_types</span>. لذلك تم تنفيذ هذا التحليل بالاعتماد على عمود <span dir="ltr">top5_article_numbers</span>.
</p>

<p>
بناءً على ذلك، فإن هذه النتائج تقيس مدى ظهور رقم المادة القانونية الصحيح ضمن أفضل النتائج، وليست تقييمًا صارمًا قائمًا على <span dir="ltr">document_unit_id</span> كما في التقييم الأساسي.
</p>

<div style="background:#fff3cd; border-right:5px solid #ffc107; padding:12px 16px; border-radius:8px; margin:12px 0;">
<b>ملاحظة منهجية:</b>
تُستخدم هذه النتائج كتفسير تشخيصي إضافي، بينما تبقى نتائج <span dir="ltr">Full Corpus Evaluation</span> هي النتائج الرسمية الأساسية لتقييم نظام الاسترجاع.
</div>

<hr>

<h3>ملخص النتائج التشخيصية</h3>

<table style="width:100%; border-collapse:collapse; text-align:right; direction:rtl;">
<thead>
<tr style="background-color:#1f4e79; color:white;">
<th style="padding:8px; border:1px solid #ddd;">نوع التقييم</th>
<th style="padding:8px; border:1px solid #ddd;">عدد الأسئلة</th>
<th style="padding:8px; border:1px solid #ddd;"><span dir="ltr">Recall@1</span></th>
<th style="padding:8px; border:1px solid #ddd;"><span dir="ltr">Recall@3</span></th>
<th style="padding:8px; border:1px solid #ddd;"><span dir="ltr">Recall@5</span></th>
<th style="padding:8px; border:1px solid #ddd;"><span dir="ltr">MRR</span></th>
<th style="padding:8px; border:1px solid #ddd;"><span dir="ltr">nDCG@5</span></th>
<th style="padding:8px; border:1px solid #ddd;">متوسط الزمن</th>
</tr>
</thead>

<tbody>
<tr>
<td style="padding:8px; border:1px solid #ddd;"><span dir="ltr">faq_paraphrase_retrieval</span></td>
<td style="padding:8px; border:1px solid #ddd;">131</td>
<td style="padding:8px; border:1px solid #ddd;">0.9771</td>
<td style="padding:8px; border:1px solid #ddd;">0.9924</td>
<td style="padding:8px; border:1px solid #ddd;">1.0000</td>
<td style="padding:8px; border:1px solid #ddd;">0.9866</td>
<td style="padding:8px; border:1px solid #ddd;">0.9900</td>
<td style="padding:8px; border:1px solid #ddd;">0.4166 ثانية</td>
</tr>

<tr>
<td style="padding:8px; border:1px solid #ddd;"><span dir="ltr">legal_article_retrieval</span></td>
<td style="padding:8px; border:1px solid #ddd;">104</td>
<td style="padding:8px; border:1px solid #ddd;">0.6635</td>
<td style="padding:8px; border:1px solid #ddd;">0.8365</td>
<td style="padding:8px; border:1px solid #ddd;">0.8654</td>
<td style="padding:8px; border:1px solid #ddd;">0.7452</td>
<td style="padding:8px; border:1px solid #ddd;">0.7609</td>
<td style="padding:8px; border:1px solid #ddd;">0.3946 ثانية</td>
</tr>

<tr>
<td style="padding:8px; border:1px solid #ddd;"><span dir="ltr">overall_diagnostic</span></td>
<td style="padding:8px; border:1px solid #ddd;">235</td>
<td style="padding:8px; border:1px solid #ddd;">0.8383</td>
<td style="padding:8px; border:1px solid #ddd;">0.9234</td>
<td style="padding:8px; border:1px solid #ddd;">0.9277</td>
<td style="padding:8px; border:1px solid #ddd;">0.8798</td>
<td style="padding:8px; border:1px solid #ddd;">0.8892</td>
<td style="padding:8px; border:1px solid #ddd;">0.4070 ثانية</td>
</tr>
</tbody>
</table>

<hr>

<h3>تفسير نتائج الأسئلة الشائعة FAQ</h3>

<p>
أظهرت أسئلة <span dir="ltr">FAQ</span> أداءً مرتفعًا جدًا، حيث حققت <span dir="ltr">Recall@1 = 0.9771</span> و <span dir="ltr">Recall@5 = 1.0000</span> و <span dir="ltr">MRR = 0.9866</span>. وهذا يعني أن وثيقة <span dir="ltr">FAQ</span> الصحيحة ظهرت غالبًا في المرتبة الأولى، وظهرت ضمن أفضل خمس نتائج في جميع الحالات.
</p>

<p>
هذه النتيجة منطقية لأن أسئلة <span dir="ltr">FAQ</span> تكون غالبًا قصيرة ومباشرة وقريبة دلاليًا من محتوى <span dir="ltr">FAQ</span> المفهرس.
</p>

<hr>

<h3>تفسير نتائج الأسئلة القانونية</h3>

<p>
بالنسبة للأسئلة القانونية، أظهر التحليل التشخيصي أن رقم المادة الصحيح ظهر في المرتبة الأولى في حوالي <b>66.35%</b> من الأسئلة القانونية، وظهر ضمن أفضل خمس نتائج في حوالي <b>86.54%</b> من الأسئلة.
</p>

<p>
هذه النتيجة تختلف عن تقييم <span dir="ltr">Full Corpus</span> لأن طريقة الحساب هنا مختلفة. في التقييم الرسمي يتم الاعتماد على <span dir="ltr">gold_document_unit_id</span>، أما في هذا التحليل التشخيصي فقد تم الاعتماد على أرقام المواد <span dir="ltr">article numbers</span>.
</p>

<p>
لذلك لا يجب مقارنة هذه النتائج مباشرة مع نتائج <span dir="ltr">Full Corpus</span> على أنها نفس نوع التقييم. لكنها مفيدة لفهم أن بعض الأسئلة القانونية قد تكون قريبة من المادة الصحيحة على مستوى رقم المادة، حتى لو لم يظهر نفس <span dir="ltr">document_unit_id</span> المطلوب في التقييم الرسمي.
</p>

<hr>

<h3>مقارنة التحليل التشخيصي مع تقييم Full Corpus</h3>

<table style="width:100%; border-collapse:collapse; text-align:right; direction:rtl;">
<thead>
<tr style="background-color:#1f4e79; color:white;">
<th style="padding:8px; border:1px solid #ddd;">نوع التقييم</th>
<th style="padding:8px; border:1px solid #ddd;"><span dir="ltr">Diagnostic Recall@1</span></th>
<th style="padding:8px; border:1px solid #ddd;"><span dir="ltr">Full Recall@1</span></th>
<th style="padding:8px; border:1px solid #ddd;"><span dir="ltr">Diagnostic Recall@5</span></th>
<th style="padding:8px; border:1px solid #ddd;"><span dir="ltr">Full Recall@5</span></th>
</tr>
</thead>

<tbody>
<tr>
<td style="padding:8px; border:1px solid #ddd;"><span dir="ltr">FAQ</span></td>
<td style="padding:8px; border:1px solid #ddd;">0.9771</td>
<td style="padding:8px; border:1px solid #ddd;">0.9771</td>
<td style="padding:8px; border:1px solid #ddd;">1.0000</td>
<td style="padding:8px; border:1px solid #ddd;">1.0000</td>
</tr>

<tr>
<td style="padding:8px; border:1px solid #ddd;"><span dir="ltr">Legal Articles</span></td>
<td style="padding:8px; border:1px solid #ddd;">0.6635</td>
<td style="padding:8px; border:1px solid #ddd;">0.4904</td>
<td style="padding:8px; border:1px solid #ddd;">0.8654</td>
<td style="padding:8px; border:1px solid #ddd;">0.9423</td>
</tr>
</tbody>
</table>

<p>
ارتفاع <span dir="ltr">Diagnostic Recall@1</span> في الأسئلة القانونية يشير إلى أن رقم المادة الصحيح قد يكون أقرب في الترتيب عند النظر إلى أرقام المواد. أما اختلاف <span dir="ltr">Diagnostic Recall@5</span> عن <span dir="ltr">Full Recall@5</span> فينتج عن اختلاف طريقة القياس، وليس بالضرورة عن تراجع فعلي في أداء النظام.
</p>

<hr>

<h3>هل النتائج منطقية؟</h3>

<p>
نعم، النتائج منطقية إذا تم تفسيرها كتحليل تشخيصي وليس كتقييم رسمي بديل. فالخلية تساعد في فهم أن بعض حالات الفشل في الأسئلة القانونية قد تكون ناتجة عن خلط المصادر أو عن اختلاف طريقة تمثيل المرجع الصحيح، وليس بالضرورة عن غياب المعلومة أو فشل كامل في الاسترجاع.
</p>

<p>
كما أن هذه النتائج تدعم الحاجة إلى إضافة طبقة توجيه <span dir="ltr">Router</span> أو فلترة حسب نوع المصدر <span dir="ltr">Source Filtering</span> في النظام النهائي، بحيث يتم توجيه الأسئلة القانونية إلى المواد القانونية، وتوجيه أسئلة الخدمات والإجراءات إلى <span dir="ltr">FAQ</span>.
</p>

<hr>

<h3>الخلاصة</h3>

<p>
يؤكد هذا التحليل أن تقييم <span dir="ltr">Full Corpus</span> يبقى هو التقييم الرسمي الأساسي، بينما يوفر هذا التحليل التشخيصي فهمًا أعمق لسلوك النظام عند التعامل مع مصادر معرفة مختلطة.
</p>

<p>
تشير النتائج إلى أن أداء النظام في <span dir="ltr">FAQ</span> قوي جدًا، بينما تبقى الأسئلة القانونية أكثر تحديًا بسبب طول المواد القانونية وتشابه بعض الموضوعات القانونية. كما توضح النتائج أن تحسين النظام النهائي يمكن أن يتم من خلال إضافة <span dir="ltr">Source-Aware Routing</span> لتحسين توجيه الأسئلة إلى نوع المصدر المناسب.
</p>

</div>


In [40]:

# =========================================================
# Stage 17 - Out-of-Scope Retrieval Behavior and Reliability Gate Calibration
# =========================================================

# This stage does not evaluate answer generation.
# It studies how strongly out-of-scope questions retrieve in-domain chunks.

assert "best_detail" in globals(), "Run Stage 16 before this stage."

df_oos = df_eval[df_eval["is_out_of_scope_bool"]].copy()
print("Out-of-scope questions:", len(df_oos))

oos_rows = []
if len(df_oos) > 0:
    for _, row in df_oos.iterrows():
        query = str(row["question"])
        results, latency = retrieve_with_config(
            query=query,
            corpus_name=best_config["chunking_strategy"],
            retriever_name=best_config["retriever"],
            model_key=None if best_config["embedding_model"] == "none" else best_config["embedding_model"],
            alpha=None if pd.isna(best_config.get("alpha")) else float(best_config["alpha"]),
            top_k=TOP_K,
        )
        top1 = results[0] if results else {}
        oos_rows.append({
            "eval_id": row.get("eval_id", ""),
            "question": query,
            "top1_score": top1.get("score", np.nan),
            "top1_source_type": top1.get("source_type", ""),
            "top1_article_number": top1.get("article_number_int", ""),
            "top1_article_title": top1.get("article_title", ""),
            "top1_parent_document_id": top1.get("parent_document_id", ""),
            "top1_retrieval_backend": top1.get("retrieval_backend", ""),
            "latency_sec": latency,
            "recommendation": "Use a reliability threshold before LLM generation.",
        })

oos_retrieval_behavior = pd.DataFrame(oos_rows)
display(oos_retrieval_behavior)
oos_retrieval_behavior.to_csv(STAGE03_DIR / "out_of_scope_retrieval_behavior.csv", index=False, encoding="utf-8-sig")

# Score distribution for successful in-scope Top-1 retrieval vs out-of-scope Top-1 retrieval.
in_scope_success_top1 = best_detail[best_detail["hit_at_1"] == 1].copy()

score_calibration = pd.DataFrame({
    "set": ["in_scope_successful_top1_scores", "out_of_scope_top1_scores"],
    "count": [len(in_scope_success_top1), len(oos_retrieval_behavior)],
    "mean_top1_score": [
        round(float(in_scope_success_top1["top1_score"].dropna().mean()), 4) if len(in_scope_success_top1) else np.nan,
        round(float(oos_retrieval_behavior["top1_score"].dropna().mean()), 4) if len(oos_retrieval_behavior) else np.nan,
    ],
    "median_top1_score": [
        round(float(in_scope_success_top1["top1_score"].dropna().median()), 4) if len(in_scope_success_top1) else np.nan,
        round(float(oos_retrieval_behavior["top1_score"].dropna().median()), 4) if len(oos_retrieval_behavior) else np.nan,
    ],
})

# Simple threshold suggestion. It is not a legal rule; it is a first calibration value for Stage 04.
in_median = score_calibration.loc[score_calibration["set"] == "in_scope_successful_top1_scores", "median_top1_score"].iloc[0]
oos_median = score_calibration.loc[score_calibration["set"] == "out_of_scope_top1_scores", "median_top1_score"].iloc[0]
if pd.notna(in_median) and pd.notna(oos_median):
    suggested_reliability_threshold = round(float((in_median + oos_median) / 2), 4)
elif pd.notna(in_median):
    suggested_reliability_threshold = round(float(in_median * 0.75), 4)
else:
    suggested_reliability_threshold = None

score_calibration["suggested_reliability_threshold"] = suggested_reliability_threshold

display(score_calibration)
score_calibration.to_csv(STAGE03_DIR / "reliability_gate_score_calibration.csv", index=False, encoding="utf-8-sig")

print("Suggested retrieval reliability threshold for Stage 04:", suggested_reliability_threshold)


Out-of-scope questions: 15


,eval_id,question,top1_score,top1_source_type,top1_article_number,top1_article_title,top1_parent_document_id,top1_retrieval_backend,latency_sec,recommendation
0,224,ما حكم الطلاق في الشريعة الإسلامية؟,0.001631,legal_article,4.0,المادة الرابعة:,3457807712bce4c43b55e497232052e55933f7fde19df543c46153e86ca95090,chromadb+bm25+reranker,0.505728,Use a reliability threshold before LLM generation.
1,225,كم سعر العقار في الرياض؟,0.000613,faq,nan,nan,519975c6e1d666802afd82015e412a87b72ebfc35625c9b31491a21d624d28d1,chromadb+bm25+reranker,0.404889,Use a reliability threshold before LLM generation.
2,226,ما أفضل سيارة عائلية؟,0.000824,faq,nan,nan,519975c6e1d666802afd82015e412a87b72ebfc35625c9b31491a21d624d28d1,chromadb+bm25+reranker,0.448785,Use a reliability threshold before LLM generation.
3,227,كيف أفتح مؤسسة تجارية؟,0.009604,legal_article,15.0,المادة الخامسة عشرة:,d83ce1bde91b6a48be946bc9636c4a1badd44979301ce6d2f80b1dd246c13b67,chromadb+bm25+reranker,0.273389,Use a reliability threshold before LLM generation.
4,228,ما علاج الصداع؟,0.002586,faq,nan,nan,519975c6e1d666802afd82015e412a87b72ebfc35625c9b31491a21d624d28d1,chromadb+bm25+reranker,0.447143,Use a reliability threshold before LLM generation.
5,229,كم سعر الذهب اليوم؟,0.002036,legal_article,230.0,المادة الثلاثون بعد المائتين:,3faa40187393f2d6ff9b6d6cec0e6873173fe08352831d7a7e1d883f3a7db2a8,chromadb+bm25+reranker,0.470069,Use a reliability threshold before LLM generation.
6,230,ما نظام المرور السعودي؟,0.002823,legal_article,230.0,المادة الثلاثون بعد المائتين:,3faa40187393f2d6ff9b6d6cec0e6873173fe08352831d7a7e1d883f3a7db2a8,chromadb+bm25+reranker,0.658901,Use a reliability threshold before LLM generation.
7,231,اكتب لي قصيدة عن الوطن.,0.001299,faq,nan,nan,519975c6e1d666802afd82015e412a87b72ebfc35625c9b31491a21d624d28d1,chromadb+bm25+reranker,0.457435,Use a reliability threshold before LLM generation.
8,232,ما شروط القرض العقاري؟,0.012404,faq,nan,nan,6aadca0a4c26a951315155e1511f3a9c5eb13a98cd4c165f3311b9f7018140f1,chromadb+bm25+reranker,0.423198,Use a reliability threshold before LLM generation.
9,233,ما رسوم الجمارك على السيارات؟,0.002551,legal_article,230.0,المادة الثلاثون بعد المائتين:,3faa40187393f2d6ff9b6d6cec0e6873173fe08352831d7a7e1d883f3a7db2a8,chromadb+bm25+reranker,0.463144,Use a reliability threshold before LLM generation.


,set,count,mean_top1_score,median_top1_score,suggested_reliability_threshold
0,in_scope_successful_top1_scores,179,0.9024,0.9940,0.4983
1,out_of_scope_top1_scores,15,0.0074,0.0026,0.4983


Suggested retrieval reliability threshold for Stage 04: 0.4983


<div dir="rtl" style="text-align:right; font-family:Tahoma, Arial; line-height:2; font-size:16px;">

<h2>تحليل أسئلة خارج النطاق وتحديد حد الموثوقية</h2>

<p>
تهدف هذه الخلية إلى اختبار قدرة النظام على التمييز بين الأسئلة الموجودة داخل نطاق مشروع
<span dir="ltr">Saudi Labor Law RAG</span>
والأسئلة الخارجة عن النطاق. ويعد هذا الاختبار جزءًا مهمًا من طبقة التحكم في الموثوقية
<span dir="ltr">Reliability / Guardrails Layer</span>
قبل تمرير السياق إلى نموذج اللغة الكبير.
</p>

<p>
في أنظمة الاسترجاع، يقوم النظام دائمًا بإرجاع أقرب وثيقة من قاعدة المعرفة حتى لو كان السؤال غير مرتبط بالمجال. لذلك لا يكفي الاعتماد على وجود نتيجة مسترجعة، بل يجب فحص درجة التشابه أو الموثوقية قبل توليد الإجابة.
</p>

<hr>

<h3>نتائج اختبار الأسئلة خارج النطاق</h3>

<p>
تم اختبار 15 سؤالًا خارج نطاق نظام العمل السعودي، مثل الأسئلة المتعلقة بالأسعار، السيارات، القروض، الأسهم، التأشيرات السياحية، أو مواضيع دينية وعامة. ورغم أن النظام أرجع نتائج من قاعدة المعرفة، إلا أن درجات التشابه كانت منخفضة جدًا.
</p>

<table style="width:100%; border-collapse:collapse; text-align:right; direction:rtl;">
<thead>
<tr style="background-color:#1f4e79; color:white;">
<th style="padding:8px; border:1px solid #ddd;">المجموعة</th>
<th style="padding:8px; border:1px solid #ddd;">عدد الأسئلة</th>
<th style="padding:8px; border:1px solid #ddd;">متوسط درجة أول نتيجة</th>
<th style="padding:8px; border:1px solid #ddd;">وسيط درجة أول نتيجة</th>
</tr>
</thead>
<tbody>
<tr>
<td style="padding:8px; border:1px solid #ddd;">أسئلة داخل النطاق وناجحة</td>
<td style="padding:8px; border:1px solid #ddd;">179</td>
<td style="padding:8px; border:1px solid #ddd;">0.9024</td>
<td style="padding:8px; border:1px solid #ddd;">0.9940</td>
</tr>
<tr>
<td style="padding:8px; border:1px solid #ddd;">أسئلة خارج النطاق</td>
<td style="padding:8px; border:1px solid #ddd;">15</td>
<td style="padding:8px; border:1px solid #ddd;">0.0074</td>
<td style="padding:8px; border:1px solid #ddd;">0.0026</td>
</tr>
</tbody>
</table>

<p>
توضح هذه النتائج وجود فرق كبير بين درجات الأسئلة داخل النطاق والأسئلة الخارجة عن النطاق. فالأسئلة الصحيحة داخل نطاق النظام حصلت على درجات تشابه مرتفعة جدًا، بينما حصلت الأسئلة الخارجة عن النطاق على درجات منخفضة للغاية.
</p>

<hr>

<h3>تفسير النتائج</h3>

<p>
على سبيل المثال، بعض الأسئلة الخارجة عن النطاق استرجعت وثائق من نوع
<span dir="ltr">legal_article</span>
أو
<span dir="ltr">faq</span>
لأن نظام الاسترجاع يقوم بإرجاع أقرب نتيجة متاحة دائمًا. لكن درجة التشابه كانت منخفضة جدًا، وهذا يدل على أن النتيجة غير موثوقة ولا يجب استخدامها لتوليد إجابة.
</p>

<p>
لذلك، لا تعتبر هذه النتائج خطأ في الاسترجاع، بل هي دليل على أهمية وجود حد موثوقية يمنع النظام من الإجابة على أسئلة لا تنتمي إلى مجال نظام العمل السعودي.
</p>

<hr>

<h3>حد الموثوقية المقترح</h3>

<p>
بناءً على الفرق الواضح بين درجات الأسئلة داخل النطاق وخارج النطاق، تم اقتراح حد موثوقية مقداره:
</p>

<div dir="ltr" style="text-align:left; background:#f5f7fa; border:1px solid #d9e2ec; padding:12px; border-radius:8px; font-family:Consolas, monospace;">
Suggested retrieval reliability threshold = 0.4983
</div>

<p>
هذا يعني أنه إذا كانت درجة أول نتيجة مسترجعة أقل من هذا الحد، يتم اعتبار السؤال غير موثوق أو خارج النطاق، ولا يتم تمريره إلى نموذج اللغة الكبير.
</p>

<div style="background:#d1e7dd; border-right:5px solid #198754; padding:12px 16px; border-radius:8px; margin:12px 0;">
<b>القرار المقترح:</b>
إذا كان <span dir="ltr">top1_score &lt; 0.4983</span>، يجب إيقاف توليد الإجابة وإرجاع رسالة آمنة للمستخدم تفيد بأن السؤال خارج نطاق نظام العمل السعودي أو أن النظام لا يملك سياقًا كافيًا للإجابة.
</div>

<hr>

<h3>الأهمية في نظام RAG</h3>

<p>
تعد هذه الخطوة مهمة جدًا في أنظمة
<span dir="ltr">RAG</span>
لأنها تقلل من خطر الهلوسة. فبدون هذا الفلتر، قد يستخدم نموذج اللغة الكبير سياقًا غير مناسب ويولد إجابة تبدو مقنعة لكنها غير صحيحة.
</p>

<p>
أما باستخدام حد الموثوقية، فإن النظام لا يجيب إلا عندما يكون الاسترجاع قويًا بما يكفي. وهذا يجعل النظام أكثر أمانًا وملاءمة للتطبيقات القانونية.
</p>

<hr>

<h3>الخلاصة</h3>

<p>
تؤكد نتائج هذه الخلية أن الأسئلة الخارجة عن نطاق نظام العمل السعودي تحصل على درجات استرجاع منخفضة جدًا مقارنة بالأسئلة الصحيحة داخل النطاق. لذلك فإن استخدام حد موثوقية قبل مرحلة توليد الإجابة يعد خطوة ضرورية في النظام النهائي.
</p>

<p>
وبناءً على النتائج، سيتم استخدام حد موثوقية تقريبي مقداره
<span dir="ltr">0.50</span>
أو القيمة الدقيقة
<span dir="ltr">0.4983</span>
لمنع تمرير الأسئلة غير الموثوقة إلى نموذج اللغة الكبير.
</p>

</div>

<div dir="rtl" style="text-align: right; line-height: 2; font-size: 16px;">

## اختبار الأسئلة خارج النطاق


الهدف هو بناء بوابة موثوقية قبل إرسال السياق إلى نموذج اللغة. إذا كان الاسترجاع ضعيفاً أو غير موثوق، يجب أن يرفض النظام الإجابة بدلاً من توليد إجابة غير مرتبطة.

</div>

In [42]:
# =========================================================
# Stage 15 - Academic Retrieval Visualisations Without Matplotlib
# رسوم أكاديمية بدون matplotlib بسبب حظر DLL في بيئة Windows
# =========================================================

import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display, HTML

# ---------------------------------------------------------
# 0) Prepare output directory
# ---------------------------------------------------------

if "FIGURES_DIR" not in globals():
    FIGURES_DIR = STAGE03_DIR / "figures"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("FIGURES_DIR:", FIGURES_DIR)


# ---------------------------------------------------------
# 1) Helper functions
# ---------------------------------------------------------

def save_html_file(filename, html_content):
    path = FIGURES_DIR / filename
    with open(path, "w", encoding="utf-8") as f:
        f.write(html_content)
    print("Saved HTML figure:", path)
    return path


def fmt_num(x, digits=4):
    if pd.isna(x):
        return ""
    try:
        return f"{float(x):.{digits}f}"
    except Exception:
        return str(x)


def display_status_card(title, message, status="info"):
    colors = {
        "success": ("#d1e7dd", "#0f5132", "#badbcc"),
        "warning": ("#fff3cd", "#664d03", "#ffecb5"),
        "danger":  ("#f8d7da", "#842029", "#f5c2c7"),
        "info":    ("#cff4fc", "#055160", "#b6effb"),
    }

    bg, color, border = colors.get(status, colors["info"])

    display(HTML(f"""
    <div dir="rtl" style="
        text-align:right;
        font-family:Tahoma, Arial;
        line-height:2;
        font-size:15px;
        background:{bg};
        color:{color};
        border:1px solid {border};
        padding:14px 16px;
        border-radius:8px;
        margin:12px 0;">
        <b>{title}</b><br>
        {message}
    </div>
    """))


def display_professional_table(df, title="Table", max_rows=20):
    view_df = df.head(max_rows).copy()

    html_table = view_df.to_html(index=False, escape=True, border=0)

    html = f"""
    <div dir="rtl" style="
        text-align:right;
        font-family:Tahoma, Arial;
        line-height:2;
        margin:18px 0 8px 0;">
        <h3 style="color:#17324d;">{title}</h3>
    </div>

    <style>
        .stage15-table-wrapper {{
            width:100%;
            overflow-x:auto;
            border:1px solid #d9e2ec;
            border-radius:10px;
            background:#ffffff;
            margin-bottom:20px;
        }}

        .stage15-table-wrapper table {{
            border-collapse:collapse;
            width:max-content;
            min-width:1100px;
            font-family:Tahoma, Arial, sans-serif;
            font-size:13px;
        }}

        .stage15-table-wrapper th {{
            background-color:#1f4e79;
            color:white;
            text-align:center;
            padding:9px 12px;
            border:1px solid #d9e2ec;
            white-space:nowrap;
        }}

        .stage15-table-wrapper td {{
            padding:8px 12px;
            border:1px solid #e5eaf0;
            text-align:center;
            white-space:nowrap;
        }}

        .stage15-table-wrapper tr:nth-child(even) {{
            background-color:#f8fbfd;
        }}
    </style>

    <div class="stage15-table-wrapper">
        {html_table}
    </div>
    """

    display(HTML(html))


def make_bar(value, max_value=1.0, width_px=260):
    if pd.isna(value):
        value = 0

    value = float(value)
    pct = max(0, min(value / max_value, 1)) * 100

    return f"""
    <div style="display:flex; align-items:center; gap:8px; min-width:{width_px}px;">
        <div style="width:{width_px}px; background:#eef2f6; border-radius:6px; overflow:hidden; height:18px;">
            <div style="width:{pct:.1f}%; background:#1f4e79; height:18px;"></div>
        </div>
        <span dir="ltr" style="font-family:Consolas, monospace; min-width:60px;">{value:.4f}</span>
    </div>
    """


def display_metric_bars(df, title, label_col, metric_col, max_rows=12, max_value=1.0):
    view_df = df.head(max_rows).copy()

    rows_html = ""

    for _, row in view_df.iterrows():
        label = str(row[label_col])
        value = row[metric_col]

        rows_html += f"""
        <tr>
            <td style="padding:9px 12px; border:1px solid #e5eaf0; text-align:right;">{label}</td>
            <td style="padding:9px 12px; border:1px solid #e5eaf0;">{make_bar(value, max_value=max_value)}</td>
        </tr>
        """

    html = f"""
    <div dir="rtl" style="font-family:Tahoma, Arial; text-align:right; line-height:2; margin-top:18px;">
        <h3 style="color:#17324d;">{title}</h3>
    </div>

    <div style="width:100%; overflow-x:auto; border:1px solid #d9e2ec; border-radius:10px; margin-bottom:22px;">
        <table style="width:100%; border-collapse:collapse; font-family:Tahoma, Arial; font-size:13px;">
            <thead>
                <tr style="background:#1f4e79; color:white;">
                    <th style="padding:9px 12px; border:1px solid #d9e2ec; text-align:right;">الإعداد</th>
                    <th style="padding:9px 12px; border:1px solid #d9e2ec; text-align:center;">{metric_col}</th>
                </tr>
            </thead>
            <tbody>
                {rows_html}
            </tbody>
        </table>
    </div>
    """

    display(HTML(html))
    return html


def display_grouped_metric_table(df, title, index_col, metrics):
    rows_html = ""

    for _, row in df.iterrows():
        metric_cells = ""

        for m in metrics:
            metric_cells += f"""
            <td style="padding:8px 10px; border:1px solid #e5eaf0;">
                {make_bar(row[m], max_value=1.0, width_px=180)}
            </td>
            """

        rows_html += f"""
        <tr>
            <td style="padding:8px 10px; border:1px solid #e5eaf0; text-align:right; white-space:nowrap;">
                <span dir="ltr">{row[index_col]}</span>
            </td>
            {metric_cells}
        </tr>
        """

    header_cells = "".join([
        f'<th style="padding:8px 10px; border:1px solid #d9e2ec;">{m}</th>'
        for m in metrics
    ])

    html = f"""
    <div dir="rtl" style="font-family:Tahoma, Arial; text-align:right; line-height:2; margin-top:18px;">
        <h3 style="color:#17324d;">{title}</h3>
    </div>

    <div style="width:100%; overflow-x:auto; border:1px solid #d9e2ec; border-radius:10px; margin-bottom:22px;">
        <table style="width:max-content; min-width:1200px; border-collapse:collapse; font-family:Tahoma, Arial; font-size:13px;">
            <thead>
                <tr style="background:#1f4e79; color:white;">
                    <th style="padding:8px 10px; border:1px solid #d9e2ec;">نوع التقييم</th>
                    {header_cells}
                </tr>
            </thead>
            <tbody>
                {rows_html}
            </tbody>
        </table>
    </div>
    """

    display(HTML(html))
    return html


def make_scatter_svg(df, x_col, y_col, label_col, title, max_points=12):
    plot_df = df.head(max_points).copy()

    width = 920
    height = 460
    left = 70
    right = 30
    top = 50
    bottom = 70

    x_min = float(plot_df[x_col].min())
    x_max = float(plot_df[x_col].max())
    y_min = float(plot_df[y_col].min())
    y_max = float(plot_df[y_col].max())

    if x_min == x_max:
        x_max = x_min + 1e-6
    if y_min == y_max:
        y_max = y_min + 1e-6

    def sx(x):
        return left + ((float(x) - x_min) / (x_max - x_min)) * (width - left - right)

    def sy(y):
        return height - bottom - ((float(y) - y_min) / (y_max - y_min)) * (height - top - bottom)

    circles = ""

    for _, row in plot_df.iterrows():
        x = sx(row[x_col])
        y = sy(row[y_col])
        label = str(row[label_col])[:32]

        circles += f"""
        <circle cx="{x:.1f}" cy="{y:.1f}" r="6" fill="#1f4e79"></circle>
        <text x="{x + 8:.1f}" y="{y - 8:.1f}" font-size="11" fill="#17324d">{label}</text>
        """

    svg = f"""
    <div dir="rtl" style="font-family:Tahoma, Arial; text-align:right; line-height:2; margin-top:18px;">
        <h3 style="color:#17324d;">{title}</h3>
    </div>

    <div style="width:100%; overflow-x:auto; border:1px solid #d9e2ec; border-radius:10px; background:#ffffff; padding:12px; margin-bottom:22px;">
        <svg width="{width}" height="{height}" xmlns="http://www.w3.org/2000/svg">
            <rect x="0" y="0" width="{width}" height="{height}" fill="#ffffff"></rect>

            <line x1="{left}" y1="{height-bottom}" x2="{width-right}" y2="{height-bottom}" stroke="#52616b" stroke-width="1"></line>
            <line x1="{left}" y1="{top}" x2="{left}" y2="{height-bottom}" stroke="#52616b" stroke-width="1"></line>

            <text x="{width/2}" y="28" text-anchor="middle" font-size="16" font-weight="700" fill="#17324d">{title}</text>
            <text x="{width/2}" y="{height-22}" text-anchor="middle" font-size="13" fill="#52616b">Average Latency per Query</text>
            <text x="20" y="{height/2}" text-anchor="middle" font-size="13" fill="#52616b" transform="rotate(-90 20,{height/2})">MRR</text>

            <text x="{left}" y="{height-bottom+22}" font-size="11" fill="#52616b">{x_min:.3f}</text>
            <text x="{width-right-35}" y="{height-bottom+22}" font-size="11" fill="#52616b">{x_max:.3f}</text>
            <text x="{left-48}" y="{height-bottom}" font-size="11" fill="#52616b">{y_min:.3f}</text>
            <text x="{left-48}" y="{top+5}" font-size="11" fill="#52616b">{y_max:.3f}</text>

            {circles}
        </svg>
    </div>
    """

    display(HTML(svg))
    return svg


# ---------------------------------------------------------
# 2) Best configuration performance by evaluation type
# ---------------------------------------------------------

best = retrieval_summary.iloc[0].copy()

best_rows = retrieval_detail[
    (retrieval_detail["chunking_strategy"] == best["chunking_strategy"]) &
    (retrieval_detail["embedding_model"] == best["embedding_model"]) &
    (retrieval_detail["retriever"] == best["retriever"])
].copy()

if pd.isna(best["alpha"]):
    best_rows = best_rows[best_rows["alpha"].isna()]
else:
    best_rows = best_rows[
        best_rows["alpha"].astype(float).round(4) == float(best["alpha"])
    ]

performance_by_type = (
    best_rows
    .groupby("eval_type")
    .agg(
        questions=("question", "count"),
        recall_at_1=("hit_at_1", "mean"),
        recall_at_3=("hit_at_3", "mean"),
        recall_at_5=("hit_at_5", "mean"),
        mrr=("rr", "mean"),
        ndcg_at_5=("ndcg_at_5", "mean"),
        avg_latency_sec=("latency_sec", "mean"),
    )
    .reset_index()
)

for col in ["recall_at_1", "recall_at_3", "recall_at_5", "mrr", "ndcg_at_5", "avg_latency_sec"]:
    performance_by_type[col] = performance_by_type[col].round(4)

display_professional_table(
    performance_by_type,
    title="Performance by Evaluation Type for Best Configuration",
    max_rows=20
)

metrics = ["recall_at_1", "recall_at_3", "recall_at_5", "mrr", "ndcg_at_5"]

html_1 = display_grouped_metric_table(
    performance_by_type,
    title="Best Retrieval Configuration Performance by Evaluation Type",
    index_col="eval_type",
    metrics=metrics
)

save_html_file("stage03_01_best_config_performance_by_eval_type.html", html_1)


# ---------------------------------------------------------
# 3) Legal Retrieval Only - Top Configurations by MRR
# ---------------------------------------------------------

legal_summary = (
    retrieval_detail[retrieval_detail["eval_type"] == "legal_article_retrieval"]
    .groupby(["chunking_strategy", "embedding_model", "retriever", "alpha"], dropna=False)
    .agg(
        questions=("question", "count"),
        recall_at_1=("hit_at_1", "mean"),
        recall_at_3=("hit_at_3", "mean"),
        recall_at_5=("hit_at_5", "mean"),
        mrr=("rr", "mean"),
        ndcg_at_5=("ndcg_at_5", "mean"),
        avg_latency_sec=("latency_sec", "mean"),
    )
    .reset_index()
)

legal_summary = legal_summary.sort_values(
    ["mrr", "recall_at_5", "recall_at_1", "avg_latency_sec"],
    ascending=[False, False, False, True]
).reset_index(drop=True)

legal_top = legal_summary.head(8).copy()

legal_top["label"] = (
    legal_top["chunking_strategy"].astype(str) + " | " +
    legal_top["embedding_model"].astype(str) + " | " +
    legal_top["retriever"].astype(str) + " | a=" +
    legal_top["alpha"].fillna("").astype(str)
)

for col in ["recall_at_1", "recall_at_3", "recall_at_5", "mrr", "ndcg_at_5", "avg_latency_sec"]:
    legal_top[col] = legal_top[col].round(4)

display_professional_table(
    legal_top,
    title="Legal Article Retrieval - Top Configurations",
    max_rows=8
)

html_2 = display_metric_bars(
    legal_top,
    title="Legal Article Retrieval: Top Configurations by MRR",
    label_col="label",
    metric_col="mrr",
    max_rows=8,
    max_value=1.0
)

save_html_file("stage03_02_legal_top_configurations_by_mrr.html", html_2)


# ---------------------------------------------------------
# 4) Accuracy-Latency Trade-off
# ---------------------------------------------------------

tradeoff_df = retrieval_summary.copy()

tradeoff_df["label"] = (
    tradeoff_df["retriever"].astype(str) + " | " +
    tradeoff_df["embedding_model"].astype(str)
)

tradeoff_df = tradeoff_df.sort_values(
    ["mrr", "recall_at_5", "recall_at_1"],
    ascending=[False, False, False]
).reset_index(drop=True)

html_3 = make_scatter_svg(
    tradeoff_df,
    x_col="avg_latency_sec",
    y_col="mrr",
    label_col="label",
    title="Retrieval Accuracy-Latency Trade-off",
    max_points=12
)

save_html_file("stage03_03_accuracy_latency_tradeoff.html", html_3)


# ---------------------------------------------------------
# 5) Reliability Gate Calibration
# ---------------------------------------------------------

if "score_calibration" in globals():
    score_plot = score_calibration.copy()

    display_professional_table(
        score_plot,
        title="Reliability Gate Calibration Summary",
        max_rows=10
    )

    rows_html = ""

    threshold = float(score_plot["suggested_reliability_threshold"].iloc[0])

    for _, row in score_plot.iterrows():
        rows_html += f"""
        <tr>
            <td style="padding:9px 12px; border:1px solid #e5eaf0; text-align:right;">
                <span dir="ltr">{row['set']}</span>
            </td>
            <td style="padding:9px 12px; border:1px solid #e5eaf0;">
                {make_bar(row['median_top1_score'], max_value=1.0)}
            </td>
        </tr>
        """

    html_4 = f"""
    <div dir="rtl" style="font-family:Tahoma, Arial; text-align:right; line-height:2; margin-top:18px;">
        <h3 style="color:#17324d;">Reliability Gate Calibration</h3>
        <p>
            الخط المتقطع في الرسم الأصلي تم تمثيله هنا كقيمة مرجعية للحد المقترح:
            <span dir="ltr">{threshold:.4f}</span>
        </p>
    </div>

    <div style="width:100%; overflow-x:auto; border:1px solid #d9e2ec; border-radius:10px; margin-bottom:22px;">
        <table style="width:100%; border-collapse:collapse; font-family:Tahoma, Arial; font-size:13px;">
            <thead>
                <tr style="background:#1f4e79; color:white;">
                    <th style="padding:9px 12px; border:1px solid #d9e2ec; text-align:right;">Query Set</th>
                    <th style="padding:9px 12px; border:1px solid #d9e2ec; text-align:center;">Median Top-1 Score</th>
                </tr>
            </thead>
            <tbody>
                {rows_html}
            </tbody>
        </table>
    </div>

    <div dir="rtl" style="
        background:#d1e7dd;
        border-right:5px solid #198754;
        padding:12px 16px;
        border-radius:8px;
        font-family:Tahoma, Arial;
        line-height:2;
        margin-bottom:20px;">
        <b>Suggested reliability threshold:</b>
        <span dir="ltr">{threshold:.4f}</span>
    </div>
    """

    display(HTML(html_4))
    save_html_file("stage03_04_reliability_gate_calibration.html", html_4)

else:
    display_status_card(
        title="score_calibration غير موجود",
        message="شغّل خلية Reliability Gate Calibration أولًا حتى يتم إنشاء الرسم الخاص بحد الموثوقية.",
        status="warning"
    )


# ---------------------------------------------------------
# 6) Save tabular outputs
# ---------------------------------------------------------

performance_by_type.to_csv(
    STAGE03_DIR / "stage03_visual_performance_by_type.csv",
    index=False,
    encoding="utf-8-sig"
)

legal_top.to_csv(
    STAGE03_DIR / "stage03_visual_legal_top_configurations.csv",
    index=False,
    encoding="utf-8-sig"
)

tradeoff_df.to_csv(
    STAGE03_DIR / "stage03_visual_accuracy_latency_tradeoff.csv",
    index=False,
    encoding="utf-8-sig"
)

display_status_card(
    title="Stage 15 completed successfully",
    message=(
        "تم إنشاء الرسوم الأكاديمية بصيغة HTML بدل PNG بسبب حظر matplotlib في بيئة الجهاز. "
        "تم حفظ ملفات HTML و CSV داخل مجلد النتائج."
    ),
    status="success"
)

FIGURES_DIR: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\05_reports\figures_stage03


eval_type,questions,recall_at_1,recall_at_3,recall_at_5,mrr,ndcg_at_5,avg_latency_sec
faq_paraphrase_retrieval,131,0.9771,0.9924,1.0000,0.9866,0.9900,0.4166
legal_article_retrieval,104,0.4904,0.8462,0.9423,0.6716,0.7352,0.3948


نوع التقييم,recall_at_1,recall_at_3,recall_at_5,mrr,ndcg_at_5
faq_paraphrase_retrieval,0.9771,0.9924,1.0000,0.9866,0.9900
legal_article_retrieval,0.4904,0.8462,0.9423,0.6716,0.7352


Saved HTML figure: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\05_reports\figures_stage03\stage03_01_best_config_performance_by_eval_type.html


chunking_strategy,embedding_model,retriever,alpha,questions,recall_at_1,recall_at_3,recall_at_5,mrr,ndcg_at_5,avg_latency_sec,label
structural,bge_m3,hybrid_reranker,0.8,104,0.4904,0.8462,0.9423,0.6716,0.7352,0.3948,structural | bge_m3 | hybrid_reranker | a=0.8
fixed_size,bge_m3,hybrid_reranker,0.8,104,0.4808,0.8558,0.9423,0.6676,0.7325,0.3791,fixed_size | bge_m3 | hybrid_reranker | a=0.8
fixed_size,e5_large,hybrid_reranker,0.8,104,0.4423,0.8173,0.8942,0.6300,0.6925,0.3600,fixed_size | e5_large | hybrid_reranker | a=0.8
structural,e5_large,hybrid_reranker,0.8,104,0.4423,0.7981,0.8846,0.6207,0.6834,0.3757,structural | e5_large | hybrid_reranker | a=0.8
fixed_size,bge_m3,dense,NaN,104,0.3654,0.7308,0.8077,0.5402,0.6049,0.0121,fixed_size | bge_m3 | dense | a=
structural,bge_m3,dense,NaN,104,0.2981,0.6731,0.7981,0.4845,0.5595,0.0114,structural | bge_m3 | dense | a=
fixed_size,bge_m3,hybrid,0.9,104,0.2981,0.6635,0.7885,0.4776,0.5562,0.0220,fixed_size | bge_m3 | hybrid | a=0.9
structural,bge_m3,hybrid,0.9,104,0.2596,0.6058,0.7981,0.4481,0.5352,0.0232,structural | bge_m3 | hybrid | a=0.9


...

Saved HTML figure: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\05_reports\figures_stage03\stage03_02_legal_top_configurations_by_mrr.html


Saved HTML figure: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\05_reports\figures_stage03\stage03_03_accuracy_latency_tradeoff.html


set,count,mean_top1_score,median_top1_score,suggested_reliability_threshold
in_scope_successful_top1_scores,179,0.9024,0.9940,0.4983
out_of_scope_top1_scores,15,0.0074,0.0026,0.4983


Query Set,Median Top-1 Score
in_scope_successful_top1_scores,0.9940
out_of_scope_top1_scores,0.0026


Saved HTML figure: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\05_reports\figures_stage03\stage03_04_reliability_gate_calibration.html



<div dir="rtl" style="text-align: right; line-height: 2; font-size: 16px;">

## تجهيز طبقة RAG النهائية للمرحلة الرابعة

بعد اختيار أفضل إعداد للاسترجاع، يتم في هذه الخطوة بناء دالة استرجاع نهائية قابلة للاستخدام في المرحلة الرابعة. هذه الدالة لا تستدعي نموذج اللغة الكبير، بل ترجع المقاطع القانونية أو الأسئلة الشائعة الأكثر صلة مع مصادرها، ثم تبني حزمة سياق جاهزة لإرسالها إلى نموذج اللغة لاحقاً.

بهذا تكون المرحلة الثالثة قد أنجزت طبقة الاسترجاع كاملة: قاعدة المتجهات، الاسترجاع، الترتيب، فحص الثقة، وتجهيز السياق.

</div>


In [43]:
# =========================================================
# Stage 19 - Final RAG Retriever Packaging for Stage 04 LLM
# تجهيز دالة الاسترجاع النهائية وجدول أمثلة RAG
# =========================================================

import json
import time
import pandas as pd
import numpy as np
from IPython.display import display, HTML

# ---------------------------------------------------------
# 1) Required variable checks
# ---------------------------------------------------------

required_objects = [
    "best_config",
    "retrieve_with_config",
    "suggested_reliability_threshold",
    "TOP_K",
    "STAGE03_DIR"
]

missing_objects = [obj for obj in required_objects if obj not in globals()]

assert not missing_objects, (
    "Missing required objects before Stage 19: "
    + ", ".join(missing_objects)
    + ". Run the previous retrieval evaluation stages first."
)

# Convert best_config safely if it is a pandas Series
if isinstance(best_config, pd.Series):
    best_config = best_config.to_dict()

assert isinstance(best_config, dict), "best_config must be a dictionary or pandas Series."

# ---------------------------------------------------------
# 2) Extract best retrieval configuration
# ---------------------------------------------------------

best_chunking_strategy = str(best_config.get("chunking_strategy", "")).strip()
best_embedding_model = str(best_config.get("embedding_model", "")).strip()
best_retriever_name = str(best_config.get("retriever", "")).strip()

best_alpha = best_config.get("alpha", None)

try:
    if best_alpha is None or pd.isna(best_alpha):
        best_alpha = None
    else:
        best_alpha = float(best_alpha)
except Exception:
    best_alpha = None

best_model_key = None if best_embedding_model in ["", "none", "nan", "None"] else best_embedding_model

assert best_chunking_strategy != "", "best_chunking_strategy is empty."
assert best_retriever_name != "", "best_retriever_name is empty."

# ---------------------------------------------------------
# 3) Reliability threshold
# ---------------------------------------------------------

try:
    reliability_threshold = float(suggested_reliability_threshold)
except Exception:
    reliability_threshold = None

print("Best retrieval configuration:")
print("Chunking strategy:", best_chunking_strategy)
print("Embedding model:", best_embedding_model)
print("Retriever:", best_retriever_name)
print("Alpha:", best_alpha)
print("Reliability threshold:", reliability_threshold)


# ---------------------------------------------------------
# 4) Helper functions
# ---------------------------------------------------------

def safe_float(value, default=np.nan):
    try:
        if value is None:
            return default
        if pd.isna(value):
            return default
        return float(value)
    except Exception:
        return default


def safe_str(value, default=""):
    try:
        if value is None:
            return default
        if pd.isna(value):
            return default
        return str(value)
    except Exception:
        return default


def clean_text_preview(text, max_chars=800):
    text = safe_str(text).replace("\n", " ").replace("\r", " ").strip()
    text = " ".join(text.split())
    if len(text) > max_chars:
        return text[:max_chars] + "..."
    return text


def normalize_json_value(value):
    """
    Convert numpy/pandas values into JSON-safe Python values.
    """
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        if np.isnan(value):
            return None
        return float(value)
    if isinstance(value, (np.ndarray,)):
        return value.tolist()
    if pd.isna(value) if not isinstance(value, (list, dict, tuple)) else False:
        return None
    return value


def make_json_safe_dict(d):
    return {k: normalize_json_value(v) for k, v in d.items()}


def build_context_for_llm(results, max_results=5):
    """
    Build a structured context block for the next LLM stage.
    This stage does not generate an answer; it only prepares retrieved evidence.
    """
    context_blocks = []

    for r in results[:max_results]:
        rank = safe_str(r.get("rank", ""))
        source_type = safe_str(r.get("source_type", ""))
        score = safe_float(r.get("score", np.nan))
        article_no = safe_str(
            r.get("article_number_int", "")
            or r.get("article_number_label", "")
        )
        article_title = safe_str(r.get("article_title", ""))
        chunk_id = safe_str(r.get("chunk_id", ""))
        parent_id = safe_str(r.get("parent_document_id", ""))
        text = safe_str(r.get("text", ""))

        header_parts = [
            f"Rank: {rank}",
            f"Source type: {source_type}",
            f"Score: {score:.4f}" if pd.notna(score) else "Score: N/A",
            f"Chunk ID: {chunk_id}",
            f"Parent document ID: {parent_id}",
        ]

        if article_no:
            header_parts.append(f"Article number: {article_no}")

        if article_title:
            header_parts.append(f"Article title: {article_title}")

        block = "\n".join([
            " | ".join(header_parts),
            text
        ])

        context_blocks.append(block)

    return "\n\n---\n\n".join(context_blocks)


# ---------------------------------------------------------
# 5) Final RAG retrieval function
# ---------------------------------------------------------

def final_rag_retrieve(query, top_k=TOP_K):
    """
    Final retrieval function for Stage 04 LLM integration.
    
    This function:
    - uses the best retrieval configuration selected in Stage 14
    - retrieves the top-k relevant chunks
    - applies the reliability threshold
    - prepares a context package for the LLM stage
    
    It does not generate the final answer.
    """
    query = safe_str(query).strip()

    results, latency = retrieve_with_config(
        query=query,
        corpus_name=best_chunking_strategy,
        retriever_name=best_retriever_name,
        model_key=best_model_key,
        alpha=best_alpha,
        top_k=top_k,
    )

    if results is None:
        results = []

    top1 = results[0] if len(results) > 0 else {}

    top1_score = safe_float(top1.get("score", np.nan))

    if reliability_threshold is None or pd.isna(top1_score):
        is_retrieval_reliable = False
    else:
        is_retrieval_reliable = bool(top1_score >= reliability_threshold)

    context_for_llm = build_context_for_llm(results, max_results=top_k)

    output = {
        "query": query,
        "is_retrieval_reliable": is_retrieval_reliable,
        "suggested_reliability_threshold": reliability_threshold,
        "top1_score": top1_score,
        "top1_source_type": safe_str(top1.get("source_type", "")),
        "top1_article_number": safe_str(
            top1.get("article_number_int", "")
            or top1.get("article_number_label", "")
        ),
        "top1_article_title": safe_str(top1.get("article_title", "")),
        "top1_chunk_id": safe_str(top1.get("chunk_id", "")),
        "top1_parent_document_id": safe_str(top1.get("parent_document_id", "")),
        "top1_backend": safe_str(top1.get("retrieval_backend", "")),
        "latency_sec": safe_float(latency, default=np.nan),
        "context_preview": clean_text_preview(context_for_llm, max_chars=800),
        "context_for_llm": context_for_llm,
        "retrieved_results": results,
        "best_chunking_strategy": best_chunking_strategy,
        "best_embedding_model": best_embedding_model,
        "best_retriever": best_retriever_name,
        "best_alpha": best_alpha,
    }

    return output


# ---------------------------------------------------------
# 6) Build final RAG examples dataframe
# ---------------------------------------------------------

rag_example_questions = [
    # Legal questions
    "كم مدة فترة التجربة في نظام العمل السعودي؟",
    "متى يستحق العامل مكافأة نهاية الخدمة؟",
    "كم عدد ساعات العمل في نظام العمل السعودي؟",
    "هل يحق لصاحب العمل نقل العامل من فرع إلى فرع؟",
    "ما مدة الإجازة السنوية للعامل؟",
    
    # FAQ / service-style questions
    "كيف أقدم شكوى على صاحب العمل؟",
    "ما حقوق العامل عند انتهاء عقد العمل؟",
    
    # Out-of-scope tests
    "ما حكم الطلاق؟",
    "كم سعر العقار في الرياض؟",
    "ما أفضل سيارة عائلية؟"
]

rag_example_rows = []
query_errors = []

for q in rag_example_questions:
    try:
        result = final_rag_retrieve(q, top_k=TOP_K)

        rag_example_rows.append({
            "query": result["query"],
            "is_retrieval_reliable": result["is_retrieval_reliable"],
            "suggested_reliability_threshold": result["suggested_reliability_threshold"],
            "top1_score": result["top1_score"],
            "top1_source_type": result["top1_source_type"],
            "top1_article_number": result["top1_article_number"],
            "top1_article_title": result["top1_article_title"],
            "top1_chunk_id": result["top1_chunk_id"],
            "top1_parent_document_id": result["top1_parent_document_id"],
            "top1_backend": result["top1_backend"],
            "latency_sec": result["latency_sec"],
            "context_preview": result["context_preview"],
            "context_for_llm": result["context_for_llm"],
            "best_chunking_strategy": result["best_chunking_strategy"],
            "best_embedding_model": result["best_embedding_model"],
            "best_retriever": result["best_retriever"],
            "best_alpha": result["best_alpha"],
        })

    except Exception as e:
        query_errors.append((q, str(e)))

        rag_example_rows.append({
            "query": q,
            "is_retrieval_reliable": False,
            "suggested_reliability_threshold": reliability_threshold,
            "top1_score": np.nan,
            "top1_source_type": "",
            "top1_article_number": "",
            "top1_article_title": "",
            "top1_chunk_id": "",
            "top1_parent_document_id": "",
            "top1_backend": "",
            "latency_sec": np.nan,
            "context_preview": f"Retrieval error: {e}",
            "context_for_llm": "",
            "best_chunking_strategy": best_chunking_strategy,
            "best_embedding_model": best_embedding_model,
            "best_retriever": best_retriever_name,
            "best_alpha": best_alpha,
        })

rag_examples_df = pd.DataFrame(rag_example_rows)


# ---------------------------------------------------------
# 7) Save outputs using manifest-compatible filenames
# ---------------------------------------------------------

rag_examples_csv = STAGE03_DIR / "stage03_rag_retriever_context_examples.csv"
rag_examples_xlsx = STAGE03_DIR / "stage03_rag_retriever_context_examples.xlsx"
final_rag_config_json = STAGE03_DIR / "stage03_best_retriever_ready_for_llm_config.json"

rag_examples_df.to_csv(
    rag_examples_csv,
    index=False,
    encoding="utf-8-sig"
)

rag_examples_df.to_excel(
    rag_examples_xlsx,
    index=False
)

final_rag_config = {
    "best_chunking_strategy": best_chunking_strategy,
    "best_embedding_model": best_embedding_model,
    "best_retriever": best_retriever_name,
    "best_alpha": best_alpha,
    "top_k": TOP_K,
    "suggested_reliability_threshold": reliability_threshold,
    "notes": "This configuration is ready for Stage 04 LLM generation. It prepares retrieved context only and does not generate answers."
}

with open(final_rag_config_json, "w", encoding="utf-8") as f:
    json.dump(make_json_safe_dict(final_rag_config), f, ensure_ascii=False, indent=2)


# ---------------------------------------------------------
# 8) Display summary
# ---------------------------------------------------------

summary_stage19 = pd.DataFrame([{
    "best_chunking_strategy": best_chunking_strategy,
    "best_embedding_model": best_embedding_model,
    "best_retriever": best_retriever_name,
    "best_alpha": best_alpha,
    "top_k": TOP_K,
    "suggested_reliability_threshold": reliability_threshold,
    "example_questions": len(rag_examples_df),
    "reliable_examples": int(rag_examples_df["is_retrieval_reliable"].sum()),
    "unreliable_examples": int((~rag_examples_df["is_retrieval_reliable"]).sum()),
    "query_errors": len(query_errors),
    "status": "OK" if len(query_errors) == 0 else "WARN",
}])

display(HTML("""
<div dir="rtl" style="text-align:right; line-height:2; font-size:17px;">
<h3>Stage 19 — Final RAG Retriever Packaging</h3>
<p>
تم في هذه المرحلة بناء دالة الاسترجاع النهائية 
<span dir="ltr" style="display:inline-block;">final_rag_retrieve()</span>
باستخدام أفضل إعداد تم اختياره من نتائج التقييم. كما تم إنشاء جدول أمثلة 
<span dir="ltr" style="display:inline-block;">rag_examples_df</span>
لاختبار جاهزية طبقة الاسترجاع قبل تمرير السياق إلى نموذج اللغة في المرحلة الرابعة.
</p>
</div>
"""))

display(
    summary_stage19
    .style
    .set_caption("Final RAG Retriever Packaging Summary")
    .set_properties(**{
        "text-align": "center",
        "white-space": "normal",
        "font-size": "12px"
    })
    .set_table_styles([
        {
            "selector": "caption",
            "props": [
                ("caption-side", "top"),
                ("font-size", "16px"),
                ("font-weight", "bold"),
                ("text-align", "left")
            ]
        },
        {
            "selector": "th",
            "props": [
                ("background-color", "#F3F4F6"),
                ("font-weight", "bold"),
                ("text-align", "center")
            ]
        }
    ])
)

display(rag_examples_df)

print("Stage 19 completed successfully.")
print("best_retriever_name:", best_retriever_name)
print("final_rag_retrieve:", "FOUND" if "final_rag_retrieve" in globals() else "MISSING")
print("rag_examples_df:", "FOUND" if "rag_examples_df" in globals() else "MISSING")
print("rag_examples_df shape:", rag_examples_df.shape)

print("\nSaved files:")
print(rag_examples_csv)
print(rag_examples_xlsx)
print(final_rag_config_json)

if query_errors:
    print("\nWarnings:")
    for q, err in query_errors:
        print("-", q, "=>", err)

Best retrieval configuration:
Chunking strategy: structural
Embedding model: bge_m3
Retriever: hybrid_reranker
Alpha: 0.8
Reliability threshold: 0.4983


,best_chunking_strategy,best_embedding_model,best_retriever,best_alpha,top_k,suggested_reliability_threshold,example_questions,reliable_examples,unreliable_examples,query_errors,status
0,structural,bge_m3,hybrid_reranker,0.800000,5,0.498300,10,7,3,0,OK


,query,is_retrieval_reliable,suggested_reliability_threshold,top1_score,top1_source_type,top1_article_number,top1_article_title,top1_chunk_id,top1_parent_document_id,top1_backend,latency_sec,context_preview,context_for_llm,best_chunking_strategy,best_embedding_model,best_retriever,best_alpha
0,كم مدة فترة التجربة في نظام العمل السعودي؟,True,0.4983,0.970917,legal_article,53.0,المادة الثالثة والخمسون :,STRUCT_000053,b56e9c99f1c551928c8727dae90850186c14efc78d6272f429f0c448add3f3dc,chromadb+bm25+reranker,0.887309,Rank: 1 | Source type: legal_article | Score: 0.9709 | Chunk ID: STRUCT_000053 | Parent document ID: b56e9c99f1c5519...,Rank: 1 | Source type: legal_article | Score: 0.9709 | Chunk ID: STRUCT_000053 | Parent document ID: b56e9c99f1c5519...,structural,bge_m3,hybrid_reranker,0.8
1,متى يستحق العامل مكافأة نهاية الخدمة؟,True,0.4983,0.843507,legal_article,85.0,المادة الخامسة والثمانون:,STRUCT_000086,343ee48aabd2cf26f0312d0efdb39d7547ea2d705cf92251d1663ca801c6d9d0,chromadb+bm25+reranker,0.303437,Rank: 1 | Source type: legal_article | Score: 0.8435 | Chunk ID: STRUCT_000086 | Parent document ID: 343ee48aabd2cf2...,Rank: 1 | Source type: legal_article | Score: 0.8435 | Chunk ID: STRUCT_000086 | Parent document ID: 343ee48aabd2cf2...,structural,bge_m3,hybrid_reranker,0.8
2,كم عدد ساعات العمل في نظام العمل السعودي؟,True,0.4983,0.931134,legal_article,99.0,المادة التاسعة والتسعون:,STRUCT_000100,4d7b6567adef57ae0d1350a027920ae7d11610e7cf8da52e68e0c806900f687e,chromadb+bm25+reranker,0.659956,Rank: 1 | Source type: legal_article | Score: 0.9311 | Chunk ID: STRUCT_000100 | Parent document ID: 4d7b6567adef57a...,Rank: 1 | Source type: legal_article | Score: 0.9311 | Chunk ID: STRUCT_000100 | Parent document ID: 4d7b6567adef57a...,structural,bge_m3,hybrid_reranker,0.8
3,هل يحق لصاحب العمل نقل العامل من فرع إلى فرع؟,True,0.4983,0.920061,faq,nan,nan,STRUCT_000429,706550432de905d72a6ff5a565caaa61d2cbd016555a049798a31ebf73a58218,chromadb+bm25+reranker,0.363128,Rank: 1 | Source type: faq | Score: 0.9201 | Chunk ID: STRUCT_000429 | Parent document ID: 706550432de905d72a6ff5a56...,Rank: 1 | Source type: faq | Score: 0.9201 | Chunk ID: STRUCT_000429 | Parent document ID: 706550432de905d72a6ff5a56...,structural,bge_m3,hybrid_reranker,0.8
4,ما مدة الإجازة السنوية للعامل؟,True,0.4983,0.996335,faq,nan,nan,STRUCT_000426,f196cb04ac077b585020c6b00dd03592845742a0e51983c0e22991f007640855,chromadb+bm25+reranker,0.680873,Rank: 1 | Source type: faq | Score: 0.9963 | Chunk ID: STRUCT_000426 | Parent document ID: f196cb04ac077b585020c6b00...,Rank: 1 | Source type: faq | Score: 0.9963 | Chunk ID: STRUCT_000426 | Parent document ID: f196cb04ac077b585020c6b00...,structural,bge_m3,hybrid_reranker,0.8
5,كيف أقدم شكوى على صاحب العمل؟,True,0.4983,0.777631,faq,nan,nan,STRUCT_000414,f8c1fa0670dd40b9641c4117f774a20e8202b417a8efe3605431a508947e371d,chromadb+bm25+reranker,0.362387,Rank: 1 | Source type: faq | Score: 0.7776 | Chunk ID: STRUCT_000414 | Parent document ID: f8c1fa0670dd40b9641c4117f...,Rank: 1 | Source type: faq | Score: 0.7776 | Chunk ID: STRUCT_000414 | Parent document ID: f8c1fa0670dd40b9641c4117f...,structural,bge_m3,hybrid_reranker,0.8
6,ما حقوق العامل عند انتهاء عقد العمل؟,True,0.4983,0.973302,legal_article,64.0,المادة الرابعة والستون:,STRUCT_000064,75712469249444225b0d8076c807d51f8c883d982d9aa3143725251780f39f25,chromadb+bm25+reranker,0.671151,Rank: 1 | Source type: legal_article | Score: 0.9733 | Chunk ID: STRUCT_000064 | Parent document ID: 757124692494442...,Rank: 1 | Source type: legal_article | Score: 0.9733 | Chunk ID: STRUCT_000064 | Parent document ID: 757124692494442...,structural,bge_m3,hybrid_reranker,0.8
7,ما حكم الطلاق؟,False,0.4983,0.031113,legal_article,2.0,المادة الثانية :,STRUCT_000002,4f2961046b5fffdc637e2b7de4dfb1fd862ccc7b9f3069349681b04ff17df638,chromadb+bm25+reranker,0.660110,Rank: 1 | Source type: legal_article | Score: 0.0311 | Chunk ID: STRUCT_000002 | Parent document ID: 4f2961046b5fffd...,Rank: 1 | Source type: legal_article |

Stage 19 completed successfully.
best_retriever_name: hybrid_reranker
final_rag_retrieve: FOUND
rag_examples_df: FOUND
rag_examples_df shape: (10, 17)

Saved files:
C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\stage03_rag_retriever_context_examples.csv
C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\stage03_rag_retriever_context_examples.xlsx
C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\stage03_best_retriever_ready_for_llm_config.json


In [44]:
# =========================================================
# Debug - Check required final RAG variables
# =========================================================

required_vars = [
    "best_config",
    "best_retriever_name",
    "final_rag_retrieve",
    "rag_examples_df",
    "suggested_reliability_threshold"
]

for var in required_vars:
    print(f"{var}:", "FOUND" if var in globals() else "MISSING")

best_config: FOUND
best_retriever_name: FOUND
final_rag_retrieve: FOUND
rag_examples_df: FOUND
suggested_reliability_threshold: FOUND


In [50]:
# =========================================================
# Final RAG Retriever Examples Table Before Stage 04 LLM
# جدول احترافي ومحمي لاختبار طبقة الاسترجاع النهائية قبل LLM
# =========================================================

from IPython.display import display, HTML
import pandas as pd
import numpy as np
import re

# ---------------------------------------------------------
# 0) Safety checks
# ---------------------------------------------------------

assert "rag_examples_df" in globals(), (
    "rag_examples_df غير معرّف. "
    "شغّل الخلية التي تولّد أمثلة final_rag_retrieve() أولاً."
)

df_raw = rag_examples_df.copy()

print("Available columns in rag_examples_df:")
print(list(df_raw.columns))


# ---------------------------------------------------------
# 1) Helper functions
# ---------------------------------------------------------

def parse_bool(value):
    """
    Convert different boolean-like values into True / False / None.
    """
    if isinstance(value, (bool, np.bool_)):
        return bool(value)

    if pd.isna(value):
        return None

    v = str(value).strip().lower()

    if v in ["true", "1", "yes", "y", "موثوق"]:
        return True

    if v in ["false", "0", "no", "n", "غير موثوق"]:
        return False

    return None


def clean_numeric_article_number(value):
    """
    Clean article number display:
    53.0 -> 53
    nan -> empty
    """
    if pd.isna(value):
        return ""

    s = str(value).strip()

    if s == "" or s.lower() in ["nan", "none", "null", "na", "n/a"]:
        return ""

    try:
        f = float(s)
        if f.is_integer():
            return str(int(f))
        return str(f)
    except Exception:
        return s


def clean_title(value):
    """
    Clean article title display.
    """
    if pd.isna(value):
        return ""

    s = str(value).strip()

    if s == "" or s.lower() in ["nan", "none", "null", "na", "n/a"]:
        return ""

    return s


def strip_ids_and_metadata(text):
    """
    Remove common technical metadata from retrieved context.
    """
    if pd.isna(text):
        return ""

    t = str(text)

    # Normalize spacing
    t = t.replace("\n", " ")
    t = re.sub(r"\s+", " ", t).strip()

    # Remove technical IDs
    t = re.sub(r"\bSTRUCT_\d+\b", " ", t, flags=re.IGNORECASE)
    t = re.sub(r"\bTRUCT_\d+\b", " ", t, flags=re.IGNORECASE)
    t = re.sub(r"\bFAQ_[A-Z_]*\d+\b", " ", t, flags=re.IGNORECASE)
    t = re.sub(r"\b[a-f0-9]{24,}\b", " ", t, flags=re.IGNORECASE)

    # Remove English metadata
    english_patterns = [
        r"Rank\s*:\s*\d+\s*\|?",
        r"Source type\s*:\s*[^|]+?\|?",
        r"Score\s*:\s*[\d.]+\s*\|?",
        r"Chunk ID\s*:\s*[^|]+?\|?",
        r"Parent document ID\s*:\s*[^|]+?\|?",
        r"Document ID\s*:\s*[^|]+?\|?",
        r"Article number\s*:\s*[\d.nanNaN]+\s*\|?",
        r"Article title\s*:\s*[^|]+?\|?",
        r"legal_article\s+",
        r"\bfaq\b\s+",
    ]

    for p in english_patterns:
        t = re.sub(p, " ", t, flags=re.IGNORECASE)

    # Remove Arabic metadata labels, but keep useful legal/FAQ content
    arabic_patterns = [
        r"التصنيف\s*[:：]\s*[^:：]{0,120}",
        r"المصدر\s*[:：]\s*[^:：]{0,120}",
        r"الموقع الفرعي\s*[:：]\s*[^:：]{0,120}",
        r"القسم\s*[:：]\s*[^:：]{0,120}",
        r"الباب\s*[:：]\s*[^:：]{0,120}",
        r"الفصل\s*[:：]\s*[^:：]{0,120}",
        r"عنوان المادة\s*[:：]\s*[^:：]{0,120}",
        r"رقم المادة\s*[:：]\s*[\d.]+",
        r"حالة المادة\s*[:：]\s*[^:：]{0,60}",
        r"active\s*",
    ]

    for p in arabic_patterns:
        t = re.sub(p, " ", t, flags=re.IGNORECASE)

    # Clean separators
    t = t.replace("|", " ")
    t = re.sub(r"\s+", " ", t).strip()

    return t


def extract_useful_context(text, source_type="", max_chars=520):
    """
    Extract useful LLM context:
    - For legal articles: prefer text after النص / النص القانوني
    - For FAQ: prefer text after الإجابة
    - Otherwise clean metadata and return readable preview
    """
    if pd.isna(text):
        return ""

    original = str(text).replace("\n", " ")
    original = re.sub(r"\s+", " ", original).strip()

    source_type = str(source_type).strip().lower()

    candidate = original

    # Prefer legal article body
    legal_patterns = [
        r"النص\s+القانوني\s*[:：]\s*(.+)",
        r"النص\s*[:：]\s*(.+)",
    ]

    # Prefer FAQ answer body
    faq_patterns = [
        r"الإجابة\s*[:：]\s*(.+)",
        r"الجواب\s*[:：]\s*(.+)",
    ]

    extracted = ""

    if source_type == "legal_article":
        for p in legal_patterns:
            m = re.search(p, original, flags=re.IGNORECASE)
            if m:
                extracted = m.group(1)
                break

    elif source_type == "faq":
        for p in faq_patterns:
            m = re.search(p, original, flags=re.IGNORECASE)
            if m:
                extracted = m.group(1)
                break

    # If source-specific extraction failed, try both
    if not extracted:
        for p in legal_patterns + faq_patterns:
            m = re.search(p, original, flags=re.IGNORECASE)
            if m:
                extracted = m.group(1)
                break

    if extracted:
        candidate = extracted

    cleaned = strip_ids_and_metadata(candidate)

    if not cleaned:
        cleaned = strip_ids_and_metadata(original)

    if not cleaned:
        return "لا توجد معاينة نصية واضحة للسياق."

    if source_type == "legal_article":
        prefix = "مقتطف من مادة قانونية: "
    elif source_type == "faq":
        prefix = "مقتطف من FAQ رسمي: "
    else:
        prefix = "مقتطف من السياق المسترجع: "

    cleaned = prefix + cleaned

    if len(cleaned) > max_chars:
        cleaned = cleaned[:max_chars].strip() + "..."

    return cleaned


def build_context_preview(row):
    """
    Build display context:
    - If unreliable: do not show retrieved context.
    - If reliable: show cleaned useful context for LLM.
    """
    reliable = parse_bool(row.get("is_retrieval_reliable", None))

    if reliable is False:
        return "غير موثوق: لن يتم تمرير هذا السياق إلى نموذج اللغة الكبير."

    source_type = row.get("top1_source_type", "")

    # context_for_llm is the real final context before LLM.
    priority_cols = [
        "context_for_llm",
        "retrieved_context",
        "top1_text",
        "top1_chunk_text",
        "context",
        "context_preview"
    ]

    for col in priority_cols:
        if col in row.index:
            value = row.get(col, "")
            if pd.notna(value) and str(value).strip():
                return extract_useful_context(
                    value,
                    source_type=source_type,
                    max_chars=520
                )

    return "لا يوجد سياق مسترجع متاح للعرض."


def build_reliability_status(row):
    reliable = parse_bool(row.get("is_retrieval_reliable", None))

    if reliable is True:
        return "موثوق"

    if reliable is False:
        return "غير موثوق"

    return ""


def build_source_note(row):
    reliable = parse_bool(row.get("is_retrieval_reliable", None))
    source_type = str(row.get("top1_source_type", "")).strip().lower()

    if reliable is False:
        return "محجوب بسبب انخفاض درجة الموثوقية."

    if source_type == "legal_article":
        return "مصدر قانوني مباشر مع رقم مادة."

    if source_type == "faq":
        return "مصدر FAQ رسمي. لا يوجد رقم مادة مباشر."

    return "مصدر غير محدد."


# ---------------------------------------------------------
# 2) Build clean display dataframe
# ---------------------------------------------------------

df = df_raw.copy()

df["reliability_status"] = df.apply(build_reliability_status, axis=1)
df["context_preview_clean"] = df.apply(build_context_preview, axis=1)
df["source_note"] = df.apply(build_source_note, axis=1)

if "top1_article_number" in df.columns:
    df["top1_article_number"] = df["top1_article_number"].apply(clean_numeric_article_number)

if "top1_article_title" in df.columns:
    df["top1_article_title"] = df["top1_article_title"].apply(clean_title)

if "top1_source_type" in df.columns:
    df["top1_source_type"] = df["top1_source_type"].fillna("").astype(str).replace("nan", "")

if "top1_backend" in df.columns:
    df["top1_backend"] = df["top1_backend"].fillna("").astype(str).replace("nan", "")


wanted_cols = [
    "query",
    "reliability_status",
    "top1_score",
    "top1_source_type",
    "top1_article_number",
    "top1_article_title",
    "source_note",
    "top1_backend",
    "latency_sec",
    "context_preview_clean",
]

present_cols = [c for c in wanted_cols if c in df.columns]
df_display = df[present_cols].copy()

rename_map = {
    "query": "السؤال",
    "reliability_status": "حالة الاسترجاع",
    "top1_score": "درجة أول نتيجة",
    "top1_source_type": "نوع المصدر",
    "top1_article_number": "رقم المادة",
    "top1_article_title": "عنوان المادة",
    "source_note": "ملاحظة المصدر",
    "top1_backend": "طريقة الاسترجاع",
    "latency_sec": "زمن الاسترجاع بالثواني",
    "context_preview_clean": "معاينة السياق المسترجع للـ LLM",
}

df_display = df_display.rename(columns={k: v for k, v in rename_map.items() if k in df_display.columns})


# ---------------------------------------------------------
# 3) Header
# ---------------------------------------------------------

display(HTML("""
<div dir="rtl" style="
    text-align:right;
    font-family:Tahoma, Arial;
    line-height:2;
    font-size:17px;
    background:#f8fbfd;
    border-right:5px solid #1f4e79;
    padding:14px 18px;
    border-radius:8px;
    margin-bottom:14px;">
<h3>اختبار طبقة الاسترجاع النهائية قبل مرحلة LLM</h3>
<p>
يعرض هذا الجدول أمثلة لاختبار دالة
<span dir="ltr" style="display:inline-block;">final_rag_retrieve()</span>
قبل مرحلة توليد الإجابة. تم تنظيف عمود السياق بحيث يعرض مقتطفًا مفهومًا من المادة القانونية أو FAQ الرسمي،
مع منع تمرير السياق إلى نموذج اللغة الكبير عندما تكون درجة الاسترجاع غير موثوقة.
</p>
</div>
"""))


# ---------------------------------------------------------
# 4) Styling helpers
# ---------------------------------------------------------

def color_reliability(v):
    if v == "موثوق":
        return "background-color: #DFF2E1; font-weight: bold; color: #1F6E43;"
    if v == "غير موثوق":
        return "background-color: #F8D7DA; font-weight: bold; color: #842029;"
    return ""


def color_score(v):
    try:
        score = float(v)
    except Exception:
        return ""

    if "suggested_reliability_threshold" in globals():
        thr = float(suggested_reliability_threshold)
    elif "score_calibration" in globals() and "suggested_reliability_threshold" in score_calibration.columns:
        thr = float(score_calibration["suggested_reliability_threshold"].iloc[0])
    else:
        thr = 0.4983

    if score >= thr:
        return "background-color: #DFF2E1;"
    return "background-color: #F8D7DA;"


def color_source_note(v):
    s = str(v)
    if "مصدر قانوني" in s:
        return "background-color: #E7F1FF; color: #084298;"
    if "FAQ" in s:
        return "background-color: #FFF3CD; color: #664D03;"
    if "محجوب" in s:
        return "background-color: #F8D7DA; color: #842029;"
    return ""


# ---------------------------------------------------------
# 5) Display professional styled table
# ---------------------------------------------------------

sty = df_display.style.set_caption("Final RAG Retriever Examples for Stage 04 LLM")

format_dict = {}

if "درجة أول نتيجة" in df_display.columns:
    format_dict["درجة أول نتيجة"] = "{:.4f}"

if "زمن الاسترجاع بالثواني" in df_display.columns:
    format_dict["زمن الاسترجاع بالثواني"] = "{:.4f}"

sty = sty.format(format_dict)

sty = sty.set_table_styles([
    {
        "selector": "caption",
        "props": [
            ("caption-side", "top"),
            ("font-size", "18px"),
            ("font-weight", "bold"),
            ("text-align", "center"),
            ("padding", "10px"),
            ("color", "#17324d"),
        ],
    },
    {
        "selector": "th",
        "props": [
            ("background-color", "#F2F2F2"),
            ("font-weight", "bold"),
            ("text-align", "center"),
            ("border", "1px solid #CCCCCC"),
            ("padding", "8px"),
            ("white-space", "nowrap"),
        ],
    },
    {
        "selector": "td",
        "props": [
            ("text-align", "center"),
            ("border", "1px solid #DDDDDD"),
            ("padding", "8px"),
            ("vertical-align", "top"),
        ],
    },
    {
        "selector": "table",
        "props": [
            ("border-collapse", "collapse"),
            ("width", "100%"),
            ("font-size", "14px"),
            ("font-family", "Tahoma, Arial"),
        ],
    },
])

# Apply conditional colors with compatibility
if "حالة الاسترجاع" in df_display.columns:
    if hasattr(sty, "map"):
        sty = sty.map(color_reliability, subset=["حالة الاسترجاع"])
    else:
        sty = sty.applymap(color_reliability, subset=["حالة الاسترجاع"])

if "درجة أول نتيجة" in df_display.columns:
    if hasattr(sty, "map"):
        sty = sty.map(color_score, subset=["درجة أول نتيجة"])
    else:
        sty = sty.applymap(color_score, subset=["درجة أول نتيجة"])

if "ملاحظة المصدر" in df_display.columns:
    if hasattr(sty, "map"):
        sty = sty.map(color_source_note, subset=["ملاحظة المصدر"])
    else:
        sty = sty.applymap(color_source_note, subset=["ملاحظة المصدر"])

# Arabic text columns
rtl_cols = [
    c for c in [
        "السؤال",
        "عنوان المادة",
        "ملاحظة المصدر",
        "معاينة السياق المسترجع للـ LLM"
    ]
    if c in df_display.columns
]

if rtl_cols:
    sty = sty.set_properties(
        subset=rtl_cols,
        **{
            "text-align": "right",
            "direction": "rtl",
            "white-space": "normal",
            "line-height": "1.8",
        }
    )

# Wider context column
if "معاينة السياق المسترجع للـ LLM" in df_display.columns:
    sty = sty.set_properties(
        subset=["معاينة السياق المسترجع للـ LLM"],
        **{
            "max-width": "620px",
            "min-width": "420px",
        }
    )

# English / technical columns
ltr_cols = [
    c for c in ["نوع المصدر", "طريقة الاسترجاع"]
    if c in df_display.columns
]

if ltr_cols:
    sty = sty.set_properties(
        subset=ltr_cols,
        **{
            "direction": "ltr",
            "text-align": "center",
            "white-space": "nowrap",
        }
    )

display(sty)


# ---------------------------------------------------------
# 6) Optional save outputs
# ---------------------------------------------------------

if "STAGE03_DIR" in globals():
    output_csv = STAGE03_DIR / "stage04_final_rag_retriever_examples_clean.csv"
    output_xlsx = STAGE03_DIR / "stage04_final_rag_retriever_examples_clean.xlsx"

    df_display.to_csv(output_csv, index=False, encoding="utf-8-sig")

    try:
        with pd.ExcelWriter(output_xlsx, engine="openpyxl") as writer:
            df_display.to_excel(writer, index=False, sheet_name="final_retriever_examples")
        print("Saved Excel:", output_xlsx)
    except Exception as e:
        print("Excel save skipped:", e)

    print("Saved CSV:", output_csv)

Available columns in rag_examples_df:
['query', 'is_retrieval_reliable', 'suggested_reliability_threshold', 'top1_score', 'top1_source_type', 'top1_article_number', 'top1_article_title', 'top1_chunk_id', 'top1_parent_document_id', 'top1_backend', 'latency_sec', 'context_preview', 'context_for_llm', 'best_chunking_strategy', 'best_embedding_model', 'best_retriever', 'best_alpha']


,السؤال,حالة الاسترجاع,درجة أول نتيجة,نوع المصدر,رقم المادة,عنوان المادة,ملاحظة المصدر,طريقة الاسترجاع,زمن الاسترجاع بالثواني,معاينة السياق المسترجع للـ LLM
0,كم مدة فترة التجربة في نظام العمل السعودي؟,موثوق,0.9709,legal_article,53,المادة الثالثة والخمسون :,مصدر قانوني مباشر مع رقم مادة.,chromadb+bm25+reranker,0.8873,مقتطف من مادة قانونية: علاقات العمل - الفصل الأول - المادة 53 : : إذا كان العامل خاضعاً للتجربة، وجب النص على ذلك صراحة في عقد العمل، وتحديد مدتها بوضوح، على ألا يزيد مجموع المدة في جميع الأحوال على (مائة وثمانين) يوماً. وتبين اللائحة الأحكام المتصلة بذلك بما في ذلك ما يتعلق بالإجازات التي لا تدخل في حساب المدة. ولكل من الطرفين الحق في إنهاء العقد خلال هذه المدة. --- egal_article لمادة الرابعة والخمسون: التصنيف القانوني: علاقات العمل : الفصل الأول رقم المادة: الرابعة والخمسون : النص القانوني: علاقات العمل - الفصل ا...
1,متى يستحق العامل مكافأة نهاية الخدمة؟,موثوق,0.8435,legal_article,85,المادة الخامسة والثمانون:,مصدر قانوني مباشر مع رقم مادة.,chromadb+bm25+reranker,0.3034,مقتطف من مادة قانونية: علاقات العمل - الفصل الرابع - المادة 85 : : إذا كان انتهاء علاقة العمل بسبب استقالة العامل يستحق في هذه الحالة ثلث المكافأة بعد خدمة لا تقل مدتها عن سنتين متتاليتين ، ولا تزيد على خمس سنوات ، ويستحق ثلثيها إذا زادت مدة خدمته على خمس سنوات متتالية ولم تبلغ عشر سنوات ويستحق المكافأة كاملة إذا بلغت مدة خدمته عشر سنوات فأكثر . --- aq an : قطاع الخدمة المدنية (العام) : ما مدى امكانية صرف مكافأة نهاية الخدمة المنصوص عليها بالمادة (53) من لائحة الحقوق والمزايا المالية للعامل المعين على بند الأجور عن...
2,كم عدد ساعات العمل في نظام العمل السعودي؟,موثوق,0.9311,legal_article,99,المادة التاسعة والتسعون:,مصدر قانوني مباشر مع رقم مادة.,chromadb+bm25+reranker,0.6600,مقتطف من مادة قانونية: شروط العمل وظروفه - الفصل الثاني - المادة 99 : : يجوز زيادة ساعات العمل المنصوص عليها في المادة الثامنة والتسعين من هذا النظام إلى تسع ساعات في اليوم الواحد لبعض فئات العمال ، أو في بعض الصناعات والأعمال التي لا يشتغل فيها العامل بصفة مستمرة . كما يجوز تخفيضها إلى سبع ساعات في اليوم الواحد لبعض فئات العمال أو في بعض الصناعات والأعمال الخطرة أو الضارة . وتحدد فئات العمال والصناعات والأعمال المشار إليها بقرار من الوزير . --- aq an : قطاع العمل : كم عدد ساعات العمل ؟ النص المسترجع: : كم عدد ساعا...
3,هل يحق لصاحب العمل نقل العامل من فرع إلى فرع؟,موثوق,0.9201,faq,,,مصدر FAQ رسمي. لا يوجد رقم مادة مباشر.,chromadb+bm25+reranker,0.3631,مقتطف من FAQ رسمي: لا يجوز تكليف العامل بعمل يختلف اختلافًا جوهريًّا عن العمل المتفق عليه بغير موافقته الكتابية، إلا في حالات الضرورة التي قد تقتضيها ظروف عارضة، ولمدة لا تتجاوز ثلاثين يومًا في السنة. لصاحب العمل - في حالات الضرورة التي قد تقتضيها ظروف عارضة ولمدة لا تتجاوز ثلاثين يوماً في السنة – تكليف العامل بعمل في مكان يختلف عن المكان المتفق عليه دون اشتراط موافقته، على أن يتحمل صاحب العمل تكاليف انتقال العامل وإقامته خلال تلك المدة. لا يجوز لصاحب العمل أن ينقل العامل بغير موافقته -كتابة- من مكان عمله الأصلي إل...
4,ما مدة الإجازة السنوية للعامل؟,موثوق,0.9963,faq,,,مصدر FAQ رسمي. لا يوجد رقم مادة مباشر.,chromadb+bm25+reranker,0.6809,مقتطف من FAQ رسمي: يستحق العامل عن كل عام إجازة سنوية لا تقل مدتها عن واحد وعشرين يومًا، تُزاد إلى مدة لا تقل عن ثلاثين يومًا إذا أمضى العامل في خدمة صاحب العمل خمس سنوات متصلة، وتكون الإجازة بأجر يدفع مقدمًا. يجب أن يتمتع العامل بإجازته في سنة استحقاقها، ولا يجوز النزول عنها، أو أن يتقاضى بدلًا نقديًّا عوضًا عن الحصول عليها أثناء خدمته، ولصاحب العمل أن يحدد مواعيد هذه الإجازات وفقًا لمقتضيات العمل، أو يمنحها بالتناوب لكي يؤمن سير عمله، وعليه إشعار العامل بالميعاد المحدد لتمتعه بالإجازة بوقت كافٍ لا يقل عن ثلاثين ي...
5,كيف أقدم شكوى على صاحب العمل؟,موثوق,0.7776,faq,,,مصدر FAQ رسمي. لا يوجد رقم مادة مباشر.,chromadb+bm25+reranker,0.3624,مقتطف من FAQ رسمي: من أنواع الشكاوى: تأخير رواتب. التكليف بعمل مختلف عن طبيعة العمل المتفق عليه في العقد الموقع بين الطرفين سوء المعاملة عدم وجود سكن للموظف (في حال لم ينص عقد العمل عليه أو لم تكن المنشاة من ضمن شركات تأجير العمالة فأنها لا تلزم فيه كونه أحد البدلات المتفق عليها) مخالفة أحد شروط العقد --- aq an : قطاع العمل : كيف اتابع الشكوى وما هي الوثائق المطلوب

Saved Excel: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\stage04_final_rag_retriever_examples_clean.xlsx
Saved CSV: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\stage04_final_rag_retriever_examples_clean.csv


In [51]:
# =========================================================
# Stage 20 - Final Academic Retrieval Report
# =========================================================

final_report = {
    "project_dir": str(PROJECT_DIR),
    "stage03_dir": str(STAGE03_DIR),
    "evaluation_questions": int(len(df_eval_run)),
    "retrieval_experiments": int(len(experiment_configs)),
    "embedding_models_loaded": list(loaded_embedding_models.keys()),
    "reranker_loaded": reranker_model is not None,
    "chromadb_required": REQUIRE_CHROMADB,
    "chromadb_vector_dir": str(VECTOR_DIR),
    "chromadb_collections": chroma_manifest.to_dict(orient="records"),
    "best_config": best_config,
    "suggested_reliability_threshold": suggested_reliability_threshold,
    "input_safety_checks": safety_df.to_dict(orient="records"),
}

with open(STAGE03_DIR / "stage03_final_retrieval_report.json", "w", encoding="utf-8") as f:
    json.dump(final_report, f, ensure_ascii=False, indent=2)

final_summary_table = pd.DataFrame([{
    "evaluation_questions": int(len(df_eval_run)),
    "experiment_configurations": int(len(experiment_configs)),
    "embedding_models_loaded": ", ".join(loaded_embedding_models.keys()),
    "reranker_loaded": bool(reranker_model is not None),
    "chromadb_used": True,
    "chroma_collections": int(len(chroma_manifest)),
    "best_chunking_strategy": best_config.get("chunking_strategy"),
    "best_embedding_model": best_config.get("embedding_model"),
    "best_retriever": best_config.get("retriever"),
    "best_alpha": best_config.get("alpha"),
    "best_recall_at_1": best_config.get("recall_at_1"),
    "best_recall_at_3": best_config.get("recall_at_3"),
    "best_recall_at_5": best_config.get("recall_at_5"),
    "best_mrr": best_config.get("mrr"),
    "best_ndcg_at_5": best_config.get("ndcg_at_5"),
    "best_avg_latency_sec": best_config.get("avg_latency_sec"),
    "suggested_reliability_threshold": suggested_reliability_threshold,
}])

final_summary_table.to_csv(STAGE03_DIR / "stage03_final_summary_table.csv", index=False, encoding="utf-8-sig")
final_summary_table.to_excel(STAGE03_DIR / "stage03_final_summary_table.xlsx", index=False)

display(final_summary_table)
print("Final retrieval report saved:", STAGE03_DIR / "stage03_final_retrieval_report.json")

,evaluation_questions,experiment_configurations,embedding_models_loaded,reranker_loaded,chromadb_used,chroma_collections,best_chunking_strategy,best_embedding_model,best_retriever,best_alpha,best_recall_at_1,best_recall_at_3,best_recall_at_5,best_mrr,best_ndcg_at_5,best_avg_latency_sec,suggested_reliability_threshold
0,235,22,"e5_large, bge_m3",True,True,4,structural,bge_m3,hybrid_reranker,0.8,0.7617,0.9277,0.9745,0.8472,0.8773,0.407,0.4983


Final retrieval report saved: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\stage03_final_retrieval_report.json


In [49]:
# =========================================================
# Stage 21 - Voice Router + RAG Integration Test
# ربط طبقة الأسئلة العامة مع RAG استعداداً للمرحلة الرابعة
# =========================================================

import re
import pandas as pd
from IPython.display import display, HTML

# ---------------------------------------------------------
# Safety check: final_rag_retrieve must already exist
# ---------------------------------------------------------

assert "final_rag_retrieve" in globals(), (
    "final_rag_retrieve is not defined. "
    "Run Stage 19 - Final RAG Retriever Packaging for Stage 04 LLM first."
)

# ---------------------------------------------------------
# 1) Arabic normalization for routing only
# ---------------------------------------------------------

def normalize_arabic_query(text):
    """
    Normalize Arabic text for routing decisions only.
    This is not used for final answer generation.
    """
    text = str(text).strip().lower()

    # Remove Arabic diacritics and tatweel
    text = re.sub(r"[\u064B-\u065F\u0670]", "", text)
    text = text.replace("ـ", "")

    # Normalize common Arabic letter variations
    text = re.sub("[إأآا]", "ا", text)
    text = text.replace("ى", "ي")
    text = text.replace("ة", "ه")
    text = text.replace("ؤ", "و")
    text = text.replace("ئ", "ي")

    # Remove punctuation except Arabic/English letters and numbers
    text = re.sub(r"[^\w\s\u0600-\u06FF]", " ", text)

    # Normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text


def contains_any(text, phrases):
    normalized_phrases = [normalize_arabic_query(p) for p in phrases]
    return any(p in text for p in normalized_phrases)


# ---------------------------------------------------------
# 2) Intent patterns
# ---------------------------------------------------------

GREETING_PATTERNS = [
    "السلام عليكم", "سلام عليكم", "السلام", "سلام",
    "هلا", "اهلا", "مرحبا", "حياك", "صباح الخير", "مساء الخير",
    "كيف الحال", "كيفك", "كيف حالك", "وش اخبارك", "ايش اخبارك",
    "شلونك", "عامل ايه", "اخبارك"
]

IDENTITY_PATTERNS = [
    "انت مين", "مين انت", "من انت", "مين معي", "مع مين",
    "ايش اسمك", "وش اسمك", "اسمك", "اكلم مين", "اتكلم مع مين"
]

CAPABILITY_PATTERNS = [
    "وش تقدر تساعدني", "ايش تقدر تساعدني", "كيف تساعدني",
    "وش خدماتك", "ايش خدماتك", "بماذا تساعدني",
    "تقدر تساعدني في ايش", "ايش تقدر تسوي", "وش تقدر تسوي"
]

THANKS_PATTERNS = [
    "شكرا", "شكرا لك", "يعطيك العافيه", "مشكور", "تسلم", "الله يعطيك العافيه"
]

GOODBYE_PATTERNS = [
    "مع السلامه", "باي", "وداعا", "شكرا خلاص", "خلاص شكرا"
]

OUT_OF_SCOPE_PATTERNS = [
    "الطقس", "درجة الحراره", "مطعم", "فندق", "رحله", "سفر",
    "سعر العقار", "سعر البيت", "اسعار العقار", "شراء بيت",
    "الاسهم", "سهم", "بيتكوين", "عملات رقميه",
    "طلاق", "زواج", "ميراث", "جنائي", "قضيه جنائيه",
    "علاج", "دواء", "اعراض", "طبيب"
]

LEGAL_LABOR_PATTERNS = [
    "نظام العمل", "مكتب العمل", "وزاره الموارد", "الموارد البشريه",
    "عامل", "العامل", "صاحب العمل", "منشاه", "الشركه",
    "عقد العمل", "العقد", "راتب", "اجر", "الاجر",
    "مكافاه", "نهايه الخدمه", "فتره التجربه", "تجربه",
    "استقاله", "فصل", "انهاء العقد", "ساعات العمل",
    "دوام", "اجازه", "اجازات", "نقل العامل", "اصابه عمل",
    "تعويض", "انذار", "غياب", "تأخير", "تدريب", "سعوده"
]

# ---------------------------------------------------------
# 3) Router function
# ---------------------------------------------------------

def route_voice_query(user_query):
    """
    Route Arabic voice-agent query before RAG and LLM.

    Returns a routing decision:
    - direct_response for greetings, identity, thanks, and clear out-of-scope
    - rag_retrieval for legal or possibly legal questions
    """

    raw_query = str(user_query).strip()
    q = normalize_arabic_query(raw_query)

    if not q:
        return {
            "intent": "empty_or_unclear",
            "route": "ask_clarification",
            "needs_rag": False,
            "direct_response": "عذراً، لم أسمع السؤال بوضوح. ممكن تعيد صياغة السؤال؟",
            "reason": "Empty query after normalization."
        }

    if contains_any(q, GREETING_PATTERNS):
        return {
            "intent": "greeting_or_small_talk",
            "route": "direct_response",
            "needs_rag": False,
            "direct_response": "أهلاً وسهلاً، معك المساعد الذكي لنظام العمل السعودي. كيف أقدر أساعدك؟",
            "reason": "Greeting or small-talk detected."
        }

    if contains_any(q, IDENTITY_PATTERNS):
        return {
            "intent": "agent_identity",
            "route": "direct_response",
            "needs_rag": False,
            "direct_response": "معك وكيل صوتي ذكي مخصص للإجابة عن الاستفسارات المتعلقة بنظام العمل السعودي.",
            "reason": "Agent identity question detected."
        }

    if contains_any(q, CAPABILITY_PATTERNS):
        return {
            "intent": "agent_capability",
            "route": "direct_response",
            "needs_rag": False,
            "direct_response": "أقدر أساعدك في أسئلة نظام العمل السعودي مثل فترة التجربة، ساعات العمل، الإجازات، الاستقالة، ونهاية الخدمة.",
            "reason": "Capability question detected."
        }

    if contains_any(q, THANKS_PATTERNS):
        return {
            "intent": "thanks",
            "route": "direct_response",
            "needs_rag": False,
            "direct_response": "حياك الله، سعيد بخدمتك. هل عندك استفسار آخر بخصوص نظام العمل السعودي؟",
            "reason": "Thanks detected."
        }

    if contains_any(q, GOODBYE_PATTERNS):
        return {
            "intent": "goodbye",
            "route": "direct_response",
            "needs_rag": False,
            "direct_response": "مع السلامة، سعيد بخدمتك.",
            "reason": "Goodbye detected."
        }

    if contains_any(q, OUT_OF_SCOPE_PATTERNS):
        return {
            "intent": "out_of_scope",
            "route": "safe_rejection",
            "needs_rag": False,
            "direct_response": "عذراً، هذا السؤال خارج نطاق اختصاصي. أستطيع مساعدتك فقط في الاستفسارات المتعلقة بنظام العمل السعودي.",
            "reason": "Clearly outside Saudi labor law scope."
        }

    if contains_any(q, LEGAL_LABOR_PATTERNS):
        return {
            "intent": "labor_law_question",
            "route": "rag_retrieval",
            "needs_rag": True,
            "direct_response": None,
            "reason": "Labor-law keyword detected; send to RAG retrieval."
        }

    return {
        "intent": "possible_labor_question",
        "route": "rag_retrieval_with_reliability_gate",
        "needs_rag": True,
        "direct_response": None,
        "reason": "Not small-talk and not clearly out-of-scope; allow RAG then apply reliability gate."
    }


# ---------------------------------------------------------
# 4) Route then retrieve bridge for Stage 04 LLM
# ---------------------------------------------------------

def route_then_retrieve_for_llm(user_query, top_k=5):
    """
    Bridge function between Stage 03 RAG and Stage 04 LLM.

    If the query is small talk, identity, thanks, or clearly out-of-scope:
        return a direct response without RAG and without LLM.

    If the query is legal or possibly related to Saudi labor law:
        run final_rag_retrieve and prepare retrieved context for the LLM.
    """

    routing_decision = route_voice_query(user_query)

    # Direct response path: no RAG, no LLM needed
    if not routing_decision["needs_rag"]:
        return {
            "user_query": user_query,
            "intent": routing_decision["intent"],
            "route": routing_decision["route"],
            "needs_rag": False,
            "direct_answer": routing_decision["direct_response"],
            "retrieval_result": None,
            "send_to_llm": False,
            "reason": routing_decision["reason"],
            "is_retrieval_reliable": None,
            "top1_score": None,
            "latency_sec": None,
        }

      # RAG path: retrieve context for LLM
    retrieval_result = final_rag_retrieve(user_query, top_k=top_k)

    is_reliable = bool(retrieval_result.get("is_retrieval_reliable", False))

    # If retrieval is not reliable, do not send to LLM
    if not is_reliable:
        return {
            "user_query": user_query,
            "intent": routing_decision["intent"],
            "route": "rag_retrieval_blocked_by_reliability_gate",
            "needs_rag": True,
            "direct_answer": "عذرًا، لم أجد نصًا موثوقًا في نظام العمل السعودي يدعم إجابة دقيقة على هذا السؤال. يمكنك إعادة صياغة السؤال أو طرح سؤال متعلق مباشرة بنظام العمل السعودي، أو اضغط رقم 1 لتحويلك إلى موظف خدمة العملاء..",
            "retrieval_result": retrieval_result,
            "send_to_llm": False,
            "reason": "RAG retrieval score is below the reliability threshold.",
            "is_retrieval_reliable": False,
            "top1_score": retrieval_result.get("top1_score"),
            "latency_sec": retrieval_result.get("latency_sec"),
        }

    return {
        "user_query": user_query,
        "intent": routing_decision["intent"],
        "route": routing_decision["route"],
        "needs_rag": True,
        "direct_answer": None,
        "retrieval_result": retrieval_result,
        "send_to_llm": True,
        "reason": routing_decision["reason"],
        "is_retrieval_reliable": True,
        "top1_score": retrieval_result.get("top1_score"),
        "latency_sec": retrieval_result.get("latency_sec"),
    }


# ---------------------------------------------------------
# 5) Quick integration test
# ---------------------------------------------------------

integration_test_queries = [
    "السلام عليكم",
    "كيف الحال؟",
    "مين معي؟",
    "وش تقدر تساعدني؟",
    "كم مدة فترة التجربة؟",
    "متى يستحق العامل مكافأة نهاية الخدمة؟",
    "كم عدد ساعات العمل؟",
    "كم سعر العقار في الرياض؟",
    "ما حكم الطلاق؟",
    "هل يحق للعامل الحصول على سيارة من الشركة؟"
]

integration_rows = []

for q in integration_test_queries:
    result = route_then_retrieve_for_llm(q, top_k=5)

    integration_rows.append({
        "user_query": q,
        "intent": result["intent"],
        "route": result["route"],
        "needs_rag": result["needs_rag"],
        "send_to_llm": result["send_to_llm"],
        "is_retrieval_reliable": result["is_retrieval_reliable"],
        "top1_score": result["top1_score"],
        "latency_sec": result["latency_sec"],
        "direct_answer": result["direct_answer"],
        "reason": result["reason"],
    })

integration_test_df = pd.DataFrame(integration_rows)

display(HTML("""
<div dir="rtl" style="text-align:right; line-height:2; font-size:17px;">
<h3>اختبار ربط Router مع طبقة RAG قبل مرحلة LLM</h3>
<p>
يوضح الجدول التالي كيف يقرر النظام هل يرد مباشرة على السؤال، أو يرسل السؤال إلى طبقة الاسترجاع،
أو يمنع تمريره إلى نموذج اللغة إذا كانت نتيجة الاسترجاع غير موثوقة.
</p>
</div>
"""))

# =========================================================
# Professional Display for Voice Router + RAG Integration
# عرض احترافي لاختبار Router + RAG
# =========================================================

from IPython.display import display, HTML
import pandas as pd
import numpy as np

integration_display = integration_test_df.copy()

# ---------------------------------------------------------
# 1) تحويل القيم التقنية إلى قيم مفهومة للعرض
# ---------------------------------------------------------

integration_display["needs_rag_label"] = integration_display["needs_rag"].map({
    True: "يحتاج RAG",
    False: "لا يحتاج RAG"
})

integration_display["send_to_llm_label"] = integration_display["send_to_llm"].map({
    True: "يُرسل إلى LLM",
    False: "لا يُرسل إلى LLM"
})

integration_display["reliability_label"] = integration_display["is_retrieval_reliable"].map({
    True: "موثوق",
    False: "غير موثوق",
    None: "غير مطلوب"
}).fillna("غير مطلوب")

# اختصار الرد المباشر للعرض فقط
integration_display["direct_answer_preview"] = (
    integration_display["direct_answer"]
    .fillna("—")
    .astype(str)
    .str.replace("\n", " ", regex=False)
    .str.slice(0, 120)
)

# اختصار السبب للعرض فقط
integration_display["reason_preview"] = (
    integration_display["reason"]
    .fillna("—")
    .astype(str)
    .str.replace("\n", " ", regex=False)
    .str.slice(0, 110)
)

# ---------------------------------------------------------
# 2) ترتيب الأعمدة للعرض الأكاديمي
# ---------------------------------------------------------

display_cols = [
    "user_query",
    "intent",
    "route",
    "needs_rag_label",
    "send_to_llm_label",
    "reliability_label",
    "top1_score",
    "latency_sec",
    "direct_answer_preview",
    "reason_preview",
]

integration_display = integration_display[display_cols]

integration_display = integration_display.rename(columns={
    "user_query": "سؤال المستخدم",
    "intent": "نوع النية",
    "route": "مسار المعالجة",
    "needs_rag_label": "هل يحتاج RAG؟",
    "send_to_llm_label": "قرار الإرسال إلى LLM",
    "reliability_label": "موثوقية الاسترجاع",
    "top1_score": "درجة أول نتيجة",
    "latency_sec": "زمن الاسترجاع",
    "direct_answer_preview": "الرد المباشر",
    "reason_preview": "سبب القرار",
})

# ---------------------------------------------------------
# 3) عنوان توضيحي قبل الجدول
# ---------------------------------------------------------

display(HTML("""
<div dir="rtl" style="text-align:right; line-height:2; font-size:17px;">
<h3>اختبار تكامل طبقة Router مع RAG قبل مرحلة LLM</h3>
<p>
يعرض الجدول التالي كيف يتعامل الوكيل الصوتي مع أنواع مختلفة من الأسئلة:
التحيات والأسئلة العامة يتم الرد عليها مباشرة، والأسئلة القانونية الموثوقة يتم تمريرها إلى نموذج اللغة،
أما الأسئلة الخارجة عن النطاق أو غير الموثوقة فيتم منعها من الوصول إلى نموذج اللغة.
</p>
</div>
"""))

# ---------------------------------------------------------
# 4) تنسيق احترافي للجدول
# ---------------------------------------------------------

def style_reliability(v):
    if v == "موثوق":
        return "background-color: #DFF2E1; color: #1F6E43; font-weight: bold;"
    if v == "غير موثوق":
        return "background-color: #F8D7DA; color: #842029; font-weight: bold;"
    return "background-color: #F2F2F2; color: #555555;"

def style_llm_decision(v):
    if v == "يُرسل إلى LLM":
        return "background-color: #DFF2E1; color: #1F6E43; font-weight: bold;"
    return "background-color: #FFF3CD; color: #664D03; font-weight: bold;"

def style_rag_need(v):
    if v == "يحتاج RAG":
        return "background-color: #D9EAF7; color: #084C7C; font-weight: bold;"
    return "background-color: #F2F2F2; color: #555555;"

def style_score(v):
    try:
        if pd.isna(v):
            return "background-color: #F2F2F2; color: #777777;"
        if float(v) >= float(suggested_reliability_threshold):
            return "background-color: #DFF2E1; color: #1F6E43; font-weight: bold;"
        return "background-color: #F8D7DA; color: #842029; font-weight: bold;"
    except Exception:
        return ""

styled_integration_table = (
    integration_display.style
    .set_caption("Voice Router + RAG Integration Test")
    .format({
        "درجة أول نتيجة": lambda x: "—" if pd.isna(x) else f"{float(x):.4f}",
        "زمن الاسترجاع": lambda x: "—" if pd.isna(x) else f"{float(x):.4f}",
    })
    .set_table_styles([
        {
            "selector": "caption",
            "props": [
                ("caption-side", "top"),
                ("font-size", "18px"),
                ("font-weight", "bold"),
                ("text-align", "center"),
                ("padding", "10px"),
            ],
        },
        {
            "selector": "th",
            "props": [
                ("background-color", "#F2F2F2"),
                ("font-weight", "bold"),
                ("text-align", "center"),
                ("border", "1px solid #CCCCCC"),
                ("padding", "8px"),
            ],
        },
        {
            "selector": "td",
            "props": [
                ("text-align", "center"),
                ("border", "1px solid #DDDDDD"),
                ("padding", "8px"),
                ("vertical-align", "top"),
            ],
        },
        {
            "selector": "table",
            "props": [
                ("border-collapse", "collapse"),
                ("width", "100%"),
                ("font-size", "14px"),
            ],
        },
    ])
    .set_properties(
        subset=["سؤال المستخدم", "الرد المباشر", "سبب القرار"],
        **{
            "text-align": "right",
            "direction": "rtl",
            "white-space": "normal",
            "max-width": "360px",
        }
    )
    .map(style_rag_need, subset=["هل يحتاج RAG؟"])
.map(style_llm_decision, subset=["قرار الإرسال إلى LLM"])
.map(style_reliability, subset=["موثوقية الاسترجاع"])
.map(style_score, subset=["درجة أول نتيجة"])
)

display(styled_integration_table)

,سؤال المستخدم,نوع النية,مسار المعالجة,هل يحتاج RAG؟,قرار الإرسال إلى LLM,موثوقية الاسترجاع,درجة أول نتيجة,زمن الاسترجاع,الرد المباشر,سبب القرار
0,السلام عليكم,greeting_or_small_talk,direct_response,لا يحتاج RAG,لا يُرسل إلى LLM,غير مطلوب,—,—,أهلاً وسهلاً، معك المساعد الذكي لنظام العمل السعودي. كيف أقدر أساعدك؟,Greeting or small-talk detected.
1,كيف الحال؟,greeting_or_small_talk,direct_response,لا يحتاج RAG,لا يُرسل إلى LLM,غير مطلوب,—,—,أهلاً وسهلاً، معك المساعد الذكي لنظام العمل السعودي. كيف أقدر أساعدك؟,Greeting or small-talk detected.
2,مين معي؟,agent_identity,direct_response,لا يحتاج RAG,لا يُرسل إلى LLM,غير مطلوب,—,—,معك وكيل صوتي ذكي مخصص للإجابة عن الاستفسارات المتعلقة بنظام العمل السعودي.,Agent identity question detected.
3,وش تقدر تساعدني؟,agent_capability,direct_response,لا يحتاج RAG,لا يُرسل إلى LLM,غير مطلوب,—,—,أقدر أساعدك في أسئلة نظام العمل السعودي مثل فترة التجربة، ساعات العمل، الإجازات، الاستقالة، ونهاية الخدمة.,Capability question detected.
4,كم مدة فترة التجربة؟,labor_law_question,rag_retrieval,يحتاج RAG,يُرسل إلى LLM,موثوق,0.5968,0.6236,—,Labor-law keyword detected; send to RAG retrieval.
5,متى يستحق العامل مكافأة نهاية الخدمة؟,labor_law_question,rag_retrieval,يحتاج RAG,يُرسل إلى LLM,موثوق,0.8435,0.3063,—,Labor-law keyword detected; send to RAG retrieval.
6,كم عدد ساعات العمل؟,labor_law_question,rag_retrieval,يحتاج RAG,يُرسل إلى LLM,موثوق,0.9989,0.6436,—,Labor-law keyword detected; send to RAG retrieval.
7,كم سعر العقار في الرياض؟,out_of_scope,safe_rejection,لا يحتاج RAG,لا يُرسل إلى LLM,غير مطلوب,—,—,عذراً، هذا السؤال خارج نطاق اختصاصي. أستطيع مساعدتك فقط في الاستفسارات المتعلقة بنظام العمل السعودي.,Clearly outside Saudi labor law scope.
8,ما حكم الطلاق؟,out_of_scope,safe_rejection,لا يحتاج RAG,لا يُرسل إلى LLM,غير مطلوب,—,—,عذراً، هذا السؤال خارج نطاق اختصاصي. أستطيع مساعدتك فقط في الاستفسارات المتعلقة بنظام العمل السعودي.,Clearly outside Saudi labor law scope.
9,هل يحق للعامل الحصول على سيارة من الشركة؟,labor_law_question,rag_retrieval_blocked_by_reliability_gate,يحتاج RAG,لا يُرسل إلى LLM,غير موثوق,0.1144,0.3588,عذرًا، لم أجد نصًا موثوقًا في نظام العمل السعودي يدعم إجابة دقيقة على هذا السؤال. يمكنك إعادة صياغة السؤال أو طرح سؤال م,RAG retrieval score is below the reliability threshold.


In [52]:
# =========================================================
# Save Function - Voice Router + RAG Integration Results
# دالة حفظ نتائج اختبار Router + RAG
# =========================================================

from pathlib import Path
import pandas as pd

def save_voice_router_rag_results(
    raw_df,
    display_df=None,
    output_dir=None,
    base_name="voice_router_rag_integration_test"
):
    """
    Save Voice Router + RAG integration test results.

    Saves:
    1. Raw full results as CSV and Excel
    2. Display/clean table as CSV and Excel
    3. Styled HTML table if display_df is available

    Parameters:
        raw_df: original full DataFrame
        display_df: cleaned/renamed display DataFrame
        output_dir: output directory
        base_name: base filename without extension
    """

    if output_dir is None:
        output_dir = STAGE03_DIR

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    # -----------------------------------------------------
    # 1) Save raw full table
    # -----------------------------------------------------

    raw_csv_path = output_dir / f"{base_name}_raw.csv"
    raw_xlsx_path = output_dir / f"{base_name}_raw.xlsx"

    raw_df.to_csv(
        raw_csv_path,
        index=False,
        encoding="utf-8-sig"
    )

    raw_df.to_excel(
        raw_xlsx_path,
        index=False
    )

    saved_files = {
        "raw_csv": str(raw_csv_path),
        "raw_excel": str(raw_xlsx_path),
    }

    # -----------------------------------------------------
    # 2) Save display table if provided
    # -----------------------------------------------------

    if display_df is not None:
        display_csv_path = output_dir / f"{base_name}_display.csv"
        display_xlsx_path = output_dir / f"{base_name}_display.xlsx"

        display_df.to_csv(
            display_csv_path,
            index=False,
            encoding="utf-8-sig"
        )

        display_df.to_excel(
            display_xlsx_path,
            index=False
        )

        saved_files["display_csv"] = str(display_csv_path)
        saved_files["display_excel"] = str(display_xlsx_path)

        # -------------------------------------------------
        # 3) Save styled HTML version
        # -------------------------------------------------

        html_path = output_dir / f"{base_name}_styled.html"

        try:
            styled_html = styled_integration_table.to_html()
            with open(html_path, "w", encoding="utf-8") as f:
                f.write(styled_html)

            saved_files["styled_html"] = str(html_path)

        except Exception as e:
            saved_files["styled_html_error"] = str(e)

    # -----------------------------------------------------
    # 4) Save manifest
    # -----------------------------------------------------

    manifest_df = pd.DataFrame(
        [{"file_type": k, "file_path": v} for k, v in saved_files.items()]
    )

    manifest_path = output_dir / f"{base_name}_save_manifest.csv"

    manifest_df.to_csv(
        manifest_path,
        index=False,
        encoding="utf-8-sig"
    )

    saved_files["manifest"] = str(manifest_path)

    print("Saved Voice Router + RAG integration outputs:")
    for k, v in saved_files.items():
        print(f"- {k}: {v}")

    return saved_files


# =========================================================
# Run Save Function
# تشغيل دالة الحفظ
# =========================================================

saved_router_files = save_voice_router_rag_results(
    raw_df=integration_test_df,
    display_df=integration_display,
    output_dir=STAGE03_DIR,
    base_name="voice_router_rag_integration_test"
)

Saved Voice Router + RAG integration outputs:
- raw_csv: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\voice_router_rag_integration_test_raw.csv
- raw_excel: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\voice_router_rag_integration_test_raw.xlsx
- display_csv: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\voice_router_rag_integration_test_display.csv
- display_excel: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\voice_router_rag_integration_test_display.xlsx
- styled_html: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\voice_router_rag_integration_test_styled.html
- manifest: C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\voice_router_rag_integration_test_save_manifest

In [37]:
# =========================================================
# Stage 22 - Final Stage 03 Output Manifest
# فهرس نهائي لمخرجات المرحلة الثالثة
# =========================================================

import pandas as pd
from pathlib import Path
from IPython.display import display, HTML

# ---------------------------------------------------------
# 1) Required output files
# ملفات أساسية يجب أن تكون موجودة
# ---------------------------------------------------------

required_output_files = [
    STAGE03_DIR / "stage03_input_safety_checks.csv",

    # ChromaDB manifests
    STAGE03_DIR / "chroma_collections_manifest.csv",
    STAGE03_DIR / "chroma_collections_manifest.xlsx",
    STAGE03_DIR / "chroma_collections_manifest.json",

    # Corpus and embeddings
    STAGE03_DIR / "corpus_summary_by_chunking_strategy.csv",
    STAGE03_DIR / "embedding_generation_summary.csv",
    STAGE03_DIR / "embedding_generation_summary.xlsx",

    # Fixed FAQ retrieval evaluation
    STAGE03_DIR / "stage03_fixed_faq_and_legal_retrieval_eval_dataset.csv",

    # Retrieval evaluation
    STAGE03_DIR / "retrieval_evaluation_detailed_results.csv",
    STAGE03_DIR / "retrieval_evaluation_summary.csv",
    STAGE03_DIR / "retrieval_evaluation_summary.xlsx",
    STAGE03_DIR / "retrieval_summary_by_eval_type.csv",
    STAGE03_DIR / "retrieval_summary_by_eval_type.xlsx",

    # Best configuration
    STAGE03_DIR / "best_retrieval_config.json",

    # Out-of-scope and reliability gate
    STAGE03_DIR / "out_of_scope_retrieval_behavior.csv",
    STAGE03_DIR / "reliability_gate_score_calibration.csv",

    # Final RAG retriever packaging
    STAGE03_DIR / "stage03_rag_retriever_context_examples.csv",
    STAGE03_DIR / "stage03_rag_retriever_context_examples.xlsx",
    STAGE03_DIR / "stage03_best_retriever_ready_for_llm_config.json",

    # Voice Router + RAG integration
    STAGE03_DIR / "voice_router_rag_integration_test_raw.csv",
    STAGE03_DIR / "voice_router_rag_integration_test_raw.xlsx",
    STAGE03_DIR / "voice_router_rag_integration_test_display.csv",
    STAGE03_DIR / "voice_router_rag_integration_test_display.xlsx",
    STAGE03_DIR / "voice_router_rag_integration_test_save_manifest.csv",

    # Final academic report
    STAGE03_DIR / "stage03_final_retrieval_report.json",
    STAGE03_DIR / "stage03_final_summary_table.csv",
    STAGE03_DIR / "stage03_final_summary_table.xlsx",
]

# ---------------------------------------------------------
# 2) Optional output files
# ملفات اختيارية قد لا تظهر إذا لم توجد أخطاء أو إذا لم يتم تشغيل خلية معينة
# ---------------------------------------------------------

optional_output_files = [
    STAGE03_DIR / "best_config_error_summary.csv",
    STAGE03_DIR / "best_config_failures_at_5.csv",

    STAGE03_DIR / "legal_retrieval_evaluation_summary.csv",
    STAGE03_DIR / "legal_retrieval_evaluation_summary.xlsx",
    STAGE03_DIR / "best_legal_retrieval_config.json",

    STAGE03_DIR / "voice_router_rag_integration_test_styled.html",
]

# ---------------------------------------------------------
# 3) Figure files
# الرسومات الأكاديمية
# ---------------------------------------------------------

figure_files = sorted(FIGURES_DIR.glob("stage03_*.png"))

# ---------------------------------------------------------
# 4) Build manifest rows
# ---------------------------------------------------------

manifest_rows = []

def add_manifest_row(path, file_type, requirement):
    path = Path(path)
    exists = path.exists()

    manifest_rows.append({
        "file_type": file_type,
        "requirement": requirement,
        "file_name": path.name,
        "path": str(path),
        "exists": exists,
        "size_kb": round(path.stat().st_size / 1024, 2) if exists else 0,
        "status": "OK" if exists else ("MISSING_REQUIRED" if requirement == "required" else "MISSING_OPTIONAL"),
    })

for p in required_output_files:
    add_manifest_row(p, "data_or_report", "required")

for p in optional_output_files:
    add_manifest_row(p, "data_or_report", "optional")

for p in figure_files:
    add_manifest_row(p, "figure", "required")

manifest = pd.DataFrame(manifest_rows)

# ---------------------------------------------------------
# 5) Save manifest
# ---------------------------------------------------------

manifest_csv_path = STAGE03_DIR / "stage03_output_manifest.csv"
manifest_xlsx_path = STAGE03_DIR / "stage03_output_manifest.xlsx"

manifest.to_csv(
    manifest_csv_path,
    index=False,
    encoding="utf-8-sig"
)

manifest.to_excel(
    manifest_xlsx_path,
    index=False
)

# ---------------------------------------------------------
# 6) Display professional summary
# ---------------------------------------------------------

summary_manifest = (
    manifest
    .groupby(["file_type", "requirement", "status"])
    .size()
    .reset_index(name="count")
)

display(HTML("""
<div dir="rtl" style="text-align:right; line-height:2; font-size:17px;">
<h3>فهرس مخرجات المرحلة الثالثة</h3>
<p>
يوضح هذا الجدول حالة جميع الملفات الناتجة من مرحلة بناء وتقييم طبقة الاسترجاع.
تم الفصل بين الملفات الأساسية المطلوبة والملفات الاختيارية التي قد لا تظهر إلا في حالات معينة مثل وجود أخطاء في الاسترجاع.
</p>
</div>
"""))

display(summary_manifest)
display(manifest)

# ---------------------------------------------------------
# 7) Print file status
# ---------------------------------------------------------

print("Stage 03 output files:")
for _, r in manifest.iterrows():
    icon = "✅" if r["exists"] else ("❌" if r["requirement"] == "required" else "⚠️")
    print(icon, r["requirement"], "|", r["path"])

print("\nSaved manifest files:")
print(manifest_csv_path)
print(manifest_xlsx_path)

# ---------------------------------------------------------
# 8) Required files assertion only
# ---------------------------------------------------------

missing_required = manifest[
    (manifest["requirement"] == "required") &
    (manifest["exists"] == False)
]

if len(missing_required) > 0:
    print("\nMissing required files:")
    display(missing_required)
else:
    print("\nAll required Stage 03 output files exist.")

assert missing_required.empty, "Some required Stage 03 output files are missing."

print("Stage 03 completed successfully.")

,file_type,requirement,status,count
0,data_or_report,optional,MISSING_OPTIONAL,3
1,data_or_report,optional,OK,3
2,data_or_report,required,OK,27
3,figure,required,OK,8


,file_type,requirement,file_name,path,exists,size_kb,status
0,data_or_report,required,stage03_input_safety_checks.csv,C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\stage03_input_sa...,True,0.17,OK
1,data_or_report,required,chroma_collections_manifest.csv,C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\chroma_collectio...,True,0.65,OK
2,data_or_report,required,chroma_collections_manifest.xlsx,C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\chroma_collectio...,True,5.03,OK
3,data_or_report,required,chroma_collections_manifest.json,C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\chroma_collectio...,True,1.26,OK
4,data_or_report,required,corpus_summary_by_chunking_strategy.csv,C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\corpus_summary_b...,True,0.38,OK
5,data_or_report,required,embedding_generation_summary.csv,C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\embedding_genera...,True,0.49,OK
6,data_or_report,required,embedding_generation_summary.xlsx,C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\embedding_genera...,True,5.08,OK
7,data_or_report,required,stage03_fixed_faq_and_legal_retrieval_eval_dataset.csv,C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\stage03_fixed_fa...,True,109.54,OK
8,data_or_report,required,retrieval_evaluation_detailed_results.csv,C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\retrieval_evalua...,True,2707.50,OK
9,data_or_report,required,retrieval_evaluation_summary.csv,C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\retrieval_evalua...,True,1.38,OK


Stage 03 output files:
✅ required | C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\stage03_input_safety_checks.csv
✅ required | C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\chroma_collections_manifest.csv
✅ required | C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\chroma_collections_manifest.xlsx
✅ required | C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\chroma_collections_manifest.json
✅ required | C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\corpus_summary_by_chunking_strategy.csv
✅ required | C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\embedding_generation_summary.csv
✅ required | C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_sta

In [53]:
from pathlib import Path

files_to_find = [
    "stage03_final_summary_table.xlsx",
    "retrieval_evaluation_summary.xlsx",
    "retrieval_summary_by_eval_type.xlsx",
    "stage03_output_manifest.xlsx",
]

print("STAGE03_DIR:")
print(STAGE03_DIR)

print("\nFiles status:")
for f in files_to_find:
    p = STAGE03_DIR / f
    print(("✅" if p.exists() else "❌"), p)

STAGE03_DIR:
C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results

Files status:
✅ C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\stage03_final_summary_table.xlsx
✅ C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\retrieval_evaluation_summary.xlsx
✅ C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\retrieval_summary_by_eval_type.xlsx
✅ C:\Users\PC\Desktop\data collection code\saudi_labor_law_voice_agent_project\08_stage03_rag_results\stage03_output_manifest.xlsx
